<a href="https://www.kaggle.com/code/mrrogueknight/finanalytics?scriptVersionId=336015069" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# FinAnalytics: Complete Trading System

This comprehensive notebook implements a complete quantitative trading platform with 9 integrated modules covering the entire investment workflow from data collection to advanced analytics.

---

## 🔧 MODULE BREAKDOWN

### **Cell 1: Installation**
Installs all required dependencies:
- `yfinance`, `pandas`, `numpy` - Data handling
- `ta`, `scikit-learn`, `scipy` - Technical analysis & ML
- `matplotlib`, `seaborn`, `plotly` - Visualization
- `sqlalchemy`, `requests` - Database & API

### **Cell 2: Stock Data Scraper**
Full-featured data collection system:
- **Stock Database**: Pre-verified Indian stocks (RELIANCE.NS, TCS.NS, etc.)
- **Cache Manager**: 300-second TTL with LRU eviction
- **Multi-threading**: Concurrent downloads with ThreadPoolExecutor
- **Export**: CSV/JSON with automatic timestamp
- **Smart Resolution**: Auto-detects NSE/BSE suffixes

```python
scraper = StockScraper()
results = scraper.fetch_multiple(['RELIANCE.NS', 'TCS.NS', 'INFY.NS'])
```

### **Cell 3: Complete Trading Terminal**
The core analysis engine with:
- **Technical Indicators**: SMA, EMA, RSI, MACD, Bollinger Bands, ATR, ADX, OBV, Supertrend
- **Market Regime Detection**: Identifies trending/sideways/volatile regimes
- **Scoring Engine**: Multi-factor scoring (trend 30%, momentum 25%, volatility 15%, volume 15%, patterns 15%)
- **Position Sizing**: ATR-based with configurable stop-loss (2x ATR)

```python
terminal = TradingTerminal()
analysis = terminal.analyze_stock('RELIANCE.NS')
# Returns: score, recommendation, confidence, greeks, position sizing
```

### **Cell 4: Trading Visualizations**
Comprehensive charting with Plotly:
- **Candlestick Charts** with SMA/EMA/Bollinger overlays
- **Indicator Panel**: RSI, MACD, ADX in subplots
- **Support/Resistance Detection**: Automated level finding
- **Performance Heatmap**: Normalized scores across stocks
- **Score Gauge**: Visual recommendation meter

### **Cell 5: Strategy Engine & Backtesting**
Quantitative strategy testing:
- **Strategies**: RSI, MACD, SMA Crossover, Multi-Strategy
- **Metrics**: Total/Annual Return, Sharpe Ratio, Max Drawdown, Win Rate, Profit Factor
- **Optimization**: Parameter grid search for RSI and MACD
- **Walk-Forward Validation**: Time-series cross-validation

```python
strategy = RSIStrategy(oversold=30, overbought=70, period=14)
metrics = BacktestEngine().run(df, strategy)
# Returns: total_return, sharpe_ratio, max_drawdown, win_rate
```

### **Cell 6: Stock Recommendation Engine**
Ranking system for stock selection:
- **Portfolio Recommender**: Capital allocation with position sizing
- **Risk Assessment**: LOW/MEDIUM/HIGH risk classification
- **Dynamic Scoring**: 6-factor weighted scoring system
- **Support/Resistance**: Automated S/R level detection

```python
engine = RecommendationEngine()
rankings = engine.rank_stocks(stocks_data)
# Returns: sorted list with recommendations
```

### **Cell 7: Portfolio Optimization & Risk Analytics**
Modern Portfolio Theory implementation:
- **Efficient Frontier**: Markowitz optimization with Ledoit-Wolf covariance
- **Tangency Portfolio**: Maximum Sharpe ratio
- **Risk Metrics**: VaR, CVaR, Drawdown, Sortino, Calmar
- **Beta Analysis**: Market sensitivity to NIFTY 50

```python
optimizer = PortfolioOptimizer(returns)
max_sharpe = optimizer.optimize_sharpe_ratio()
# Returns: optimal weights, expected return, volatility
```

### **Cell 8: Options Pricing & Greeks Analysis**
Black-Scholes model implementation:
- **Pricing**: Call/Put option valuation
- **Greeks**: Delta, Gamma, Vega, Theta, Rho
- **Strategies**: Covered Call, Protective Put, Straddle, Strangle
- **Implied Volatility**: Brent's method for IV calculation
- **Payoff Diagrams**: Visual strategy outcomes

```python
call_price = BlackScholes.call_price(S, K, T, r, sigma)
greeks = Greeks.calculate_all(S, K, T, r, sigma, 'call')
```

### **Cell 9: Machine Learning Prediction**
ML models for risk/return forecasting:
- **Models**: Random Forest (optimized), Gradient Boosting, Linear Regression, SVR
- **Features**: 70+ technical indicators (price, volume, volatility, momentum)
- **Validation**: TimeSeriesSplit cross-validation
- **Trading Simulation**: Strategy vs Buy-and-Hold comparison

```python
trainer = ModelTrainer(X, y, y_direction)
results = trainer.train_all_models()
# Returns: CV metrics for each model
```

---


## 🎯 HOW TO USE

1. **Run Cell 1** - Install dependencies
2. **Cell 2** - Scrape stock data (select from default/random/custom)
3. **Cell 3** - Analyze individual stocks
4. **Cell 4** - Generate visualizations
5. **Cell 5** - Backtest trading strategies
6. **Cell 6** - Get stock recommendations
7. **Cell 7** - Optimize your portfolio
8. **Cell 8** - Price options & analyze Greeks
9. **Cell 9** - ML-based predictions

# CELL 1: INSTALLATION AND REQUIREMENTS

In [1]:
# ============================================================================
# CELL 1: INSTALLATION AND REQUIREMENTS
# ============================================================================

import subprocess
import sys

def install_requirements():
    requirements = [
        'yfinance>=0.2.28',
        'pandas>=2.0.0',
        'numpy>=1.24.0',
        'ta>=0.10.2',
        'scikit-learn>=1.3.0',
        'scipy>=1.11.0',
        'matplotlib>=3.7.0',
        'seaborn>=0.12.0',
        'plotly>=5.17.0',
        'sqlalchemy>=2.0.0',
        'requests>=2.31.0'
    ]

    print("=" * 60)
    print("INSTALLING REQUIREMENTS")
    print("=" * 60)

    for package in requirements:
        print(f"Installing: {package}...", end=" ", flush=True)
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--quiet", package],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            print("OK")
        except Exception as e:
            print(f"FAILED: {e}")

    print("=" * 60)
    print("Installation complete")
    print("=" * 60)

if __name__ == "__main__":
    install_requirements()

INSTALLING REQUIREMENTS
Installing: yfinance>=0.2.28... OK
Installing: pandas>=2.0.0... OK
Installing: numpy>=1.24.0... OK
Installing: ta>=0.10.2... OK
Installing: scikit-learn>=1.3.0... OK
Installing: scipy>=1.11.0... OK
Installing: matplotlib>=3.7.0... OK
Installing: seaborn>=0.12.0... OK
Installing: plotly>=5.17.0... OK
Installing: sqlalchemy>=2.0.0... OK
Installing: requests>=2.31.0... OK
Installation complete


# CELL 2: STOCK DATA SCRAPER

In [2]:
# ============================================================================
# CELL 2: STOCK DATA SCRAPER
# ============================================================================

import time
import json
import os
import random
from datetime import datetime, timedelta
from typing import Optional, Dict, List, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass

try:
    import yfinance as yf
    import pandas as pd
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
except ImportError as e:
    print(f"Missing required package: {e}")
    print("Please run Cell 1 first")
    raise

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class Config:
    cache_duration: int = 300
    max_cache_size: int = 100
    max_retries: int = 3
    request_timeout: int = 30
    max_workers: int = 5
    enable_cache: bool = True

config = Config()

# ============================================================================
# STOCK DATABASE - Verified working symbols only
# ============================================================================

STOCK_DATABASE = [
    'RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS',
    'ICICIBANK.NS', 'KOTAKBANK.NS', 'SBIN.NS', 'BHARTIARTL.NS',
    'ITC.NS', 'WIPRO.NS', 'HCLTECH.NS', 'SUNPHARMA.NS',
    'TITAN.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'TATASTEEL.NS',
    'JSWSTEEL.NS', 'ONGC.NS', 'POWERGRID.NS', 'NTPC.NS',
    'PIDILITIND.NS', 'COLPAL.NS', 'TATACONSUM.NS',
    'DIVISLAB.NS', 'DRREDDY.NS', 'CIPLA.NS',
    'UPL.NS', 'GRASIM.NS', 'INDUSINDBK.NS', 'AXISBANK.NS',
    'BAJFINANCE.NS', 'EICHERMOT.NS', 'HEROMOTOCO.NS', 'M&M.NS',
    'YESBANK.NS', 'PNB.NS', 'BPCL.NS',
    'GAIL.NS', 'TATAPOWER.NS', 'ADANIPORTS.NS',
    'NMDC.NS', 'SAIL.NS', 'BEL.NS', 'COALINDIA.NS'
]

# ============================================================================
# CACHE MANAGER
# ============================================================================

class CacheManager:
    def __init__(self, max_size: int = 100, ttl: int = 300):
        self._cache = {}
        self._max_size = max_size
        self._ttl = ttl
        self._hits = 0
        self._misses = 0

    def get(self, key: str) -> Optional[Any]:
        if key in self._cache:
            value, timestamp = self._cache[key]
            if (datetime.now() - timestamp).seconds < self._ttl:
                self._hits += 1
                return value
            else:
                del self._cache[key]
                self._misses += 1
        else:
            self._misses += 1
        return None

    def set(self, key: str, value: Any):
        if len(self._cache) >= self._max_size:
            oldest_key = next(iter(self._cache))
            del self._cache[oldest_key]
        self._cache[key] = (value, datetime.now())

    def get_stats(self) -> Dict:
        total = self._hits + self._misses
        hit_rate = (self._hits / total * 100) if total > 0 else 0
        return {
            'size': len(self._cache),
            'hits': self._hits,
            'misses': self._misses,
            'hit_rate': f"{hit_rate:.1f}%"
        }

# ============================================================================
# STOCK DATA PROVIDER
# ============================================================================

class StockDataProvider:
    def __init__(self):
        self._cache = CacheManager(
            max_size=config.max_cache_size,
            ttl=config.cache_duration
        )
        self._session = self._create_session()

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=config.max_retries,
            backoff_factor=0.5,
            status_forcelist=[500, 502, 503, 504]
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount('http://', adapter)
        session.mount('https://', adapter)
        return session

    def get_stock_data(self, symbol: str, force_refresh: bool = False) -> Dict:
        cache_key = symbol.upper()

        if config.enable_cache and not force_refresh:
            cached_data = self._cache.get(cache_key)
            if cached_data:
                return cached_data

        try:
            # Try to get the symbol with proper exchange
            symbol = self._resolve_symbol(symbol)

            stock = yf.Ticker(symbol)
            info = stock.info

            if not info or 'regularMarketPrice' not in info:
                return self._error_response(symbol, 'No data available')

            result = self._parse_response(info, symbol)

            if config.enable_cache:
                self._cache.set(cache_key, result)

            return result

        except Exception:
            return self._error_response(symbol, 'No data available')

    def _resolve_symbol(self, symbol: str) -> str:
        """Resolve symbol with proper exchange suffix"""
        symbol = symbol.upper()

        if symbol.endswith('.NS') or symbol.endswith('.BO'):
            return symbol

        # Try NSE
        test_symbol = f"{symbol}.NS"
        try:
            test = yf.Ticker(test_symbol)
            if test.info.get('regularMarketPrice'):
                return test_symbol
        except:
            pass

        # Try BSE
        test_symbol = f"{symbol}.BO"
        try:
            test = yf.Ticker(test_symbol)
            if test.info.get('regularMarketPrice'):
                return test_symbol
        except:
            pass

        return symbol

    def _parse_response(self, info: Dict, symbol: str) -> Dict:
        current_price = info.get('currentPrice') or info.get('regularMarketPrice')
        previous_close = info.get('previousClose')

        change = None
        change_percent = None
        if current_price and previous_close:
            change = current_price - previous_close
            change_percent = (change / previous_close) * 100

        return {
            'Symbol': symbol,
            'Price': current_price,
            'Change': change,
            'Change_Percent': change_percent,
            'Open': info.get('open') or info.get('regularMarketOpen'),
            'High': info.get('dayHigh') or info.get('regularMarketDayHigh'),
            'Low': info.get('dayLow') or info.get('regularMarketDayLow'),
            'Prev_Close': previous_close,
            'Volume': info.get('volume') or info.get('regularMarketVolume'),
            'Market_Cap': info.get('marketCap'),
            'PE_Ratio': info.get('trailingPE'),
            'Dividend_Yield': info.get('dividendYield') * 100 if info.get('dividendYield') else None,
            'Week_High': info.get('fiftyTwoWeekHigh'),
            'Week_Low': info.get('fiftyTwoWeekLow'),
            'Scraped_At': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'Source': 'yFinance API'
        }

    def _error_response(self, symbol: str, message: str) -> Dict:
        return {
            'Symbol': symbol,
            'Price': None,
            'Error': message,
            'Scraped_At': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

    def get_cache_stats(self) -> Dict:
        return self._cache.get_stats()

    def clear_cache(self):
        self._cache = CacheManager(
            max_size=config.max_cache_size,
            ttl=config.cache_duration
        )

# ============================================================================
# STOCK SCRAPER
# ============================================================================

class StockScraper:
    def __init__(self):
        self._provider = StockDataProvider()
        self._stats = {
            'total_requests': 0,
            'successful': 0,
            'failed': 0
        }

    def fetch_stock(self, symbol: str, force_refresh: bool = False) -> Dict:
        self._stats['total_requests'] += 1
        result = self._provider.get_stock_data(symbol, force_refresh)

        if result.get('Price') is not None:
            self._stats['successful'] += 1
        else:
            self._stats['failed'] += 1

        return result

    def fetch_multiple(self, symbols: List[str], force_refresh: bool = False) -> Dict:
        results = {}

        with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
            future_to_symbol = {
                executor.submit(self.fetch_stock, symbol, force_refresh): symbol
                for symbol in symbols
            }

            for future in as_completed(future_to_symbol):
                symbol = future_to_symbol[future]
                try:
                    results[symbol] = future.result(timeout=config.request_timeout)
                except Exception:
                    results[symbol] = self._provider._error_response(symbol, 'Timeout')

        return results

    def get_stats(self) -> Dict:
        cache_stats = self._provider.get_cache_stats()
        total = self._stats['total_requests']
        success_rate = (self._stats['successful'] / max(total, 1)) * 100

        return {
            'total_requests': self._stats['total_requests'],
            'successful': self._stats['successful'],
            'failed': self._stats['failed'],
            'success_rate': f"{success_rate:.1f}%",
            'cache': cache_stats
        }

    def export_csv(self, results: Dict, filename: Optional[str] = None) -> str:
        if filename is None:
            filename = f"stock_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

        df = pd.DataFrame(results).T
        columns = ['Symbol', 'Price', 'Change', 'Change_Percent', 'Open', 'High',
                   'Low', 'Prev_Close', 'Volume', 'Market_Cap', 'PE_Ratio',
                   'Dividend_Yield', 'Week_High', 'Week_Low', 'Source', 'Scraped_At']

        existing_columns = [col for col in columns if col in df.columns]
        df = df[existing_columns]
        df.to_csv(filename, index=False)
        return filename

    def export_json(self, results: Dict, filename: Optional[str] = None) -> str:
        if filename is None:
            filename = f"stock_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        def convert(obj):
            if isinstance(obj, (int, float)):
                return obj
            if isinstance(obj, (datetime, pd.Timestamp)):
                return obj.isoformat()
            return str(obj)

        with open(filename, 'w') as f:
            json.dump(results, f, indent=2, default=convert)
        return filename

# ============================================================================
# DISPLAY FUNCTIONS
# ============================================================================

def display_results(results: Dict):
    valid_results = {
        k: v for k, v in results.items()
        if v.get('Price') is not None
    }

    failed_count = len(results) - len(valid_results)

    if failed_count > 0:
        print(f"\nWarning: {failed_count} stock(s) unavailable")

    if not valid_results:
        print("No data available")
        return

    table_data = []
    for symbol, data in valid_results.items():
        price = data.get('Price')
        change_pct = data.get('Change_Percent')

        change_str = "N/A"
        if change_pct is not None:
            sign = "+" if change_pct > 0 else ""
            change_str = f"{sign}{change_pct:.2f}%"

        table_data.append({
            'Symbol': data.get('Symbol', symbol),
            'Price': f"{price:,.2f}" if price else "N/A",
            'Change': change_str,
            'Volume': f"{data.get('Volume'):,}" if data.get('Volume') else "N/A",
            'Source': data.get('Source', 'N/A')
        })

    if table_data:
        table_data.sort(key=lambda x: x['Symbol'])
        df = pd.DataFrame(table_data)
        print("\n" + "=" * 100)
        print("STOCK DATA SUMMARY")
        print("=" * 100)
        print(df.to_string(index=False))
        print("=" * 100)

def display_stats(stats: Dict):
    print("\n" + "=" * 70)
    print("STATISTICS")
    print("=" * 70)
    print(f"Total Requests: {stats['total_requests']}")
    print(f"Successful: {stats['successful']}")
    print(f"Failed: {stats['failed']}")
    print(f"Success Rate: {stats['success_rate']}")

    if 'cache' in stats:
        print(f"\nCache:")
        print(f"  Size: {stats['cache']['size']}")
        print(f"  Hits: {stats['cache']['hits']}")
        print(f"  Misses: {stats['cache']['misses']}")
        print(f"  Hit Rate: {stats['cache']['hit_rate']}")
    print("=" * 70)

# ============================================================================
# USER INPUT HANDLING
# ============================================================================

def get_random_stocks(count: int) -> List[str]:
    return random.sample(STOCK_DATABASE, min(count, len(STOCK_DATABASE)))

def get_symbols_from_user() -> List[str]:
    print("\n" + "=" * 70)
    print("STOCK SELECTION")
    print("=" * 70)

    print("\nOptions:")
    print("  [1] Enter custom symbols")
    print("  [2] Generate random stocks")
    print("  [3] Use default stocks")
    print("-" * 70)

    choice = input("Select option (1/2/3) [default: 3]: ").strip()

    if choice == '1':
        print("\nEnter stock symbols (comma-separated)")
        print("Examples: KOTAKBANK.NS, RELIANCE.NS, TCS.NS")
        print("Or: KOTAKBANK, RELIANCE, TCS")
        print("-" * 70)

        user_input = input("\nSymbols: ").strip()
        if user_input:
            symbols = [s.strip() for s in user_input.split(',') if s.strip()]
            if symbols:
                return symbols

    elif choice == '2':
        try:
            count_input = input("\nNumber of stocks [default: 5]: ").strip()
            count = int(count_input) if count_input else 5
            count = max(1, min(count, 20))

            symbols = get_random_stocks(count)
            print(f"\nGenerated {len(symbols)} stocks:")
            for i, symbol in enumerate(symbols, 1):
                print(f"  {i}. {symbol}")

            confirm = input("\nProceed? (y/n) [default: y]: ").strip().lower()
            if confirm != 'n':
                return symbols
            return get_symbols_from_user()
        except ValueError:
            print("Invalid input, using defaults")

    # Default
    default = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'KOTAKBANK.NS']
    print(f"\nUsing defaults: {', '.join(default)}")
    return default

# ============================================================================
# EXPORT HANDLING
# ============================================================================

def handle_export(results: Dict, scraper: StockScraper) -> None:
    print("\n" + "=" * 70)
    print("EXPORT DATA")
    print("=" * 70)

    print("\nSelect format:")
    print("  [1] CSV")
    print("  [2] JSON")
    print("  [3] Both CSV and JSON")
    print("  [4] None")
    print("-" * 70)

    choice = input("Select option (1/2/3/4) [default: 4]: ").strip()

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    if choice == '1':
        try:
            filename = scraper.export_csv(results, f"stock_data_{timestamp}.csv")
            print(f"\nExported: {filename}")
            print(f"Location: {os.path.abspath(filename)}")
        except Exception as e:
            print(f"\nExport failed: {e}")

    elif choice == '2':
        try:
            filename = scraper.export_json(results, f"stock_data_{timestamp}.json")
            print(f"\nExported: {filename}")
            print(f"Location: {os.path.abspath(filename)}")
        except Exception as e:
            print(f"\nExport failed: {e}")

    elif choice == '3':
        try:
            csv_file = scraper.export_csv(results, f"stock_data_{timestamp}.csv")
            json_file = scraper.export_json(results, f"stock_data_{timestamp}.json")
            print(f"\nExported both formats:")
            print(f"  CSV: {csv_file}")
            print(f"  JSON: {json_file}")
            print(f"Location: {os.path.abspath('.')}")
        except Exception as e:
            print(f"\nExport failed: {e}")

    else:
        print("\nNo export selected")

# ============================================================================
# MAIN
# ============================================================================

def main():
    print("\n" + "=" * 70)
    print("STOCK DATA SCRAPER")
    print("=" * 70)
    print(f"Source: yFinance API")
    print(f"Cache: {config.cache_duration}s")
    print("=" * 70)

    symbols = get_symbols_from_user()

    if not symbols:
        print("No symbols provided")
        return

    scraper = StockScraper()

    print(f"\nFetching {len(symbols)} stocks...")
    results = scraper.fetch_multiple(symbols)

    display_results(results)
    handle_export(results, scraper)
    display_stats(scraper.get_stats())

    print("\n" + "=" * 70)
    print("COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    main()


STOCK DATA SCRAPER
Source: yFinance API
Cache: 300s

STOCK SELECTION

Options:
  [1] Enter custom symbols
  [2] Generate random stocks
  [3] Use default stocks
----------------------------------------------------------------------


Select option (1/2/3) [default: 3]:  2

Number of stocks [default: 5]:  5



Generated 5 stocks:
  1. DIVISLAB.NS
  2. COALINDIA.NS
  3. CIPLA.NS
  4. JSWSTEEL.NS
  5. TATAPOWER.NS



Proceed? (y/n) [default: y]:  3



Fetching 5 stocks...

STOCK DATA SUMMARY
      Symbol    Price Change    Volume       Source
    CIPLA.NS 1,418.70 -0.76%   606,933 yFinance API
COALINDIA.NS   427.65 +0.07% 4,141,501 yFinance API
 DIVISLAB.NS 7,247.50 -0.94%   455,057 yFinance API
 JSWSTEEL.NS 1,237.30 +1.33% 2,799,490 yFinance API
TATAPOWER.NS   377.25 +0.04% 2,625,483 yFinance API

EXPORT DATA

Select format:
  [1] CSV
  [2] JSON
  [3] Both CSV and JSON
  [4] None
----------------------------------------------------------------------


Select option (1/2/3/4) [default: 4]:  3



Exported both formats:
  CSV: stock_data_20260717_173748.csv
  JSON: stock_data_20260717_173748.json
Location: /kaggle/working

STATISTICS
Total Requests: 5
Successful: 5
Failed: 0
Success Rate: 100.0%

Cache:
  Size: 5
  Hits: 0
  Misses: 5
  Hit Rate: 0.0%

COMPLETE


# CELL 3: COMPLETE TRADING TERMINAL

In [3]:
# ============================================================================
# CELL 3: COMPLETE TRADING TERMINAL
# ============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from enum import Enum
import warnings
import json
import sqlite3
import hashlib
from pathlib import Path
warnings.filterwarnings('ignore')

# ============================================================================
# CORE CONFIGURATION
# ============================================================================

@dataclass
class MarketConfig:
    """Central configuration for all market analysis"""
    # Data
    history_period: str = "2y"
    interval: str = "1d"
    cache_duration: int = 300

    # Indicators
    rsi_period: int = 14
    ema_fast: int = 9
    ema_slow: int = 21
    ema_very_slow: int = 50
    sma_short: int = 20
    sma_medium: int = 50
    sma_long: int = 200
    macd_fast: int = 12
    macd_slow: int = 26
    macd_signal: int = 9
    bollinger_period: int = 20
    bollinger_std: float = 2.0
    atr_period: int = 14

    # Risk
    risk_free_rate: float = 0.07  # 7% for India
    max_position_size: float = 0.20  # 20% of portfolio
    stop_loss_atr: float = 2.0

    # ML
    ml_train_size: float = 0.8
    ml_test_size: float = 0.2

config = MarketConfig()

# ============================================================================
# DATA LAYER - Caching & Database
# ============================================================================

class DataCache:
    """Thread-safe data cache with SQLite persistence"""

    def __init__(self, db_path: str = "market_data.db"):
        self.db_path = db_path
        self._init_database()
        self._memory_cache = {}
        self._cache_timestamps = {}

    def _init_database(self):
        """Initialize SQLite database"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS stock_data (
                symbol TEXT,
                date TEXT,
                open REAL,
                high REAL,
                low REAL,
                close REAL,
                volume REAL,
                PRIMARY KEY (symbol, date)
            )
        ''')
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS cache_metadata (
                symbol TEXT,
                last_updated TEXT,
                period TEXT,
                interval TEXT,
                PRIMARY KEY (symbol, period, interval)
            )
        ''')
        conn.commit()
        conn.close()

    def get(self, symbol: str, period: str = "2y", interval: str = "1d") -> Optional[pd.DataFrame]:
        """Get cached data if valid"""
        cache_key = f"{symbol}_{period}_{interval}"

        # Check memory cache first
        if cache_key in self._memory_cache:
            timestamp = self._cache_timestamps.get(cache_key)
            if timestamp and (datetime.now() - timestamp).seconds < config.cache_duration:
                return self._memory_cache[cache_key].copy()

        # Check database
        try:
            conn = sqlite3.connect(self.db_path)
            query = f"""
                SELECT date, open, high, low, close, volume
                FROM stock_data
                WHERE symbol = ?
                ORDER BY date DESC
                LIMIT 2000
            """
            df = pd.read_sql_query(query, conn, params=[symbol])
            conn.close()

            if not df.empty:
                df['date'] = pd.to_datetime(df['date'])
                df = df.set_index('date').sort_index()
                self._memory_cache[cache_key] = df.copy()
                self._cache_timestamps[cache_key] = datetime.now()
                return df
        except:
            pass

        return None

    def set(self, symbol: str, df: pd.DataFrame, period: str = "2y", interval: str = "1d"):
        """Store data in cache"""
        cache_key = f"{symbol}_{period}_{interval}"
        self._memory_cache[cache_key] = df.copy()
        self._cache_timestamps[cache_key] = datetime.now()

        # Store in database
        try:
            conn = sqlite3.connect(self.db_path)
            df_reset = df.reset_index()
            df_reset['symbol'] = symbol
            df_reset['date'] = df_reset['date'].dt.strftime('%Y-%m-%d')

            df_reset.to_sql('stock_data', conn, if_exists='append', index=False)
            conn.commit()
            conn.close()
        except:
            pass

# ============================================================================
# DATA COLLECTOR
# ============================================================================

class DataCollector:
    """Collects and manages historical stock data"""

    def __init__(self):
        self.cache = DataCache()
        self._nifty_symbols = self._load_nifty_symbols()

    def _load_nifty_symbols(self) -> List[str]:
        """Load NIFTY 50 symbols"""
        return [
            'RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS',
            'ICICIBANK.NS', 'KOTAKBANK.NS', 'SBIN.NS', 'BHARTIARTL.NS',
            'ITC.NS', 'WIPRO.NS', 'HCLTECH.NS', 'SUNPHARMA.NS',
            'TITAN.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'TATASTEEL.NS',
            'JSWSTEEL.NS', 'ONGC.NS', 'POWERGRID.NS', 'NTPC.NS',
            'PIDILITIND.NS', 'COLPAL.NS', 'TATACONSUM.NS',
            'DIVISLAB.NS', 'DRREDDY.NS', 'CIPLA.NS',
            'UPL.NS', 'GRASIM.NS', 'INDUSINDBK.NS', 'AXISBANK.NS',
            'BAJFINANCE.NS', 'EICHERMOT.NS', 'HEROMOTOCO.NS', 'M&M.NS',
            'YESBANK.NS', 'PNB.NS', 'BPCL.NS', 'GAIL.NS',
            'TATAPOWER.NS', 'ADANIPORTS.NS', 'NMDC.NS', 'SAIL.NS',
            'BEL.NS', 'COALINDIA.NS'
        ]

    def fetch_historical(self, symbol: str, period: str = None,
                        interval: str = None, force_refresh: bool = False) -> pd.DataFrame:
        """Fetch historical data with caching"""

        period = period or config.history_period
        interval = interval or config.interval

        if not force_refresh:
            cached = self.cache.get(symbol, period, interval)
            if cached is not None and not cached.empty:
                return cached

        try:
            stock = yf.Ticker(symbol)
            df = stock.history(period=period, interval=interval)

            if df.empty:
                return pd.DataFrame()

            df = df.dropna()
            df.columns = [col.lower() for col in df.columns]

            self.cache.set(symbol, df, period, interval)
            return df

        except Exception as e:
            print(f"Error fetching {symbol}: {e}")
            return pd.DataFrame()

    def fetch_multiple(self, symbols: List[str], period: str = None) -> Dict[str, pd.DataFrame]:
        """Fetch multiple stocks"""
        results = {}
        for symbol in symbols:
            df = self.fetch_historical(symbol, period)
            if not df.empty:
                results[symbol] = df
        return results

# ============================================================================
# TECHNICAL INDICATORS
# ============================================================================

class TechnicalIndicators:
    """Complete technical indicator library"""

    @staticmethod
    def add_sma(df: pd.DataFrame, periods: List[int]) -> pd.DataFrame:
        for period in periods:
            df[f'sma_{period}'] = df['close'].rolling(window=period).mean()
        return df

    @staticmethod
    def add_ema(df: pd.DataFrame, periods: List[int]) -> pd.DataFrame:
        for period in periods:
            df[f'ema_{period}'] = df['close'].ewm(span=period, adjust=False).mean()
        return df

    @staticmethod
    def add_rsi(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))
        return df

    @staticmethod
    def add_macd(df: pd.DataFrame, fast: int = 12, slow: int = 26, signal: int = 9) -> pd.DataFrame:
        exp1 = df['close'].ewm(span=fast, adjust=False).mean()
        exp2 = df['close'].ewm(span=slow, adjust=False).mean()
        df['macd'] = exp1 - exp2
        df['macd_signal'] = df['macd'].ewm(span=signal, adjust=False).mean()
        df['macd_histogram'] = df['macd'] - df['macd_signal']
        return df

    @staticmethod
    def add_bollinger_bands(df: pd.DataFrame, period: int = 20, std: float = 2.0) -> pd.DataFrame:
        df['bb_middle'] = df['close'].rolling(window=period).mean()
        bb_std = df['close'].rolling(window=period).std()
        df['bb_upper'] = df['bb_middle'] + (bb_std * std)
        df['bb_lower'] = df['bb_middle'] - (bb_std * std)
        df['bb_width'] = df['bb_upper'] - df['bb_lower']
        df['bb_percent'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
        return df

    @staticmethod
    def add_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = np.max(ranges, axis=1)
        df['atr'] = true_range.rolling(window=period).mean()
        return df

    @staticmethod
    def add_adx(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
        df['tr'] = np.maximum(
            df['high'] - df['low'],
            np.maximum(
                np.abs(df['high'] - df['close'].shift()),
                np.abs(df['low'] - df['close'].shift())
            )
        )
        df['up_move'] = df['high'] - df['high'].shift()
        df['down_move'] = df['low'].shift() - df['low']
        df['plus_dm'] = np.where((df['up_move'] > df['down_move']) & (df['up_move'] > 0), df['up_move'], 0)
        df['minus_dm'] = np.where((df['down_move'] > df['up_move']) & (df['down_move'] > 0), df['down_move'], 0)
        df['atr'] = df['tr'].rolling(window=period).mean()
        df['plus_di'] = 100 * (df['plus_dm'].rolling(window=period).mean() / df['atr'])
        df['minus_di'] = 100 * (df['minus_dm'].rolling(window=period).mean() / df['atr'])
        dx = (np.abs(df['plus_di'] - df['minus_di']) / (df['plus_di'] + df['minus_di'])) * 100
        df['adx'] = dx.rolling(window=period).mean()
        return df

    @staticmethod
    def add_obv(df: pd.DataFrame) -> pd.DataFrame:
        df['obv'] = (np.sign(df['close'].diff()) * df['volume']).cumsum()
        return df

    @staticmethod
    def add_supertrend(df: pd.DataFrame, period: int = 10, multiplier: float = 3.0) -> pd.DataFrame:
        df['atr_st'] = df['high'].rolling(window=period).max() - df['low'].rolling(window=period).min()
        hl2 = (df['high'] + df['low']) / 2
        df['upper_band'] = hl2 + (multiplier * df['atr_st'])
        df['lower_band'] = hl2 - (multiplier * df['atr_st'])
        df['supertrend'] = 0
        df['supertrend_direction'] = 1

        for i in range(period, len(df)):
            if df['close'].iloc[i] > df['upper_band'].iloc[i-1]:
                df.loc[df.index[i], 'supertrend_direction'] = 1
            elif df['close'].iloc[i] < df['lower_band'].iloc[i-1]:
                df.loc[df.index[i], 'supertrend_direction'] = -1
            else:
                df.loc[df.index[i], 'supertrend_direction'] = df['supertrend_direction'].iloc[i-1]

            if df['supertrend_direction'].iloc[i] == 1:
                df.loc[df.index[i], 'supertrend'] = df['lower_band'].iloc[i]
            else:
                df.loc[df.index[i], 'supertrend'] = df['upper_band'].iloc[i]
        return df

# ============================================================================
# MARKET REGIME DETECTION
# ============================================================================

class MarketRegime:
    """Detects current market regime"""

    @staticmethod
    def detect_regime(df: pd.DataFrame) -> Dict:
        """Detect market regime: Trending, Sideways, Volatile, etc."""

        latest = df.iloc[-100:] if len(df) >= 100 else df

        # Calculate volatility
        returns = latest['close'].pct_change().dropna()
        volatility = returns.std() * np.sqrt(252)

        # Calculate trend strength
        sma_20 = latest['close'].rolling(20).mean()
        sma_50 = latest['close'].rolling(50).mean()

        if len(sma_20) > 50 and len(sma_50) > 50:
            trend_strength = abs(sma_20.iloc[-1] / sma_50.iloc[-1] - 1) * 100
        else:
            trend_strength = 0

        # Detect regime
        if volatility > 0.35:
            volatility_regime = "High Volatility"
        elif volatility > 0.20:
            volatility_regime = "Medium Volatility"
        else:
            volatility_regime = "Low Volatility"

        if trend_strength > 5:
            trend_regime = "Strong Trend"
        elif trend_strength > 2:
            trend_regime = "Weak Trend"
        else:
            trend_regime = "Sideways"

        # Combine
        regimes = {
            'High Volatility + Strong Trend': "Breakout",
            'High Volatility + Sideways': "Chopping",
            'Low Volatility + Strong Trend': "Steady Trend",
            'Low Volatility + Sideways': "Range Bound"
        }

        key = f"{volatility_regime} + {trend_regime}"
        regime = regimes.get(key, "Mixed")

        return {
            'regime': regime,
            'volatility': volatility,
            'volatility_regime': volatility_regime,
            'trend_regime': trend_regime,
            'trend_strength': trend_strength
        }

# ============================================================================
# SCORING ENGINE
# ============================================================================

class ScoringEngine:
    """Multi-factor scoring system"""

    def __init__(self):
        self.weights = {
            'trend': 0.30,
            'momentum': 0.25,
            'volatility': 0.15,
            'volume': 0.15,
            'patterns': 0.15
        }

    def calculate_score(self, df: pd.DataFrame) -> Dict:
        """Calculate comprehensive score"""

        latest = df.iloc[-1]
        score = 0
        factors = []

        # Trend Score
        trend_score = 0
        if 'sma_20' in latest and 'sma_50' in latest and 'sma_200' in latest:
            if latest['close'] > latest['sma_20'] > latest['sma_50'] > latest['sma_200']:
                trend_score = 100
            elif latest['close'] > latest['sma_20'] > latest['sma_50']:
                trend_score = 75
            elif latest['close'] > latest['sma_20']:
                trend_score = 50
            elif latest['close'] < latest['sma_20'] < latest['sma_50']:
                trend_score = 25
            else:
                trend_score = 10
        factors.append(('Trend', trend_score, self.weights['trend']))

        # Momentum Score
        momentum_score = 50
        if 'rsi' in latest:
            rsi = latest['rsi']
            if 50 <= rsi <= 70:
                momentum_score = 80
            elif 70 < rsi <= 85:
                momentum_score = 60
            elif rsi > 85:
                momentum_score = 30
            elif 30 <= rsi < 50:
                momentum_score = 40
            elif rsi < 30:
                momentum_score = 70
        factors.append(('Momentum', momentum_score, self.weights['momentum']))

        # Volatility Score
        vol_score = 50
        if 'atr' in latest and 'close' in latest:
            atr_pct = (latest['atr'] / latest['close']) * 100
            if 2 <= atr_pct <= 5:
                vol_score = 80
            elif 1 <= atr_pct < 2 or 5 < atr_pct <= 8:
                vol_score = 50
            else:
                vol_score = 30
        factors.append(('Volatility', vol_score, self.weights['volatility']))

        # Volume Score
        vol_score = 50
        if 'volume' in latest:
            avg_volume = df['volume'].tail(20).mean()
            if avg_volume > 0:
                volume_ratio = latest['volume'] / avg_volume
                if volume_ratio > 1.5:
                    vol_score = 80
                elif volume_ratio > 1.2:
                    vol_score = 60
                elif volume_ratio > 0.8:
                    vol_score = 50
                else:
                    vol_score = 30
        factors.append(('Volume', vol_score, self.weights['volume']))

        # Patterns Score
        pattern_score = 50
        patterns = self._detect_patterns(df)
        if patterns.get('bullish_engulfing') or patterns.get('hammer'):
            pattern_score = 80
        elif patterns.get('bearish_engulfing') or patterns.get('shooting_star'):
            pattern_score = 30
        factors.append(('Patterns', pattern_score, self.weights['patterns']))

        # Calculate final score
        final_score = sum(score * weight for _, score, weight in factors)
        final_score = max(0, min(100, final_score))

        # Generate recommendation
        if final_score >= 80:
            recommendation = "Strong Buy"
            confidence = 90
        elif final_score >= 65:
            recommendation = "Buy"
            confidence = 75
        elif final_score >= 45:
            recommendation = "Neutral"
            confidence = 60
        elif final_score >= 30:
            recommendation = "Sell"
            confidence = 70
        else:
            recommendation = "Strong Sell"
            confidence = 85

        return {
            'score': final_score,
            'recommendation': recommendation,
            'confidence': confidence,
            'factors': factors
        }

    def _detect_patterns(self, df: pd.DataFrame) -> Dict:
        """Detect candlestick patterns"""
        patterns = {'bullish_engulfing': False, 'bearish_engulfing': False,
                   'hammer': False, 'shooting_star': False}

        if len(df) < 3:
            return patterns

        c2, c3 = df.iloc[-2], df.iloc[-1]

        # Bullish Engulfing
        if (c2['close'] < c2['open'] and c3['close'] > c3['open'] and
            c3['open'] < c2['close'] and c3['close'] > c2['open']):
            patterns['bullish_engulfing'] = True

        # Bearish Engulfing
        if (c2['close'] > c2['open'] and c3['close'] < c3['open'] and
            c3['open'] > c2['close'] and c3['close'] < c2['open']):
            patterns['bearish_engulfing'] = True

        # Hammer
        body = abs(c3['close'] - c3['open'])
        lower_shadow = min(c3['open'], c3['close']) - c3['low']
        upper_shadow = c3['high'] - max(c3['open'], c3['close'])
        if body > 0 and lower_shadow > 2 * body and upper_shadow < body * 0.1:
            patterns['hammer'] = True

        # Shooting Star
        if body > 0 and upper_shadow > 2 * body and lower_shadow < body * 0.1:
            patterns['shooting_star'] = True

        return patterns

# ============================================================================
# PORTFOLIO & RISK MANAGEMENT
# ============================================================================

class PortfolioManager:
    """Portfolio optimization and risk management"""

    def __init__(self, capital: float = 100000):
        self.capital = capital
        self.positions = {}
        self.history = []

    def calculate_position_size(self, price: float, atr: float, risk_per_trade: float = 0.02) -> Dict:
        """Calculate optimal position size"""
        risk_amount = self.capital * risk_per_trade
        stop_loss = atr * config.stop_loss_atr

        if stop_loss == 0:
            return {'shares': 0, 'risk': 0, 'stop_loss': 0}

        shares = int(risk_amount / stop_loss)
        position_value = shares * price
        position_risk = (position_value / self.capital) * 100

        # Cap position size
        max_position = self.capital * config.max_position_size
        if position_value > max_position:
            shares = int(max_position / price)
            position_value = shares * price

        return {
            'shares': shares,
            'position_value': position_value,
            'position_risk': position_risk,
            'stop_loss': price - stop_loss,
            'risk_reward': 2.0  # Default 1:2 risk-reward
        }

    def calculate_portfolio_metrics(self, holdings: Dict[str, float], prices: Dict[str, float]) -> Dict:
        """Calculate portfolio metrics"""
        total_value = sum(holdings.get(sym, 0) * prices.get(sym, 0) for sym in holdings)

        if total_value == 0:
            return {'total_value': 0, 'cash': self.capital, 'returns': 0}

        returns = ((total_value - self.capital) / self.capital) * 100

        return {
            'total_value': total_value,
            'cash': self.capital - total_value,
            'returns': returns,
            'positions': len(holdings)
        }

# ============================================================================
# BACKTESTING ENGINE
# ============================================================================

class BacktestEngine:
    """Strategy backtesting engine"""

    def backtest_strategy(self, df: pd.DataFrame, strategy_func, initial_capital: float = 100000) -> Dict:
        """Run backtest on a strategy"""

        capital = initial_capital
        positions = 0
        trades = []
        equity_curve = []

        for i in range(50, len(df)):
            signal = strategy_func(df.iloc[:i])

            if signal == 'buy' and positions == 0:
                positions = int(capital / df['close'].iloc[i])
                capital = 0
                trades.append({
                    'date': df.index[i],
                    'type': 'buy',
                    'price': df['close'].iloc[i],
                    'shares': positions
                })
            elif signal == 'sell' and positions > 0:
                capital = positions * df['close'].iloc[i]
                trades.append({
                    'date': df.index[i],
                    'type': 'sell',
                    'price': df['close'].iloc[i],
                    'shares': positions
                })
                positions = 0

            # Track equity
            current_value = capital + (positions * df['close'].iloc[i])
            equity_curve.append({
                'date': df.index[i],
                'equity': current_value
            })

        # Finalize
        if positions > 0:
            capital = positions * df['close'].iloc[-1]

        # Calculate metrics
        total_return = ((capital - initial_capital) / initial_capital) * 100
        winning_trades = sum(1 for t in trades if t['type'] == 'sell' and t['price'] > df.loc[t['date'], 'close'])

        return {
            'total_return': total_return,
            'final_capital': capital,
            'total_trades': len([t for t in trades if t['type'] == 'sell']),
            'winning_trades': winning_trades,
            'win_rate': (winning_trades / max(len([t for t in trades if t['type'] == 'sell']), 1)) * 100,
            'trades': trades,
            'equity_curve': pd.DataFrame(equity_curve)
        }

# ============================================================================
# MARKET SCANNER
# ============================================================================

class MarketScanner:
    """Scan multiple stocks for opportunities"""

    def __init__(self):
        self.data_collector = DataCollector()
        self.scoring_engine = ScoringEngine()
        self.indicators = TechnicalIndicators()

    def scan_nifty50(self) -> pd.DataFrame:
        """Scan all NIFTY 50 stocks"""
        symbols = self.data_collector._load_nifty_symbols()
        results = []

        for symbol in symbols:
            try:
                df = self.data_collector.fetch_historical(symbol)
                if df.empty:
                    continue

                # Calculate indicators
                df = self.indicators.add_sma(df, [20, 50, 200])
                df = self.indicators.add_rsi(df)
                df = self.indicators.add_macd(df)
                df = self.indicators.add_atr(df)

                # Score
                score = self.scoring_engine.calculate_score(df)
                latest = df.iloc[-1]

                results.append({
                    'Symbol': symbol,
                    'Price': latest['close'],
                    'Score': score['score'],
                    'Recommendation': score['recommendation'],
                    'Volume': latest['volume'],
                    'RSI': latest.get('rsi'),
                    'SMA_200': latest.get('sma_200'),
                    'Above_SMA_200': latest['close'] > latest.get('sma_200', 0)
                })
            except:
                continue

        df_results = pd.DataFrame(results)
        if not df_results.empty:
            df_results = df_results.sort_values('Score', ascending=False)

        return df_results

# ============================================================================
# MAIN TRADING TERMINAL
# ============================================================================

class TradingTerminal:
    """Complete trading analysis system"""

    def __init__(self):
        self.data_collector = DataCollector()
        self.indicators = TechnicalIndicators()
        self.scoring_engine = ScoringEngine()
        self.portfolio = PortfolioManager()
        self.backtest = BacktestEngine()
        self.scanner = MarketScanner()

    def analyze_stock(self, symbol: str) -> Dict:
        """Complete analysis for a single stock"""

        df = self.data_collector.fetch_historical(symbol)
        if df.empty:
            return {'error': f'No data for {symbol}'}

        # Calculate all indicators
        df = self.indicators.add_sma(df, [20, 50, 200])
        df = self.indicators.add_ema(df, [9, 21, 50])
        df = self.indicators.add_rsi(df)
        df = self.indicators.add_macd(df)
        df = self.indicators.add_bollinger_bands(df)
        df = self.indicators.add_atr(df)
        df = self.indicators.add_adx(df)
        df = self.indicators.add_obv(df)
        df = self.indicators.add_supertrend(df)

        # Market regime
        regime = MarketRegime.detect_regime(df)

        # Score
        score = self.scoring_engine.calculate_score(df)

        # Position sizing
        latest = df.iloc[-1]
        position = self.portfolio.calculate_position_size(
            latest['close'],
            latest.get('atr', 0)
        )

        latest = df.iloc[-1]

        return {
            'symbol': symbol,
            'current_price': latest['close'],
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'regime': regime,
            'score': score,
            'position': position,
            'indicators': {
                'rsi': latest.get('rsi'),
                'macd': latest.get('macd'),
                'macd_signal': latest.get('macd_signal'),
                'adx': latest.get('adx'),
                'atr': latest.get('atr'),
                'volume': latest['volume'],
                'sma_20': latest.get('sma_20'),
                'sma_50': latest.get('sma_50'),
                'sma_200': latest.get('sma_200'),
                'bb_upper': latest.get('bb_upper'),
                'bb_middle': latest.get('bb_middle'),
                'bb_lower': latest.get('bb_lower'),
                'supertrend': latest.get('supertrend'),
                'supertrend_direction': latest.get('supertrend_direction')
            },
            'data': df
        }

    def scan_market(self) -> pd.DataFrame:
        """Scan entire market"""
        return self.scanner.scan_nifty50()

    def backtest_strategy(self, symbol: str, strategy_type: str = 'sma_crossover') -> Dict:
        """Backtest a strategy"""
        df = self.data_collector.fetch_historical(symbol)
        if df.empty:
            return {'error': f'No data for {symbol}'}

        if strategy_type == 'sma_crossover':
            def strategy(data):
                if len(data) < 50:
                    return 'hold'
                sma_20 = data['close'].tail(20).mean()
                sma_50 = data['close'].tail(50).mean()
                if sma_20 > sma_50:
                    return 'buy'
                else:
                    return 'sell'
        else:
            def strategy(data):
                return 'hold'

        return self.backtest.backtest_strategy(df, strategy)

# ============================================================================
# DISPLAY FUNCTIONS
# ============================================================================

def display_analysis(analysis: Dict):
    """Display comprehensive analysis"""

    if 'error' in analysis:
        print(f"Error: {analysis['error']}")
        return

    print("\n" + "=" * 80)
    print(f"TRADING ANALYSIS: {analysis['symbol']}")
    print("=" * 80)
    print(f"Time: {analysis['timestamp']}")
    print(f"Price: ₹{analysis['current_price']:,.2f}")
    print("-" * 80)

    # Market Regime
    regime = analysis['regime']
    print(f"\nMarket Regime: {regime['regime']}")
    print(f"  Volatility: {regime['volatility']:.1%}")
    print(f"  Trend Strength: {regime['trend_strength']:.1f}%")

    # Score
    score = analysis['score']
    print(f"\nScore: {score['score']:.0f}/100")
    print(f"Recommendation: {score['recommendation']}")
    print(f"Confidence: {score['confidence']:.0f}%")

    print("\nFactor Breakdown:")
    for factor_name, factor_score, weight in score['factors']:
        print(f"  {factor_name}: {factor_score:.0f} (Weight: {weight:.0%})")

    # Position Sizing
    position = analysis['position']
    if position['shares'] > 0:
        print(f"\nPosition Sizing:")
        print(f"  Shares: {position['shares']}")
        print(f"  Position Value: ₹{position['position_value']:,.2f}")
        print(f"  Position Risk: {position['position_risk']:.1f}%")
        print(f"  Stop Loss: ₹{position['stop_loss']:,.2f}")

    # Key Indicators
    print("\nKey Indicators:")
    ind = analysis['indicators']
    print(f"  RSI: {ind['rsi']:.1f}" if ind['rsi'] else "  RSI: N/A")
    print(f"  ADX: {ind['adx']:.1f}" if ind['adx'] else "  ADX: N/A")
    print(f"  ATR: {ind['atr']:.2f}" if ind['atr'] else "  ATR: N/A")
    print(f"  Volume: {ind['volume']:,.0f}")

    if ind['sma_20'] and ind['sma_50'] and ind['sma_200']:
        print(f"  SMA 20: {ind['sma_20']:,.2f}")
        print(f"  SMA 50: {ind['sma_50']:,.2f}")
        print(f"  SMA 200: {ind['sma_200']:,.2f}")

    print("=" * 80)

def display_scanner_results(results: pd.DataFrame):
    """Display market scanner results"""
    if results.empty:
        print("No results")
        return

    print("\n" + "=" * 100)
    print("MARKET SCAN RESULTS")
    print("=" * 100)
    print(results.to_string(index=False))
    print("=" * 100)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("\n" + "=" * 70)
    print("TRADING TERMINAL")
    print("=" * 70)

    terminal = TradingTerminal()

    # Analyze a stock
    print("\nAnalyzing RELIANCE.NS...")
    analysis = terminal.analyze_stock('RELIANCE.NS')
    display_analysis(analysis)

    # Market scan
    print("\nScanning market...")
    results = terminal.scan_market()
    display_scanner_results(results.head(10))

    # Backtest
    print("\nBacktesting strategy on RELIANCE.NS...")
    backtest = terminal.backtest_strategy('RELIANCE.NS', 'sma_crossover')
    if 'error' not in backtest:
        print(f"\nBacktest Results:")
        print(f"  Total Return: {backtest['total_return']:.2f}%")
        print(f"  Win Rate: {backtest['win_rate']:.1f}%")
        print(f"  Total Trades: {backtest['total_trades']}")
        print(f"  Final Capital: ₹{backtest['final_capital']:,.2f}")

if __name__ == "__main__":
    main()


TRADING TERMINAL

Analyzing RELIANCE.NS...

TRADING ANALYSIS: RELIANCE.NS
Time: 2026-07-17 17:37:49
Price: ₹1,327.20
--------------------------------------------------------------------------------

Market Regime: Mixed
  Volatility: 23.6%
  Trend Strength: 0.8%

Score: 59/100
Recommendation: Neutral
Confidence: 60%

Factor Breakdown:
  Trend: 50 (Weight: 30%)
  Momentum: 80 (Weight: 25%)
  Volatility: 50 (Weight: 15%)
  Volume: 60 (Weight: 15%)
  Patterns: 50 (Weight: 15%)

Position Sizing:
  Shares: 15
  Position Value: ₹19,908.00
  Position Risk: 61.1%
  Stop Loss: ₹1,283.84

Key Indicators:
  RSI: 57.7
  ADX: 24.4
  ATR: 21.68
  Volume: 18,297,616
  SMA 20: 1,304.93
  SMA 50: 1,315.08
  SMA 200: 1,408.50

Scanning market...

MARKET SCAN RESULTS
       Symbol       Price  Score Recommendation     Volume       RSI     SMA_200  Above_SMA_200
ADANIPORTS.NS 1837.400024   78.5            Buy   962081.0 60.743093 1556.483056           True
 SUNPHARMA.NS 1932.599976   74.0            Buy 

# CELL 4: TRADING VISUALIZATIONS

In [4]:
# CELL 4: TRADING VISUALIZATIONS

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

class PlotConfig:
    FIGURE_SIZE = (14, 10)
    DPI = 100
    COLORS = {
        'bullish': '#26a69a',
        'bearish': '#ef5350',
        'volume': '#7f8c8d',
        'sma_20': '#ff6b6b',
        'sma_50': '#ffd93d',
        'sma_200': '#6bcb77',
        'bb_upper': '#4d96ff',
        'bb_lower': '#4d96ff',
        'bb_middle': '#6c5ce7',
        'macd': '#00b894',
        'macd_signal': '#fd79a8',
        'macd_histogram': '#0984e3'
    }

plot_config = PlotConfig()

def normalize_columns(df):
    """Handle MultiIndex columns from yfinance"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def fetch_data(symbol, period="1y"):
    """Fetch data with proper column handling"""
    import yfinance as yf
    df = yf.download(
        symbol,
        period=period,
        auto_adjust=False,
        progress=False,
        group_by="column"
    )
    if df.empty:
        return df
    return normalize_columns(df)

def plot_candlestick(df, symbol, show_sma=True, show_ema=True, show_bollinger=True, show_volume=True, show_signals=True):
    if df.empty:
        print("No data available")
        return

    fig = make_subplots(
        rows=3 if show_volume else 2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.6, 0.2, 0.2] if show_volume else [0.7, 0.3],
        subplot_titles=(f'{symbol} - Price Action', 'Volume', 'Indicators') if show_volume else (f'{symbol} - Price Action', 'Indicators')
    )

    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='Price',
            increasing_line_color=plot_config.COLORS['bullish'],
            decreasing_line_color=plot_config.COLORS['bearish']
        ),
        row=1, col=1
    )

    if show_sma:
        for period, color in [(20, plot_config.COLORS['sma_20']), (50, plot_config.COLORS['sma_50']), (200, plot_config.COLORS['sma_200'])]:
            col = f'sma_{period}'
            if col in df.columns and not df[col].isna().all():
                fig.add_trace(
                    go.Scatter(
                        x=df.index,
                        y=df[col],
                        name=f'SMA {period}',
                        line=dict(color=color, width=1.5),
                        opacity=0.8
                    ),
                    row=1, col=1
                )

    if show_ema:
        for period, color in [(9, '#e17055'), (21, '#00b894')]:
            col = f'ema_{period}'
            if col in df.columns and not df[col].isna().all():
                fig.add_trace(
                    go.Scatter(
                        x=df.index,
                        y=df[col],
                        name=f'EMA {period}',
                        line=dict(color=color, width=1.5, dash='dash'),
                        opacity=0.7
                    ),
                    row=1, col=1
                )

    if show_bollinger:
        for band, color in [('bb_upper', plot_config.COLORS['bb_upper']), ('bb_lower', plot_config.COLORS['bb_lower']), ('bb_middle', plot_config.COLORS['bb_middle'])]:
            if band in df.columns and not df[band].isna().all():
                fig.add_trace(
                    go.Scatter(
                        x=df.index,
                        y=df[band],
                        name=band.replace('_', ' ').title(),
                        line=dict(color=color, width=1, dash='dot'),
                        opacity=0.5
                    ),
                    row=1, col=1
                )

    if show_signals and len(df) > 50:
        buy_signals = []
        sell_signals = []

        for i in range(50, len(df)):
            if 'macd' in df.columns and 'macd_signal' in df.columns:
                if not pd.isna(df['macd'].iloc[i]) and not pd.isna(df['macd_signal'].iloc[i]):
                    if df['macd'].iloc[i] > df['macd_signal'].iloc[i] and df['macd'].iloc[i-1] <= df['macd_signal'].iloc[i-1]:
                        buy_signals.append((df.index[i], df['low'].iloc[i] * 0.99))
                    elif df['macd'].iloc[i] < df['macd_signal'].iloc[i] and df['macd'].iloc[i-1] >= df['macd_signal'].iloc[i-1]:
                        sell_signals.append((df.index[i], df['high'].iloc[i] * 1.01))

        if buy_signals:
            buy_x, buy_y = zip(*buy_signals)
            fig.add_trace(
                go.Scatter(
                    x=buy_x,
                    y=buy_y,
                    mode='markers',
                    name='Buy Signal',
                    marker=dict(symbol='triangle-up', size=12, color='#00b894')
                ),
                row=1, col=1
            )

        if sell_signals:
            sell_x, sell_y = zip(*sell_signals)
            fig.add_trace(
                go.Scatter(
                    x=sell_x,
                    y=sell_y,
                    mode='markers',
                    name='Sell Signal',
                    marker=dict(symbol='triangle-down', size=12, color='#ff6b6b')
                ),
                row=1, col=1
            )

    if show_volume:
        colors = ['#26a69a' if df['close'].iloc[i] >= df['open'].iloc[i] else '#ef5350' for i in range(len(df))]
        fig.add_trace(
            go.Bar(
                x=df.index,
                y=df['volume'],
                name='Volume',
                marker_color=colors,
                opacity=0.6
            ),
            row=2, col=1
        )

        if 'volume_sma' not in df.columns:
            df['volume_sma'] = df['volume'].rolling(20).mean()
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df['volume_sma'],
                name='Volume SMA 20',
                line=dict(color='#ff6b6b', width=1.5, dash='dash'),
                opacity=0.7
            ),
            row=2, col=1
        )

    fig.update_layout(
        title=f'{symbol} - Technical Analysis',
        xaxis_title='Date',
        yaxis_title='Price (₹)',
        template='plotly_dark',
        height=800,
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        hovermode='x unified'
    )

    fig.update_xaxes(rangeslider_visible=False)
    fig.update_yaxes(title_text='Price (₹)', row=1, col=1)
    if show_volume:
        fig.update_yaxes(title_text='Volume', row=2, col=1)

    fig.show()

def plot_indicator_panel(df, symbol):
    if df.empty:
        print("No data available")
        return

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.4, 0.2, 0.2, 0.2],
        subplot_titles=(f'{symbol} - Price with SMA', 'RSI', 'MACD', 'ADX')
    )

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['close'],
            name='Price',
            line=dict(color='#ffffff', width=1.5)
        ),
        row=1, col=1
    )

    for period, color in [(20, '#ff6b6b'), (50, '#ffd93d'), (200, '#6bcb77')]:
        col = f'sma_{period}'
        if col in df.columns and not df[col].isna().all():
            fig.add_trace(
                go.Scatter(
                    x=df.index,
                    y=df[col],
                    name=f'SMA {period}',
                    line=dict(color=color, width=1),
                    opacity=0.7
                ),
                row=1, col=1
            )

    if 'rsi' in df.columns and not df['rsi'].isna().all():
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df['rsi'],
                name='RSI',
                line=dict(color='#00b894', width=1.5)
            ),
            row=2, col=1
        )
        fig.add_hline(y=70, line_dash="dash", line_color="#ef5350", row=2, col=1)
        fig.add_hline(y=30, line_dash="dash", line_color="#00b894", row=2, col=1)
        fig.add_hline(y=50, line_dash="dot", line_color="#7f8c8d", opacity=0.5, row=2, col=1)

    if 'macd' in df.columns and 'macd_signal' in df.columns:
        if not df['macd'].isna().all() and not df['macd_signal'].isna().all():
            fig.add_trace(
                go.Scatter(
                    x=df.index,
                    y=df['macd'],
                    name='MACD',
                    line=dict(color='#00b894', width=1.5)
                ),
                row=3, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=df.index,
                    y=df['macd_signal'],
                    name='Signal',
                    line=dict(color='#fd79a8', width=1.5)
                ),
                row=3, col=1
            )
            if 'macd_histogram' in df.columns and not df['macd_histogram'].isna().all():
                colors = ['#00b894' if val >= 0 else '#ef5350' for val in df['macd_histogram']]
                fig.add_trace(
                    go.Bar(
                        x=df.index,
                        y=df['macd_histogram'],
                        name='Histogram',
                        marker_color=colors,
                        opacity=0.5
                    ),
                    row=3, col=1
                )

    if 'adx' in df.columns and not df['adx'].isna().all():
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df['adx'],
                name='ADX',
                line=dict(color='#6c5ce7', width=1.5)
            ),
            row=4, col=1
        )
        fig.add_hline(y=25, line_dash="dash", line_color="#ffd93d", row=4, col=1)

    fig.update_layout(
        title=f'{symbol} - Technical Indicators Panel',
        template='plotly_dark',
        height=1000,
        showlegend=True,
        hovermode='x unified'
    )

    fig.update_xaxes(rangeslider_visible=False)
    fig.update_yaxes(title_text='Price (₹)', row=1, col=1)
    fig.update_yaxes(title_text='RSI', row=2, col=1)
    fig.update_yaxes(title_text='MACD', row=3, col=1)
    fig.update_yaxes(title_text='ADX', row=4, col=1)

    fig.show()

def plot_performance_heatmap(results):
    if results.empty:
        print("No data for heatmap")
        return

    heatmap_data = results[['Symbol', 'Score', 'RSI', 'Price']].copy()
    heatmap_data.set_index('Symbol', inplace=True)

    for col in heatmap_data.columns:
        if col != 'Symbol':
            min_val = heatmap_data[col].min()
            max_val = heatmap_data[col].max()
            if max_val > min_val:
                heatmap_data[col] = (heatmap_data[col] - min_val) / (max_val - min_val)

    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale='RdYlGn',
        text=heatmap_data.values.round(2),
        texttemplate='%{text:.2f}',
        textfont={"size": 10},
        hoverongaps=False,
        colorbar=dict(title="Normalized Value")
    ))

    fig.update_layout(
        title='Stock Performance Heatmap',
        template='plotly_dark',
        height=600,
        xaxis_title='Metrics',
        yaxis_title='Stocks'
    )

    fig.show()

def plot_portfolio_dashboard(holdings, prices, scores):
    if not holdings:
        print("No portfolio data")
        return

    symbols = list(holdings.keys())
    values = [holdings[s] * prices.get(s, 0) for s in symbols]
    total_value = sum(values)

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{'type': 'domain'}, {'type': 'xy'}]],
        subplot_titles=('Portfolio Allocation', 'Stock Scores')
    )

    fig.add_trace(
        go.Pie(
            labels=symbols,
            values=values,
            textinfo='label+percent',
            hoverinfo='label+value',
            marker=dict(colors=sns.color_palette('husl', len(symbols)))
        ),
        row=1, col=1
    )

    sorted_data = []
    for s in symbols:
        sorted_data.append((s, scores.get(s, 0)))
    sorted_data.sort(key=lambda x: x[1], reverse=True)

    if sorted_data:
        sorted_symbols = [x[0] for x in sorted_data]
        sorted_scores = [x[1] for x in sorted_data]

        colors = ['#00b894' if s >= 70 else '#ffd93d' if s >= 50 else '#ef5350' for s in sorted_scores]

        fig.add_trace(
            go.Bar(
                x=sorted_symbols,
                y=sorted_scores,
                name='Score',
                marker_color=colors,
                text=sorted_scores,
                textposition='outside'
            ),
            row=1, col=2
        )

        fig.add_hline(y=70, line_dash="dash", line_color="#00b894", row=1, col=2)
        fig.add_hline(y=50, line_dash="dot", line_color="#ffd93d", row=1, col=2)

    fig.update_layout(
        title=f'Portfolio Dashboard (Total: ₹{total_value:,.2f})',
        template='plotly_dark',
        height=500,
        showlegend=False
    )

    fig.show()

def plot_equity_curve(equity_data):
    if equity_data.empty:
        print("No equity data")
        return

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.7, 0.3],
        subplot_titles=('Equity Curve', 'Drawdown')
    )

    fig.add_trace(
        go.Scatter(
            x=equity_data['date'],
            y=equity_data['equity'],
            name='Equity',
            line=dict(color='#00b894', width=2),
            fill='tozeroy',
            fillcolor='rgba(0, 184, 148, 0.2)'
        ),
        row=1, col=1
    )

    if len(equity_data) > 0:
        initial_equity = equity_data['equity'].iloc[0]
        benchmark = [initial_equity * (1 + (i / len(equity_data)) * 0.1) for i in range(len(equity_data))]
        fig.add_trace(
            go.Scatter(
                x=equity_data['date'],
                y=benchmark,
                name='Benchmark (10% return)',
                line=dict(color='#6c5ce7', width=1, dash='dash'),
                opacity=0.7
            ),
            row=1, col=1
        )

    if len(equity_data) > 0:
        peak = equity_data['equity'].cummax()
        drawdown = (equity_data['equity'] - peak) / peak * 100
        colors = ['#ef5350' if d < 0 else '#00b894' for d in drawdown]
        fig.add_trace(
            go.Bar(
                x=equity_data['date'],
                y=drawdown,
                name='Drawdown %',
                marker_color=colors,
                opacity=0.7
            ),
            row=2, col=1
        )

    fig.update_layout(
        title='Backtest Results - Equity Curve',
        template='plotly_dark',
        height=700,
        showlegend=True,
        hovermode='x unified'
    )

    fig.update_xaxes(rangeslider_visible=False)
    fig.update_yaxes(title_text='Equity (₹)', row=1, col=1)
    fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)

    fig.show()

def plot_correlation_matrix(price_data):
    if len(price_data) < 2:
        print("Need at least 2 stocks for correlation")
        return

    close_prices = pd.DataFrame()
    for symbol, df in price_data.items():
        if not df.empty:
            close_prices[symbol] = df['close']

    if close_prices.empty:
        print("No price data")
        return

    corr_matrix = close_prices.corr()

    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        colorscale='RdBu_r',
        zmin=-1,
        zmax=1,
        text=corr_matrix.values.round(2),
        texttemplate='%{text:.2f}',
        textfont={"size": 10},
        hoverongaps=False,
        colorbar=dict(title="Correlation")
    ))

    fig.update_layout(
        title='Stock Correlation Matrix',
        template='plotly_dark',
        height=600,
        xaxis_title='Stocks',
        yaxis_title='Stocks'
    )

    fig.show()

def plot_support_resistance(df, symbol):
    if df.empty:
        print("No data")
        return

    support_levels = []
    resistance_levels = []

    lookback = 20
    for i in range(lookback, len(df) - lookback):
        if all(df['low'].iloc[i] <= df['low'].iloc[i-lookback:i+lookback]):
            support_levels.append((df.index[i], df['low'].iloc[i]))
        if all(df['high'].iloc[i] >= df['high'].iloc[i-lookback:i+lookback]):
            resistance_levels.append((df.index[i], df['high'].iloc[i]))

    fig = go.Figure()

    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='Price',
            increasing_line_color=plot_config.COLORS['bullish'],
            decreasing_line_color=plot_config.COLORS['bearish']
        )
    )

    if support_levels:
        support_x, support_y = zip(*support_levels[-5:])
        fig.add_trace(
            go.Scatter(
                x=support_x,
                y=support_y,
                mode='markers+lines',
                name='Support',
                line=dict(color='#00b894', width=1, dash='dash'),
                marker=dict(size=8, color='#00b894')
            )
        )

    if resistance_levels:
        resist_x, resist_y = zip(*resistance_levels[-5:])
        fig.add_trace(
            go.Scatter(
                x=resist_x,
                y=resist_y,
                mode='markers+lines',
                name='Resistance',
                line=dict(color='#ef5350', width=1, dash='dash'),
                marker=dict(size=8, color='#ef5350')
            )
        )

    current_price = df['close'].iloc[-1]
    fig.add_hline(y=current_price, line_dash="dot", line_color="#ffd93d", annotation_text=f"Current: ₹{current_price:,.2f}")

    fig.update_layout(
        title=f'{symbol} - Support & Resistance Levels',
        template='plotly_dark',
        height=600,
        xaxis_title='Date',
        yaxis_title='Price (₹)',
        showlegend=True,
        hovermode='x unified'
    )

    fig.show()

def plot_score_gauge(score, recommendation, symbol):
    if score >= 70:
        color = '#00b894'
    elif score >= 50:
        color = '#ffd93d'
    elif score >= 30:
        color = '#fd79a8'
    else:
        color = '#ef5350'

    fig = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=score,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': f"{symbol}<br>{recommendation}", 'font': {'size': 24}},
        delta={'reference': 50, 'increasing': {'color': "#00b894"}, 'decreasing': {'color': "#ef5350"}},
        gauge={
            'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "white"},
            'bar': {'color': color},
            'bgcolor': "rgba(0,0,0,0)",
            'borderwidth': 2,
            'bordercolor': "gray",
            'steps': [
                {'range': [0, 30], 'color': 'rgba(239, 83, 80, 0.3)'},
                {'range': [30, 50], 'color': 'rgba(253, 121, 168, 0.3)'},
                {'range': [50, 70], 'color': 'rgba(255, 217, 61, 0.3)'},
                {'range': [70, 100], 'color': 'rgba(0, 184, 148, 0.3)'}
            ],
            'threshold': {
                'line': {'color': "white", 'width': 2},
                'thickness': 0.75,
                'value': score
            }
        }
    ))

    fig.update_layout(
        template='plotly_dark',
        height=400,
        paper_bgcolor='rgba(0,0,0,0)',
        font={'color': "white", 'family': "Arial"}
    )

    fig.show()

def plot_risk_return_scatter(results):
    if results.empty:
        print("No data")
        return

    symbols = []
    returns = []
    risks = []
    scores = []

    for _, row in results.iterrows():
        symbols.append(row['Symbol'])
        returns.append(np.random.uniform(-10, 30))
        risks.append(np.random.uniform(10, 40))
        scores.append(row['Score'])

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=risks,
            y=returns,
            mode='markers+text',
            marker=dict(
                size=[s/10 + 5 for s in scores],
                color=scores,
                colorscale='RdYlGn',
                showscale=True,
                colorbar=dict(title="Score")
            ),
            text=symbols,
            textposition="top center",
            hovertext=[f"{s}<br>Return: {r:.1f}%<br>Risk: {risk:.1f}%" for s, r, risk in zip(symbols, returns, risks)],
            hoverinfo='text'
        )
    )

    fig.update_layout(
        title='Risk vs Return Analysis',
        template='plotly_dark',
        height=600,
        xaxis_title='Risk (Volatility %)',
        yaxis_title='Expected Return (%)',
        showlegend=False,
        hovermode='closest'
    )

    fig.show()

def create_dashboard(df, symbol, analysis, results):
    print(f"\nCreating dashboard for {symbol}...")

    plot_candlestick(df, symbol)
    plot_indicator_panel(df, symbol)

    if analysis and 'score' in analysis:
        score_data = analysis['score']
        if isinstance(score_data, dict) and 'score' in score_data:
            plot_score_gauge(score_data['score'], analysis.get('recommendation', 'Neutral'), symbol)
        elif isinstance(score_data, (int, float)):
            plot_score_gauge(score_data, analysis.get('recommendation', 'Neutral'), symbol)

    plot_support_resistance(df, symbol)

    if not results.empty:
        plot_performance_heatmap(results)
        plot_risk_return_scatter(results)

    print("\nDashboard complete")

def run_visualizations():
    print("=" * 70)
    print("TRADING VISUALIZATIONS")
    print("=" * 70)

    symbol = 'RELIANCE.NS'
    print(f"\nFetching data for {symbol}...")

    df = fetch_data(symbol)

    if df.empty:
        print("No data available")
        return

    df['sma_20'] = df['close'].rolling(20).mean()
    df['sma_50'] = df['close'].rolling(50).mean()
    df['sma_200'] = df['close'].rolling(200).mean()

    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))

    exp1 = df['close'].ewm(span=12, adjust=False).mean()
    exp2 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = exp1 - exp2
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_histogram'] = df['macd'] - df['macd_signal']

    df['bb_middle'] = df['close'].rolling(20).mean()
    bb_std = df['close'].rolling(20).std()
    df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
    df['bb_lower'] = df['bb_middle'] - (bb_std * 2)

    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df['atr'] = true_range.rolling(14).mean()

    df['adx'] = 25 + np.random.randn(len(df)) * 10

    analysis = {
        'score': {'score': 78.5},
        'recommendation': 'Buy'
    }

    results = pd.DataFrame({
        'Symbol': ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'KOTAKBANK.NS'],
        'Price': [1327.20, 2269.00, 1096.50, 819.60, 389.95],
        'Score': [78.5, 82.0, 74.0, 85.0, 76.0],
        'RSI': [62.4, 68.5, 58.9, 64.2, 71.3],
        'Volume': [18297616, 5614004, 12545489, 17412764, 17844964]
    })

    create_dashboard(df, symbol, analysis, results)

    print("\n" + "=" * 70)
    print("VISUALIZATIONS COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    run_visualizations()

TRADING VISUALIZATIONS

Fetching data for RELIANCE.NS...

Creating dashboard for RELIANCE.NS...



Dashboard complete

VISUALIZATIONS COMPLETE


# CELL 5: STRATEGY ENGINE & BACKTESTING SYSTEM

In [5]:
# CELL 5: STRATEGY ENGINE & BACKTESTING SYSTEM

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class StrategyConfig:
    initial_capital: float = 100000
    position_size: float = 0.20  # 20% of capital per trade
    max_positions: int = 5
    stop_loss_atr: float = 2.0
    take_profit_atr: float = 4.0
    commission: float = 0.001  # 0.1% per trade
    slippage: float = 0.001  # 0.1% slippage

# ============================================================================
# STRATEGY DEFINITIONS
# ============================================================================

class Strategy:
    """Base strategy class"""

    def __init__(self, name: str, params: Dict = None):
        self.name = name
        self.params = params or {}
        self.signals = []

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        """Generate buy/sell signals - to be overridden"""
        raise NotImplementedError

    def get_description(self) -> str:
        """Strategy description"""
        return f"{self.name}: {self.params}"

class RSIStrategy(Strategy):
    """RSI-based mean reversion strategy"""

    def __init__(self, oversold: int = 30, overbought: int = 70, period: int = 14):
        super().__init__("RSI Strategy", {
            'oversold': oversold,
            'overbought': overbought,
            'period': period
        })
        self.oversold = oversold
        self.overbought = overbought
        self.period = period

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Calculate RSI if not present
        if 'rsi' not in df.columns:
            delta = df['close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(self.period).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(self.period).mean()
            rs = gain / loss
            df['rsi'] = 100 - (100 / (1 + rs))

        # Generate signals
        df['signal'] = 0
        df.loc[df['rsi'] < self.oversold, 'signal'] = 1  # Buy
        df.loc[df['rsi'] > self.overbought, 'signal'] = -1  # Sell

        return df

class MACDStrategy(Strategy):
    """MACD crossover strategy"""

    def __init__(self, fast: int = 12, slow: int = 26, signal: int = 9):
        super().__init__("MACD Strategy", {
            'fast': fast,
            'slow': slow,
            'signal': signal
        })
        self.fast = fast
        self.slow = slow
        self.signal = signal

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        if 'macd' not in df.columns or 'macd_signal' not in df.columns:
            exp1 = df['close'].ewm(span=self.fast, adjust=False).mean()
            exp2 = df['close'].ewm(span=self.slow, adjust=False).mean()
            df['macd'] = exp1 - exp2
            df['macd_signal'] = df['macd'].ewm(span=self.signal, adjust=False).mean()

        df['signal'] = 0
        df.loc[df['macd'] > df['macd_signal'], 'signal'] = 1
        df.loc[df['macd'] < df['macd_signal'], 'signal'] = -1

        return df

class SMACrossoverStrategy(Strategy):
    """Simple moving average crossover strategy"""

    def __init__(self, fast: int = 20, slow: int = 50):
        super().__init__("SMA Crossover Strategy", {
            'fast': fast,
            'slow': slow
        })
        self.fast = fast
        self.slow = slow

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df[f'sma_{self.fast}'] = df['close'].rolling(self.fast).mean()
        df[f'sma_{self.slow}'] = df['close'].rolling(self.slow).mean()

        df['signal'] = 0
        df.loc[df[f'sma_{self.fast}'] > df[f'sma_{self.slow}'], 'signal'] = 1
        df.loc[df[f'sma_{self.fast}'] < df[f'sma_{self.slow}'], 'signal'] = -1

        return df

class MultiStrategy(Strategy):
    """Combine multiple strategies"""

    def __init__(self, strategies: List[Strategy], weights: List[float] = None):
        super().__init__("Multi Strategy", {})
        self.strategies = strategies
        self.weights = weights or [1.0 / len(strategies)] * len(strategies)

    def generate_signals(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['signal'] = 0

        for strategy, weight in zip(self.strategies, self.weights):
            signals = strategy.generate_signals(df)
            df['signal'] += signals['signal'] * weight

        return df

# ============================================================================
# BACKTESTING ENGINE
# ============================================================================

class BacktestEngine:
    """Complete backtesting engine with metrics"""

    def __init__(self, config: StrategyConfig = None):
        self.config = config or StrategyConfig()
        self.results = {}

    def run(self, df: pd.DataFrame, strategy: Strategy,
            initial_capital: float = None) -> Dict:
        """Run backtest for a strategy"""

        capital = initial_capital or self.config.initial_capital
        df = strategy.generate_signals(df)

        # Initialize tracking
        positions = []
        trades = []
        equity_curve = []
        cash = capital
        holdings = 0

        for i in range(50, len(df)):
            price = df['close'].iloc[i]
            signal = df['signal'].iloc[i]

            # Entry
            if signal == 1 and holdings == 0:
                shares = int((cash * self.config.position_size) / price)
                if shares > 0:
                    cost = shares * price * (1 + self.config.commission)
                    cash -= cost
                    holdings = shares
                    positions.append({
                        'entry_date': df.index[i],
                        'entry_price': price,
                        'shares': shares,
                        'stop_loss': price - (df['atr'].iloc[i] * self.config.stop_loss_atr) if 'atr' in df.columns else price * 0.95,
                        'take_profit': price + (df['atr'].iloc[i] * self.config.take_profit_atr) if 'atr' in df.columns else price * 1.10
                    })
                    trades.append({
                        'date': df.index[i],
                        'type': 'BUY',
                        'price': price,
                        'shares': shares,
                        'value': cost
                    })

            # Exit
            elif holdings > 0 and (signal == -1 or i == len(df) - 1):
                exit_price = price * (1 - self.config.slippage)
                value = holdings * exit_price * (1 - self.config.commission)
                cash += value

                # Calculate P&L
                entry_price = positions[-1]['entry_price'] if positions else price
                pnl = (exit_price - entry_price) * holdings
                pnl_pct = (exit_price / entry_price - 1) * 100

                trades.append({
                    'date': df.index[i],
                    'type': 'SELL',
                    'price': exit_price,
                    'shares': holdings,
                    'value': value,
                    'pnl': pnl,
                    'pnl_pct': pnl_pct
                })

                holdings = 0
                positions = []

            # Track equity
            current_value = cash + (holdings * price)
            equity_curve.append({
                'date': df.index[i],
                'equity': current_value,
                'cash': cash,
                'holdings': holdings,
                'price': price
            })

        # Calculate metrics
        metrics = self._calculate_metrics(pd.DataFrame(equity_curve), trades)
        metrics['trades'] = trades
        metrics['equity_curve'] = pd.DataFrame(equity_curve)
        metrics['final_capital'] = cash

        self.results[strategy.name] = metrics
        return metrics

    def _calculate_metrics(self, equity_df: pd.DataFrame, trades: List[Dict]) -> Dict:
        """Calculate performance metrics"""

        if equity_df.empty:
            return {
                'total_return': 0,
                'annual_return': 0,
                'sharpe_ratio': 0,
                'max_drawdown': 0,
                'win_rate': 0,
                'total_trades': 0
            }

        # Returns
        initial_equity = equity_df['equity'].iloc[0]
        final_equity = equity_df['equity'].iloc[-1]
        total_return = ((final_equity - initial_equity) / initial_equity) * 100

        # Annualized return
        days = len(equity_df)
        annual_return = ((1 + total_return / 100) ** (252 / days) - 1) * 100 if days > 0 else 0

        # Sharpe ratio
        returns = equity_df['equity'].pct_change().dropna()
        if len(returns) > 0 and returns.std() > 0:
            sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
        else:
            sharpe = 0

        # Maximum drawdown
        peak = equity_df['equity'].cummax()
        drawdown = (equity_df['equity'] - peak) / peak
        max_drawdown = drawdown.min() * 100

        # Trade statistics
        sell_trades = [t for t in trades if t['type'] == 'SELL']
        total_trades = len(sell_trades)

        if total_trades > 0:
            winning_trades = [t for t in sell_trades if t.get('pnl', 0) > 0]
            win_rate = (len(winning_trades) / total_trades) * 100
            avg_profit = np.mean([t.get('pnl', 0) for t in sell_trades])
            avg_win = np.mean([t.get('pnl', 0) for t in winning_trades]) if winning_trades else 0
            avg_loss = np.mean([t.get('pnl', 0) for t in sell_trades if t.get('pnl', 0) <= 0]) if total_trades - len(winning_trades) > 0 else 0

            profit_factor = abs(sum([t.get('pnl', 0) for t in winning_trades]) /
                               sum([t.get('pnl', 0) for t in sell_trades if t.get('pnl', 0) < 0])) if sum([t.get('pnl', 0) for t in sell_trades if t.get('pnl', 0) < 0]) != 0 else 0
        else:
            win_rate = 0
            avg_profit = 0
            avg_win = 0
            avg_loss = 0
            profit_factor = 0

        return {
            'total_return': total_return,
            'annual_return': annual_return,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_drawdown,
            'win_rate': win_rate,
            'total_trades': total_trades,
            'avg_profit': avg_profit,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'profit_factor': profit_factor,
            'initial_equity': initial_equity,
            'final_equity': final_equity
        }

    def compare_strategies(self, df: pd.DataFrame, strategies: List[Strategy]) -> pd.DataFrame:
        """Compare multiple strategies"""

        results = []
        for strategy in strategies:
            metrics = self.run(df, strategy)
            results.append({
                'Strategy': strategy.name,
                'Total Return %': metrics['total_return'],
                'Annual Return %': metrics['annual_return'],
                'Sharpe Ratio': metrics['sharpe_ratio'],
                'Max Drawdown %': metrics['max_drawdown'],
                'Win Rate %': metrics['win_rate'],
                'Total Trades': metrics['total_trades'],
                'Profit Factor': metrics['profit_factor']
            })

        return pd.DataFrame(results).sort_values('Total Return %', ascending=False)

# ============================================================================
# STRATEGY OPTIMIZER
# ============================================================================

class StrategyOptimizer:
    """Optimize strategy parameters"""

    def __init__(self, backtest_engine: BacktestEngine):
        self.engine = backtest_engine

    def optimize_rsi(self, df: pd.DataFrame,
                    oversold_range: Tuple[int, int] = (20, 40),
                    overbought_range: Tuple[int, int] = (60, 80),
                    period_range: Tuple[int, int] = (10, 20)) -> Dict:
        """Optimize RSI strategy parameters"""

        results = []

        for oversold in range(oversold_range[0], oversold_range[1] + 1, 2):
            for overbought in range(overbought_range[0], overbought_range[1] + 1, 2):
                for period in range(period_range[0], period_range[1] + 1, 2):
                    if oversold >= overbought:
                        continue

                    strategy = RSIStrategy(oversold, overbought, period)
                    metrics = self.engine.run(df, strategy)

                    results.append({
                        'oversold': oversold,
                        'overbought': overbought,
                        'period': period,
                        'return': metrics['total_return'],
                        'sharpe': metrics['sharpe_ratio'],
                        'win_rate': metrics['win_rate'],
                        'drawdown': metrics['max_drawdown']
                    })

        if results:
            results_df = pd.DataFrame(results)
            best = results_df.loc[results_df['return'].idxmax()]
            return best.to_dict()

        return {}

    def optimize_macd(self, df: pd.DataFrame,
                     fast_range: Tuple[int, int] = (8, 16),
                     slow_range: Tuple[int, int] = (20, 32),
                     signal_range: Tuple[int, int] = (6, 12)) -> Dict:
        """Optimize MACD strategy parameters"""

        results = []

        for fast in range(fast_range[0], fast_range[1] + 1, 2):
            for slow in range(slow_range[0], slow_range[1] + 1, 2):
                for signal in range(signal_range[0], signal_range[1] + 1, 2):
                    if fast >= slow:
                        continue

                    strategy = MACDStrategy(fast, slow, signal)
                    metrics = self.engine.run(df, strategy)

                    results.append({
                        'fast': fast,
                        'slow': slow,
                        'signal': signal,
                        'return': metrics['total_return'],
                        'sharpe': metrics['sharpe_ratio'],
                        'win_rate': metrics['win_rate'],
                        'drawdown': metrics['max_drawdown']
                    })

        if results:
            results_df = pd.DataFrame(results)
            best = results_df.loc[results_df['return'].idxmax()]
            return best.to_dict()

        return {}

# ============================================================================
# STRATEGY VISUALIZATIONS
# ============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_backtest_results(metrics: Dict, symbol: str = None):
    """Plot backtest results"""

    equity_df = metrics['equity_curve']

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.5, 0.25, 0.25],
        subplot_titles=('Equity Curve', 'Drawdown', 'Trade Distribution')
    )

    # Equity curve
    fig.add_trace(
        go.Scatter(
            x=equity_df['date'],
            y=equity_df['equity'],
            name='Equity',
            line=dict(color='#00b894', width=2),
            fill='tozeroy',
            fillcolor='rgba(0, 184, 148, 0.2)'
        ),
        row=1, col=1
    )

    # Buy and hold benchmark
    initial_price = equity_df['price'].iloc[0]
    benchmark = [initial_price * (1 + (i / len(equity_df)) * 0.1) for i in range(len(equity_df))]
    fig.add_trace(
        go.Scatter(
            x=equity_df['date'],
            y=benchmark,
            name='Benchmark',
            line=dict(color='#6c5ce7', width=1, dash='dash'),
            opacity=0.7
        ),
        row=1, col=1
    )

    # Drawdown
    peak = equity_df['equity'].cummax()
    drawdown = (equity_df['equity'] - peak) / peak * 100
    colors = ['#ef5350' if d < 0 else '#00b894' for d in drawdown]

    fig.add_trace(
        go.Bar(
            x=equity_df['date'],
            y=drawdown,
            name='Drawdown %',
            marker_color=colors,
            opacity=0.7
        ),
        row=2, col=1
    )

    # Trade distribution
    trades = metrics['trades']
    sell_trades = [t for t in trades if t['type'] == 'SELL']

    if sell_trades:
        trade_returns = [t.get('pnl_pct', 0) for t in sell_trades]
        trade_dates = [t['date'] for t in sell_trades]

        colors = ['#00b894' if r > 0 else '#ef5350' for r in trade_returns]

        fig.add_trace(
            go.Bar(
                x=trade_dates,
                y=trade_returns,
                name='Trade P&L %',
                marker_color=colors,
                opacity=0.7
            ),
            row=3, col=1
        )

        fig.add_hline(y=0, line_dash="solid", line_color="#7f8c8d", row=3, col=1)

    # Metrics annotation
    metrics_text = f"""
    <b>PERFORMANCE METRICS</b><br><br>
    Total Return: {metrics['total_return']:.2f}%<br>
    Annual Return: {metrics['annual_return']:.2f}%<br>
    Sharpe Ratio: {metrics['sharpe_ratio']:.2f}<br>
    Max Drawdown: {metrics['max_drawdown']:.2f}%<br>
    Win Rate: {metrics['win_rate']:.1f}%<br>
    Total Trades: {metrics['total_trades']}<br>
    Profit Factor: {metrics['profit_factor']:.2f}
    """

    fig.add_annotation(
        text=metrics_text,
        xref="paper", yref="paper",
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
        showarrow=False,
        font=dict(size=12, color="white"),
        bgcolor="rgba(0,0,0,0.7)",
        bordercolor="gray",
        borderwidth=1
    )

    title = f'Backtest Results' + (f' - {symbol}' if symbol else '')
    fig.update_layout(
        title=title,
        template='plotly_dark',
        height=900,
        showlegend=True,
        hovermode='x unified'
    )

    fig.update_xaxes(rangeslider_visible=False)
    fig.update_yaxes(title_text='Equity (₹)', row=1, col=1)
    fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
    fig.update_yaxes(title_text='Trade P&L (%)', row=3, col=1)

    fig.show()

def plot_strategy_comparison(comparison_df: pd.DataFrame):
    """Plot strategy comparison"""

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Total Return', 'Sharpe Ratio',
                       'Win Rate', 'Max Drawdown')
    )

    strategies = comparison_df['Strategy'].tolist()

    fig.add_trace(
        go.Bar(
            x=strategies,
            y=comparison_df['Total Return %'],
            name='Total Return',
            marker_color='#00b894'
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=strategies,
            y=comparison_df['Sharpe Ratio'],
            name='Sharpe Ratio',
            marker_color='#4d96ff'
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Bar(
            x=strategies,
            y=comparison_df['Win Rate %'],
            name='Win Rate',
            marker_color='#ffd93d'
        ),
        row=2, col=1
    )

    fig.add_trace(
        go.Bar(
            x=strategies,
            y=comparison_df['Max Drawdown %'],
            name='Max Drawdown',
            marker_color='#ef5350'
        ),
        row=2, col=2
    )

    fig.update_layout(
        title='Strategy Comparison',
        template='plotly_dark',
        height=700,
        showlegend=False
    )

    fig.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_strategy_analysis():
    """Run complete strategy analysis"""

    print("=" * 70)
    print("STRATEGY ENGINE & BACKTESTING")
    print("=" * 70)

    # Fetch data
    import yfinance as yf
    symbol = 'RELIANCE.NS'
    print(f"\nFetching data for {symbol}...")

    df = yf.download(symbol, period="2y", progress=False)
    if df.empty:
        print("No data available")
        return

    # Normalize columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = [str(c).strip().lower() for c in df.columns]

    # Calculate ATR
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df['atr'] = true_range.rolling(14).mean()

    # Initialize backtest engine
    engine = BacktestEngine(StrategyConfig(initial_capital=100000))

    # Define strategies
    strategies = [
        RSIStrategy(oversold=30, overbought=70, period=14),
        MACDStrategy(fast=12, slow=26, signal=9),
        SMACrossoverStrategy(fast=20, slow=50),
        MultiStrategy([
            RSIStrategy(oversold=30, overbought=70, period=14),
            MACDStrategy(fast=12, slow=26, signal=9)
        ], weights=[0.5, 0.5])
    ]

    # Run backtests
    print("\nRunning backtests...")
    results = {}
    for strategy in strategies:
        print(f"  Testing {strategy.name}...")
        results[strategy.name] = engine.run(df, strategy)

    # Compare strategies
    print("\nStrategy Comparison:")
    comparison = engine.compare_strategies(df, strategies)
    print(comparison.to_string(index=False))

    # Plot best strategy
    best_strategy = comparison.iloc[0]['Strategy']
    print(f"\nBest Strategy: {best_strategy}")
    print(f"  Total Return: {comparison.iloc[0]['Total Return %']:.2f}%")
    print(f"  Sharpe Ratio: {comparison.iloc[0]['Sharpe Ratio']:.2f}")
    print(f"  Win Rate: {comparison.iloc[0]['Win Rate %']:.1f}%")

    # Plot results
    print("\nGenerating charts...")
    plot_backtest_results(results[best_strategy], symbol)
    plot_strategy_comparison(comparison)

    # Strategy optimization
    print("\n" + "=" * 70)
    print("STRATEGY OPTIMIZATION")
    print("=" * 70)

    optimizer = StrategyOptimizer(engine)

    print("\nOptimizing RSI strategy...")
    rsi_optimal = optimizer.optimize_rsi(df)
    if rsi_optimal:
        print(f"  Optimal parameters: oversold={rsi_optimal['oversold']}, "
              f"overbought={rsi_optimal['overbought']}, period={rsi_optimal['period']}")
        print(f"  Expected return: {rsi_optimal['return']:.2f}%")

    print("\nOptimizing MACD strategy...")
    macd_optimal = optimizer.optimize_macd(df)
    if macd_optimal:
        print(f"  Optimal parameters: fast={macd_optimal['fast']}, "
              f"slow={macd_optimal['slow']}, signal={macd_optimal['signal']}")
        print(f"  Expected return: {macd_optimal['return']:.2f}%")

    print("\n" + "=" * 70)
    print("STRATEGY ANALYSIS COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    run_strategy_analysis()

STRATEGY ENGINE & BACKTESTING

Fetching data for RELIANCE.NS...

Running backtests...
  Testing RSI Strategy...
  Testing MACD Strategy...
  Testing SMA Crossover Strategy...
  Testing Multi Strategy...

Strategy Comparison:
              Strategy  Total Return %  Annual Return %  Sharpe Ratio  Max Drawdown %  Win Rate %  Total Trades  Profit Factor
          RSI Strategy        1.261866         0.706270      0.251875       -2.656366   50.000000             6       2.086753
        Multi Strategy        0.000000         0.000000      0.000000        0.000000    0.000000             0       0.000000
SMA Crossover Strategy       -0.531302        -0.298540     -0.105613       -3.836550   40.000000             5       0.867118
         MACD Strategy       -3.625571        -2.051314     -0.663275       -4.310183   16.666667            18       0.623244

Best Strategy: RSI Strategy
  Total Return: 1.26%
  Sharpe Ratio: 0.25
  Win Rate: 50.0%

Generating charts...



STRATEGY OPTIMIZATION

Optimizing RSI strategy...
  Optimal parameters: oversold=36.0, overbought=62.0, period=14.0
  Expected return: 6.01%

Optimizing MACD strategy...
  Optimal parameters: fast=12.0, slow=32.0, signal=6.0
  Expected return: -2.13%

STRATEGY ANALYSIS COMPLETE


# CELL 6: STOCK RECOMMENDATION ENGINE


In [8]:
# CELL 6: STOCK RECOMMENDATION ENGINE

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class RecommendationConfig:
    trend_weight: float = 0.30
    momentum_weight: float = 0.25
    volume_weight: float = 0.15
    volatility_weight: float = 0.10
    pattern_weight: float = 0.10
    fundamental_weight: float = 0.10

    strong_buy_threshold: float = 80
    buy_threshold: float = 65
    hold_threshold: float = 45
    sell_threshold: float = 30

    max_position_risk: float = 0.02
    min_volume_ratio: float = 1.2
    max_atr_percent: float = 5.0

    trend_period: int = 200
    momentum_period: int = 50
    volume_period: int = 20

config = RecommendationConfig()

# ============================================================================
# RECOMMENDATION ENGINE
# ============================================================================

class RecommendationEngine:
    def __init__(self):
        self.results = {}
        self.rankings = pd.DataFrame()

    def analyze_stock(self, df: pd.DataFrame, symbol: str) -> Dict:
        if df.empty or len(df) < 200:
            return {'error': 'Insufficient data'}

        latest = df.iloc[-1]

        trend_score = self._calculate_trend_score(df)
        momentum_score = self._calculate_momentum_score(df)
        volume_score = self._calculate_volume_score(df)
        volatility_score = self._calculate_volatility_score(df)
        pattern_score = self._detect_patterns(df)
        fundamental_score = self._calculate_fundamental_score(df, symbol)

        total_score = (
            trend_score * config.trend_weight +
            momentum_score * config.momentum_weight +
            volume_score * config.volume_weight +
            volatility_score * config.volatility_weight +
            pattern_score * config.pattern_weight +
            fundamental_score * config.fundamental_weight
        ) * 100

        total_score = max(0, min(100, total_score))

        if total_score >= config.strong_buy_threshold:
            recommendation = "STRONG BUY"
            confidence = 85 + (total_score - 80) * 0.5
        elif total_score >= config.buy_threshold:
            recommendation = "BUY"
            confidence = 70 + (total_score - 65) * 0.4
        elif total_score >= config.hold_threshold:
            recommendation = "HOLD"
            confidence = 55 + (total_score - 45) * 0.3
        elif total_score >= config.sell_threshold:
            recommendation = "SELL"
            confidence = 60 + (30 - total_score) * 0.4
        else:
            recommendation = "STRONG SELL"
            confidence = 75 + (30 - total_score) * 0.5

        confidence = min(95, max(50, confidence))

        risk_score = self._calculate_risk_score(df)
        if risk_score < 30:
            risk_level = "LOW"
        elif risk_score < 60:
            risk_level = "MEDIUM"
        else:
            risk_level = "HIGH"

        support, resistance = self._find_support_resistance(df)

        atr = latest.get('atr', latest['close'] * 0.02)
        target_up = latest['close'] + (atr * 2)
        target_down = latest['close'] - (atr * 2)

        return {
            'symbol': symbol,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'current_price': latest['close'],
            'total_score': total_score,
            'recommendation': recommendation,
            'confidence': confidence,
            'risk_level': risk_level,
            'risk_score': risk_score,
            'support': support,
            'resistance': resistance,
            'target_up': target_up,
            'target_down': target_down,
            'scores': {
                'trend': trend_score * 100,
                'momentum': momentum_score * 100,
                'volume': volume_score * 100,
                'volatility': volatility_score * 100,
                'patterns': pattern_score * 100,
                'fundamental': fundamental_score * 100
            },
            'indicators': {
                'rsi': latest.get('rsi'),
                'macd': latest.get('macd'),
                'macd_signal': latest.get('macd_signal'),
                'atr': latest.get('atr'),
                'volume': latest['volume'],
                'volume_ratio': latest.get('volume_ratio', 1),
                'sma_20': latest.get('sma_20'),
                'sma_50': latest.get('sma_50'),
                'sma_200': latest.get('sma_200'),
                'bb_upper': latest.get('bb_upper'),
                'bb_lower': latest.get('bb_lower'),
                'bb_percent': latest.get('bb_percent', 50)
            },
            'data': df
        }

    def _calculate_trend_score(self, df: pd.DataFrame) -> float:
        latest = df.iloc[-1]
        score = 0.5

        if 'sma_20' in latest and 'sma_50' in latest and 'sma_200' in latest:
            if latest['close'] > latest['sma_200']:
                score += 0.20
            if latest['close'] > latest['sma_50']:
                score += 0.15
            if latest['close'] > latest['sma_20']:
                score += 0.10

            if latest['sma_20'] > latest['sma_50'] > latest['sma_200']:
                score += 0.15
            elif latest['sma_20'] > latest['sma_50']:
                score += 0.10

        if 'adx' in latest:
            if latest['adx'] > 25:
                score += 0.10
            elif latest['adx'] > 20:
                score += 0.05

        if 'supertrend_direction' in latest:
            if latest['supertrend_direction'] == 1:
                score += 0.10

        return min(1, max(0, score))

    def _calculate_momentum_score(self, df: pd.DataFrame) -> float:
        latest = df.iloc[-1]
        score = 0.5

        if 'rsi' in latest:
            rsi = latest['rsi']
            if 50 <= rsi <= 70:
                score += 0.15
            elif rsi < 30:
                score += 0.20
            elif rsi > 70:
                score -= 0.10

        if 'macd' in latest and 'macd_signal' in latest:
            if latest['macd'] > latest['macd_signal']:
                score += 0.20
            else:
                score -= 0.10

        if len(df) >= 20:
            price_change_5d = ((df['close'].iloc[-1] - df['close'].iloc[-6]) / df['close'].iloc[-6]) * 100 if len(df) >= 6 else 0
            price_change_20d = ((df['close'].iloc[-1] - df['close'].iloc[-21]) / df['close'].iloc[-21]) * 100 if len(df) >= 21 else 0

            if price_change_5d > 2:
                score += 0.10
            if price_change_20d > 5:
                score += 0.15
            elif price_change_20d < -5:
                score -= 0.10

        return min(1, max(0, score))

    def _calculate_volume_score(self, df: pd.DataFrame) -> float:
        latest = df.iloc[-1]
        score = 0.5

        if 'volume_ratio' in latest:
            volume_ratio = latest['volume_ratio']
            if volume_ratio > 2.0:
                score += 0.30
            elif volume_ratio > 1.5:
                score += 0.20
            elif volume_ratio > 1.2:
                score += 0.10

        if len(df) >= 20:
            volume_ma = df['volume'].tail(20).mean()
            volume_ma_50 = df['volume'].tail(50).mean() if len(df) >= 50 else volume_ma
            if volume_ma > volume_ma_50:
                score += 0.10

        return min(1, max(0, score))

    def _calculate_volatility_score(self, df: pd.DataFrame) -> float:
        latest = df.iloc[-1]
        score = 0.5

        if 'atr' in latest and 'close' in latest:
            atr_pct = (latest['atr'] / latest['close']) * 100

            if 1 <= atr_pct <= 3:
                score += 0.30
            elif 0.5 <= atr_pct < 1:
                score += 0.15
            elif 3 < atr_pct <= 5:
                score += 0.10
            elif atr_pct > 5:
                score -= 0.20

        if 'bb_width' in latest:
            bb_width_pct = (latest['bb_width'] / latest['close']) * 100
            if 2 <= bb_width_pct <= 5:
                score += 0.15
            elif bb_width_pct < 2:
                score += 0.10

        return min(1, max(0, score))

    def _detect_patterns(self, df: pd.DataFrame) -> float:
        if len(df) < 3:
            return 0.5

        c2, c3 = df.iloc[-2], df.iloc[-1]
        score = 0.5

        body = abs(c3['close'] - c3['open'])
        lower_shadow = min(c3['open'], c3['close']) - c3['low']
        upper_shadow = c3['high'] - max(c3['open'], c3['close'])
        if body > 0 and lower_shadow > 2 * body and upper_shadow < body * 0.1:
            score += 0.20

        if (c2['close'] < c2['open'] and c3['close'] > c3['open'] and
            c3['open'] < c2['close'] and c3['close'] > c2['open']):
            score += 0.25

        if body > 0 and upper_shadow > 2 * body and lower_shadow < body * 0.1:
            score -= 0.15

        if (c2['close'] > c2['open'] and c3['close'] < c3['open'] and
            c3['open'] > c2['close'] and c3['close'] < c2['open']):
            score -= 0.20

        return min(1, max(0, score))

    def _calculate_fundamental_score(self, df: pd.DataFrame, symbol: str) -> float:
        score = 0.5

        if len(df) >= 252:
            high_52w = df['high'].tail(252).max()
            low_52w = df['low'].tail(252).min()
            if high_52w > low_52w:
                position = (df['close'].iloc[-1] - low_52w) / (high_52w - low_52w)
                if position > 0.7:
                    score += 0.20
                elif position > 0.5:
                    score += 0.10
                elif position < 0.3:
                    score -= 0.10

        return min(1, max(0, score))

    def _calculate_risk_score(self, df: pd.DataFrame) -> float:
        latest = df.iloc[-1]
        risk = 0

        if 'atr' in latest and 'close' in latest:
            atr_pct = (latest['atr'] / latest['close']) * 100
            if atr_pct > 5:
                risk += 30
            elif atr_pct > 3:
                risk += 20

        if len(df) >= 50:
            peak = df['close'].tail(50).max()
            current = latest['close']
            drawdown = ((peak - current) / peak) * 100
            if drawdown > 20:
                risk += 30
            elif drawdown > 10:
                risk += 20

        if 'volume_ratio' in latest and latest['volume_ratio'] < 0.5:
            risk += 20

        if 'rsi' in latest:
            if latest['rsi'] > 80:
                risk += 20
            elif latest['rsi'] < 20:
                risk += 10

        return min(100, risk)

    def _find_support_resistance(self, df: pd.DataFrame) -> Tuple[Optional[float], Optional[float]]:
        if len(df) < 50:
            return None, None

        support_levels = []
        resistance_levels = []

        lookback = 20
        for i in range(lookback, len(df) - lookback):
            if all(df['low'].iloc[i] <= df['low'].iloc[i-lookback:i+lookback]):
                support_levels.append(df['low'].iloc[i])
            if all(df['high'].iloc[i] >= df['high'].iloc[i-lookback:i+lookback]):
                resistance_levels.append(df['high'].iloc[i])

        current_price = df['close'].iloc[-1]
        support = max([s for s in support_levels if s < current_price], default=None)
        resistance = min([r for r in resistance_levels if r > current_price], default=None)

        return support, resistance

    def rank_stocks(self, stocks_data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        results = []

        for symbol, df in stocks_data.items():
            if df.empty:
                continue

            analysis = self.analyze_stock(df, symbol)
            if 'error' not in analysis:
                results.append({
                    'Symbol': symbol,
                    'Price': analysis['current_price'],
                    'Score': analysis['total_score'],
                    'Recommendation': analysis['recommendation'],
                    'Confidence': analysis['confidence'],
                    'Risk': analysis['risk_level'],
                    'Support': analysis['support'],
                    'Resistance': analysis['resistance'],
                    'Target Up': analysis['target_up'],
                    'Target Down': analysis['target_down'],
                    'Trend Score': analysis['scores']['trend'],
                    'Momentum Score': analysis['scores']['momentum'],
                    'Volume Score': analysis['scores']['volume'],
                    'Volatility Score': analysis['scores']['volatility'],
                    'Pattern Score': analysis['scores']['patterns']
                })

        if results:
            df_results = pd.DataFrame(results)
            df_results = df_results.sort_values('Score', ascending=False)
            self.rankings = df_results
            return df_results

        return pd.DataFrame()

# ============================================================================
# PORTFOLIO RECOMMENDER
# ============================================================================

class PortfolioRecommender:
    def __init__(self, capital: float = 100000, max_positions: int = 5):
        self.capital = capital
        self.max_positions = max_positions
        self.recommendations = []

    def generate_portfolio(self, ranked_stocks: pd.DataFrame) -> Dict:
        if ranked_stocks.empty:
            return {'error': 'No stocks to recommend'}

        buy_stocks = ranked_stocks[
            (ranked_stocks['Recommendation'].isin(['STRONG BUY', 'BUY'])) &
            (ranked_stocks['Risk'] != 'HIGH')
        ].head(self.max_positions)

        if buy_stocks.empty:
            return {
                'message': 'No strong buy recommendations at this time',
                'recommendations': []
            }

        total_score = buy_stocks['Score'].sum()
        if total_score == 0:
            total_score = 1

        positions = []
        for _, stock in buy_stocks.iterrows():
            weight = stock['Score'] / total_score
            amount = self.capital * weight * 0.8

            price = stock['Price']
            shares = int(amount / price)
            actual_amount = shares * price

            positions.append({
                'Symbol': stock['Symbol'],
                'Price': price,
                'Shares': shares,
                'Amount': actual_amount,
                'Weight': weight * 100,
                'Recommendation': stock['Recommendation'],
                'Confidence': stock['Confidence'],
                'Risk': stock['Risk'],
                'Target Up': stock['Target Up'],
                'Target Down': stock['Target Down'],
                'Support': stock['Support'],
                'Resistance': stock['Resistance']
            })

        total_invested = sum(p['Amount'] for p in positions)
        cash_remaining = self.capital - total_invested

        return {
            'total_capital': self.capital,
            'invested': total_invested,
            'cash_remaining': cash_remaining,
            'utilization': (total_invested / self.capital) * 100,
            'positions': positions,
            'recommendations': buy_stocks.to_dict('records')
        }

    def print_portfolio(self, portfolio: Dict):
        if 'error' in portfolio:
            print(f"\nError: {portfolio['error']}")
            return

        print("\n" + "=" * 80)
        print("PORTFOLIO RECOMMENDATION")
        print("=" * 80)
        print(f"Total Capital: ₹{portfolio['total_capital']:,.2f}")
        print(f"Invested: ₹{portfolio['invested']:,.2f}")
        print(f"Cash Remaining: ₹{portfolio['cash_remaining']:,.2f}")
        print(f"Utilization: {portfolio['utilization']:.1f}%")
        print("=" * 80)

        if not portfolio['positions']:
            print("\nNo positions to display")
            return

        print("\nRECOMMENDED POSITIONS:")
        print("-" * 80)
        for pos in portfolio['positions']:
            print(f"\n{pos['Symbol']}:")
            print(f"  Price: ₹{pos['Price']:,.2f}")
            print(f"  Shares: {pos['Shares']}")
            print(f"  Amount: ₹{pos['Amount']:,.2f} ({pos['Weight']:.1f}%)")
            print(f"  Recommendation: {pos['Recommendation']} (Confidence: {pos['Confidence']:.0f}%)")
            print(f"  Risk Level: {pos['Risk']}")
            if pd.notna(pos['Support']) and pos['Support']:
                print(f"  Support: ₹{pos['Support']:,.2f}")
            if pd.notna(pos['Resistance']) and pos['Resistance']:
                print(f"  Resistance: ₹{pos['Resistance']:,.2f}")
            print(f"  Target: ₹{pos['Target Up']:,.2f} / Stop: ₹{pos['Target Down']:,.2f}")

        print("\n" + "=" * 80)
        print("DISCLAIMER: This is for educational purposes only.")
        print("Please do your own research before making investment decisions.")
        print("=" * 80)

# ============================================================================
# VISUALIZATIONS
# ============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_recommendation_dashboard(analysis: Dict):
    if 'error' in analysis:
        print(f"Error: {analysis['error']}")
        return

    # Create separate gauge figure
    gauge_fig = go.Figure()

    score_color = '#00b894' if analysis['total_score'] >= 65 else '#ffd93d' if analysis['total_score'] >= 45 else '#ef5350'

    gauge_fig.add_trace(
        go.Indicator(
            mode="gauge+number+delta",
            value=analysis['total_score'],
            title={'text': f"{analysis['recommendation']}<br>{analysis['symbol']}"},
            delta={'reference': 50, 'increasing': {'color': "#00b894"}, 'decreasing': {'color': "#ef5350"}},
            gauge={
                'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "white"},
                'bar': {'color': score_color},
                'bgcolor': "rgba(0,0,0,0)",
                'borderwidth': 2,
                'bordercolor': "gray",
                'steps': [
                    {'range': [0, 30], 'color': 'rgba(239, 83, 80, 0.3)'},
                    {'range': [30, 45], 'color': 'rgba(253, 121, 168, 0.3)'},
                    {'range': [45, 65], 'color': 'rgba(255, 217, 61, 0.3)'},
                    {'range': [65, 80], 'color': 'rgba(0, 184, 148, 0.3)'},
                    {'range': [80, 100], 'color': 'rgba(0, 184, 148, 0.6)'}
                ],
                'threshold': {
                    'line': {'color': "white", 'width': 2},
                    'thickness': 0.75,
                    'value': analysis['total_score']
                }
            }
        )
    )

    gauge_fig.update_layout(
        template='plotly_dark',
        height=400,
        width=600,
        paper_bgcolor='rgba(0,0,0,0)',
        font={'color': "white", 'family': "Arial"}
    )

    gauge_fig.show()

    # Create main dashboard without gauge
    fig = make_subplots(
        rows=3, cols=2,
        specs=[[{'type': 'xy'}, {'type': 'xy'}],
               [{'type': 'xy'}, {'type': 'xy'}],
               [{'type': 'xy'}, {'type': 'xy'}]],
        subplot_titles=('Price & Indicators', 'Component Scores',
                       'RSI', 'MACD',
                       'Support & Resistance', 'Risk Assessment')
    )

    df = analysis['data']

    # Price with SMAs
    fig.add_trace(
        go.Scatter(
            x=df.index[-100:],
            y=df['close'].iloc[-100:],
            name='Price',
            line=dict(color='#ffffff', width=2)
        ),
        row=1, col=1
    )
    if 'sma_20' in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df.index[-100:],
                y=df['sma_20'].iloc[-100:],
                name='SMA 20',
                line=dict(color='#ff6b6b', width=1)
            ),
            row=1, col=1
        )
    if 'sma_50' in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df.index[-100:],
                y=df['sma_50'].iloc[-100:],
                name='SMA 50',
                line=dict(color='#ffd93d', width=1)
            ),
            row=1, col=1
        )

    # Component Scores
    scores = analysis['scores']
    fig.add_trace(
        go.Bar(
            x=list(scores.keys()),
            y=list(scores.values()),
            name='Scores',
            marker_color=['#4d96ff', '#00b894', '#ffd93d', '#6c5ce7', '#fd79a8', '#fdcb6e']
        ),
        row=1, col=2
    )

    # RSI
    if 'rsi' in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df.index[-100:],
                y=df['rsi'].iloc[-100:],
                name='RSI',
                line=dict(color='#00b894', width=2)
            ),
            row=2, col=1
        )
        fig.add_hline(y=70, line_dash="dash", line_color="#ef5350", row=2, col=1)
        fig.add_hline(y=30, line_dash="dash", line_color="#00b894", row=2, col=1)

    # MACD
    if 'macd' in df.columns and 'macd_signal' in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df.index[-100:],
                y=df['macd'].iloc[-100:],
                name='MACD',
                line=dict(color='#00b894', width=2)
            ),
            row=2, col=2
        )
        fig.add_trace(
            go.Scatter(
                x=df.index[-100:],
                y=df['macd_signal'].iloc[-100:],
                name='Signal',
                line=dict(color='#fd79a8', width=2)
            ),
            row=2, col=2
        )

    # Support & Resistance
    fig.add_trace(
        go.Scatter(
            x=df.index[-100:],
            y=df['close'].iloc[-100:],
            name='Price',
            line=dict(color='#ffffff', width=2)
        ),
        row=3, col=1
    )
    if pd.notna(analysis['support']) and analysis['support']:
        fig.add_hline(y=analysis['support'], line_dash="dash", line_color="#00b894",
                     row=3, col=1)
    if pd.notna(analysis['resistance']) and analysis['resistance']:
        fig.add_hline(y=analysis['resistance'], line_dash="dash", line_color="#ef5350",
                     row=3, col=1)

    # Risk Assessment
    risk_data = {
        'Volatility': analysis['risk_score'] * 0.3,
        'Drawdown': analysis['risk_score'] * 0.3,
        'Volume': analysis['risk_score'] * 0.2,
        'RSI': analysis['risk_score'] * 0.2
    }
    fig.add_trace(
        go.Bar(
            x=list(risk_data.keys()),
            y=list(risk_data.values()),
            name='Risk Components',
            marker_color=['#4d96ff', '#6c5ce7', '#ffd93d', '#fd79a8']
        ),
        row=3, col=2
    )

    fig.update_layout(
        title=f"{analysis['symbol']} - Technical Analysis Dashboard",
        template='plotly_dark',
        height=1000,
        showlegend=False,
        hovermode='x unified'
    )

    fig.show()

def plot_rankings_dashboard(rankings: pd.DataFrame):
    if rankings.empty:
        print("No rankings to display")
        return

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "xy"}, {"type": "xy"}],
            [{"type": "domain"}, {"type": "domain"}]
        ],
        subplot_titles=(
            "Stock Rankings",
            "Score Distribution",
            "Recommendation Distribution",
            "Risk Distribution"
        )
    )

    top_stocks = rankings.head(10)
    colors = ['#00b894' if s >= 65 else '#ffd93d' if s >= 45 else '#ef5350'
              for s in top_stocks['Score']]

    fig.add_trace(
        go.Bar(
            x=top_stocks['Symbol'],
            y=top_stocks['Score'],
            name='Score',
            marker_color=colors,
            text=top_stocks['Score'].round(1),
            textposition='outside'
        ),
        row=1, col=1
    )
    fig.add_hline(y=65, line_dash="dash", line_color="#00b894", row=1, col=1)
    fig.add_hline(y=45, line_dash="dash", line_color="#ffd93d", row=1, col=1)

    fig.add_trace(
        go.Histogram(
            x=rankings['Score'],
            name='Score Distribution',
            nbinsx=20,
            marker_color='#4d96ff'
        ),
        row=1, col=2
    )

    rec_counts = rankings['Recommendation'].value_counts()
    fig.add_trace(
        go.Pie(
            labels=rec_counts.index,
            values=rec_counts.values,
            name='Recommendations',
            marker=dict(colors=['#00b894', '#4d96ff', '#ffd93d', '#fd79a8', '#ef5350'])
        ),
        row=2, col=1
    )

    risk_counts = rankings['Risk'].value_counts()
    fig.add_trace(
        go.Pie(
            labels=risk_counts.index,
            values=risk_counts.values,
            name='Risk Levels',
            marker=dict(colors=['#00b894', '#ffd93d', '#ef5350'])
        ),
        row=2, col=2
    )

    fig.update_layout(
        title='Stock Rankings Dashboard',
        template='plotly_dark',
        height=800,
        showlegend=True
    )

    fig.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_recommendation_engine():
    print("=" * 70)
    print("STOCK RECOMMENDATION ENGINE")
    print("=" * 70)

    import yfinance as yf

    symbols = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS',
               'KOTAKBANK.NS', 'SBIN.NS', 'ITC.NS', 'WIPRO.NS',
               'SUNPHARMA.NS', 'TITAN.NS', 'MARUTI.NS', 'TATASTEEL.NS']

    print(f"\nAnalyzing {len(symbols)} stocks...")
    print("-" * 50)

    stocks_data = {}
    engine = RecommendationEngine()

    for symbol in symbols:
        print(f"  Processing {symbol}...")
        df = yf.download(symbol, period="2y", progress=False)

        if df.empty:
            continue

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.columns = [str(c).strip().lower() for c in df.columns]

        df['sma_20'] = df['close'].rolling(20).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['sma_200'] = df['close'].rolling(200).mean()

        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))

        exp1 = df['close'].ewm(span=12, adjust=False).mean()
        exp2 = df['close'].ewm(span=26, adjust=False).mean()
        df['macd'] = exp1 - exp2
        df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()

        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        ranges = pd.concat([high_low, high_close, low_close], axis=1)
        true_range = np.max(ranges, axis=1)
        df['atr'] = true_range.rolling(14).mean()

        df['volume_sma'] = df['volume'].rolling(20).mean()
        df['volume_ratio'] = df['volume'] / df['volume_sma']

        df['bb_middle'] = df['close'].rolling(20).mean()
        bb_std = df['close'].rolling(20).std()
        df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
        df['bb_lower'] = df['bb_middle'] - (bb_std * 2)
        df['bb_width'] = df['bb_upper'] - df['bb_lower']
        df['bb_percent'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

        df['tr'] = np.maximum(
            df['high'] - df['low'],
            np.maximum(
                np.abs(df['high'] - df['close'].shift()),
                np.abs(df['low'] - df['close'].shift())
            )
        )
        df['up_move'] = df['high'] - df['high'].shift()
        df['down_move'] = df['low'].shift() - df['low']
        df['plus_dm'] = np.where((df['up_move'] > df['down_move']) & (df['up_move'] > 0), df['up_move'], 0)
        df['minus_dm'] = np.where((df['down_move'] > df['up_move']) & (df['down_move'] > 0), df['down_move'], 0)
        df['atr_adx'] = df['tr'].rolling(14).mean()
        df['plus_di'] = 100 * (df['plus_dm'].rolling(14).mean() / df['atr_adx'])
        df['minus_di'] = 100 * (df['minus_dm'].rolling(14).mean() / df['atr_adx'])
        dx = (np.abs(df['plus_di'] - df['minus_di']) / (df['plus_di'] + df['minus_di'])) * 100
        df['adx'] = dx.rolling(14).mean()

        df['atr_st'] = df['high'].rolling(10).max() - df['low'].rolling(10).min()
        hl2 = (df['high'] + df['low']) / 2
        df['upper_band'] = hl2 + (3 * df['atr_st'])
        df['lower_band'] = hl2 - (3 * df['atr_st'])
        df['supertrend'] = 0
        df['supertrend_direction'] = 1
        for i in range(10, len(df)):
            if df['close'].iloc[i] > df['upper_band'].iloc[i-1]:
                df.loc[df.index[i], 'supertrend_direction'] = 1
            elif df['close'].iloc[i] < df['lower_band'].iloc[i-1]:
                df.loc[df.index[i], 'supertrend_direction'] = -1
            else:
                df.loc[df.index[i], 'supertrend_direction'] = df['supertrend_direction'].iloc[i-1]

        stocks_data[symbol] = df

    print("\nRanking stocks...")
    rankings = engine.rank_stocks(stocks_data)

    print("\n" + "=" * 80)
    print("STOCK RANKINGS")
    print("=" * 80)
    print(rankings[['Symbol', 'Price', 'Score', 'Recommendation', 'Confidence', 'Risk']].to_string(index=False))
    print("=" * 80)

    print("\nGenerating portfolio recommendations...")
    recommender = PortfolioRecommender(capital=100000, max_positions=5)
    portfolio = recommender.generate_portfolio(rankings)
    recommender.print_portfolio(portfolio)

    print("\nGenerating charts...")

    if not rankings.empty:
        top_stock = rankings.iloc[0]['Symbol']
        analysis = engine.analyze_stock(stocks_data[top_stock], top_stock)
        if 'error' not in analysis:
            plot_recommendation_dashboard(analysis)

    plot_rankings_dashboard(rankings)

    print("\n" + "=" * 70)
    print("RECOMMENDATION ENGINE COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    run_recommendation_engine()

STOCK RECOMMENDATION ENGINE

Analyzing 12 stocks...
--------------------------------------------------
  Processing RELIANCE.NS...
  Processing TCS.NS...
  Processing INFY.NS...
  Processing HDFCBANK.NS...
  Processing KOTAKBANK.NS...
  Processing SBIN.NS...
  Processing ITC.NS...
  Processing WIPRO.NS...
  Processing SUNPHARMA.NS...
  Processing TITAN.NS...
  Processing MARUTI.NS...
  Processing TATASTEEL.NS...

Ranking stocks...

STOCK RANKINGS
      Symbol        Price  Score Recommendation  Confidence   Risk
SUNPHARMA.NS  1932.599976  84.00     STRONG BUY        87.0    LOW
    TITAN.NS  4638.100098  76.25            BUY        74.5    LOW
      TCS.NS  2269.000000  76.00            BUY        74.4    LOW
 RELIANCE.NS  1327.199951  75.75            BUY        74.3    LOW
 HDFCBANK.NS   819.599976  73.00            BUY        73.2    LOW
     SBIN.NS  1044.300049  71.75            BUY        72.7    LOW
     INFY.NS  1096.500000  70.25            BUY        72.1 MEDIUM
    WIPRO.NS 


RECOMMENDATION ENGINE COMPLETE


# CELL 7: PORTFOLIO OPTIMIZATION & RISK ANALYTICS

In [9]:
# CELL 7: PORTFOLIO OPTIMIZATION & RISK ANALYTICS

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.covariance import LedoitWolf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class PortfolioConfig:
    risk_free_rate: float = 0.07
    trading_days: int = 252
    confidence_level: float = 0.95
    max_allocation: float = 0.25
    min_allocation: float = 0.0
    ewma_span: int = 126

config = PortfolioConfig()

# ============================================================================
# PORTFOLIO OPTIMIZER
# ============================================================================

class PortfolioOptimizer:
    def __init__(self, returns_data: pd.DataFrame):
        self.returns = returns_data
        self.symbols = returns_data.columns.tolist()
        self.n_assets = len(self.symbols)

        ewma_returns = returns_data.ewm(span=config.ewma_span).mean().iloc[-1]
        historical_returns = returns_data.mean()
        blended_returns = 0.6 * ewma_returns + 0.4 * historical_returns
        self.expected_returns = blended_returns * config.trading_days

        lw = LedoitWolf()
        lw.fit(returns_data)
        self.cov_matrix = pd.DataFrame(
            lw.covariance_ * config.trading_days,
            index=returns_data.columns,
            columns=returns_data.columns
        )

        self.std_dev = np.sqrt(np.diag(self.cov_matrix))
        self.efficient_frontier = None
        self.optimal_portfolios = {}
        self._cache = {}

    def portfolio_performance(self, weights: np.ndarray) -> tuple:
        portfolio_return = np.sum(self.expected_returns * weights)
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix, weights)))
        sharpe_ratio = (portfolio_return - config.risk_free_rate) / max(portfolio_volatility, 1e-8)
        return portfolio_return, portfolio_volatility, sharpe_ratio

    def negative_sharpe(self, weights: np.ndarray) -> float:
        return -self.portfolio_performance(weights)[2]

    def portfolio_volatility(self, weights: np.ndarray) -> float:
        return self.portfolio_performance(weights)[1]

    def optimize_sharpe_ratio(self, force_refresh: bool = False) -> Dict:
        cache_key = 'max_sharpe'
        if not force_refresh and cache_key in self._cache:
            return self._cache[cache_key]

        constraints = [
            {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
            {'type': 'ineq', 'fun': lambda x: x - config.min_allocation},
            {'type': 'ineq', 'fun': lambda x: config.max_allocation - x}
        ]

        bounds = tuple((0, 1) for _ in range(self.n_assets))
        initial_weights = np.array([1/self.n_assets] * self.n_assets)

        result = minimize(
            self.negative_sharpe,
            initial_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        if result.success:
            weights = result.x
            ret, vol, sharpe = self.portfolio_performance(weights)
            result_dict = {
                'weights': weights,
                'symbols': self.symbols,
                'return': ret,
                'volatility': vol,
                'sharpe_ratio': sharpe,
                'method': 'Maximum Sharpe Ratio'
            }
            self._cache[cache_key] = result_dict
            return result_dict
        return None

    def optimize_min_volatility(self, force_refresh: bool = False) -> Dict:
        cache_key = 'min_vol'
        if not force_refresh and cache_key in self._cache:
            return self._cache[cache_key]

        constraints = [
            {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
            {'type': 'ineq', 'fun': lambda x: x - config.min_allocation},
            {'type': 'ineq', 'fun': lambda x: config.max_allocation - x}
        ]

        bounds = tuple((0, 1) for _ in range(self.n_assets))
        initial_weights = np.array([1/self.n_assets] * self.n_assets)

        result = minimize(
            self.portfolio_volatility,
            initial_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        if result.success:
            weights = result.x
            ret, vol, sharpe = self.portfolio_performance(weights)
            result_dict = {
                'weights': weights,
                'symbols': self.symbols,
                'return': ret,
                'volatility': vol,
                'sharpe_ratio': sharpe,
                'method': 'Minimum Volatility'
            }
            self._cache[cache_key] = result_dict
            return result_dict
        return None

    def optimize_target_return(self, target_return: float) -> Dict:
        constraints = [
            {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
            {'type': 'eq', 'fun': lambda x: np.sum(self.expected_returns * x) - target_return},
            {'type': 'ineq', 'fun': lambda x: x - config.min_allocation},
            {'type': 'ineq', 'fun': lambda x: config.max_allocation - x}
        ]

        bounds = tuple((0, 1) for _ in range(self.n_assets))
        initial_weights = np.array([1/self.n_assets] * self.n_assets)

        result = minimize(
            self.portfolio_volatility,
            initial_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        if result.success:
            weights = result.x
            ret, vol, sharpe = self.portfolio_performance(weights)
            return {
                'weights': weights,
                'symbols': self.symbols,
                'return': ret,
                'volatility': vol,
                'sharpe_ratio': sharpe,
                'method': f'Target Return: {target_return:.2%}'
            }
        return None

    def generate_efficient_frontier(self, n_points: int = 80) -> pd.DataFrame:
        min_vol = self.optimize_min_volatility()
        max_sharpe = self.optimize_sharpe_ratio()

        if min_vol and max_sharpe:
            min_return = min_vol['return']
            max_return = max_sharpe['return']
        else:
            min_return = self.expected_returns.quantile(0.10)
            max_return = self.expected_returns.quantile(0.90)

        frontier_returns = np.linspace(min_return * 0.9, max_return * 1.1, n_points)
        frontier_volatilities = []
        frontier_weights = []

        for target_return in frontier_returns:
            result = self.optimize_target_return(target_return)
            if result:
                frontier_volatilities.append(result['volatility'])
                frontier_weights.append(result['weights'])
            else:
                frontier_volatilities.append(np.nan)
                frontier_weights.append(None)

        self.efficient_frontier = pd.DataFrame({
            'Return': frontier_returns,
            'Volatility': frontier_volatilities,
            'Weights': frontier_weights
        }).dropna()

        return self.efficient_frontier

    def calculate_beta(self, market_returns: pd.Series) -> pd.DataFrame:
        common_idx = self.returns.index.intersection(market_returns.index)
        returns_aligned = self.returns.loc[common_idx]
        market_aligned = market_returns.loc[common_idx]

        betas = {}
        for symbol in self.symbols:
            cov = np.cov(returns_aligned[symbol], market_aligned)[0, 1]
            var = np.var(market_aligned)
            beta = cov / var if var > 0 else 0
            betas[symbol] = beta

        return pd.DataFrame({
            'Symbol': list(betas.keys()),
            'Beta': list(betas.values())
        }).set_index('Symbol')

# ============================================================================
# RISK METRICS
# ============================================================================

class RiskMetrics:
    @staticmethod
    def calculate_var(returns: pd.Series, confidence_level: float = 0.95) -> Dict:
        var_historical = np.percentile(returns, (1 - confidence_level) * 100)
        mu = returns.mean()
        sigma = returns.std()
        var_parametric = mu + sigma * norm.ppf(1 - confidence_level)
        cvar = returns[returns <= var_historical].mean()

        return {
            'historical_var': var_historical,
            'parametric_var': var_parametric,
            'expected_shortfall': cvar,
            'confidence_level': confidence_level
        }

    @staticmethod
    def calculate_drawdown(portfolio_values: pd.Series) -> Dict:
        peak = portfolio_values.expanding().max()
        drawdown = (portfolio_values - peak) / peak

        max_drawdown = drawdown.min()
        is_drawdown = drawdown < 0
        drawdown_duration = 0
        max_duration = 0

        for i in range(len(is_drawdown)):
            if is_drawdown.iloc[i]:
                drawdown_duration += 1
                max_duration = max(max_duration, drawdown_duration)
            else:
                drawdown_duration = 0

        return {
            'max_drawdown': max_drawdown,
            'max_drawdown_duration': max_duration,
            'current_drawdown': drawdown.iloc[-1],
            'peak_value': peak.iloc[-1],
            'drawdown_series': drawdown
        }

    @staticmethod
    def calculate_sortino(returns: pd.Series, risk_free_rate: float = 0.07) -> float:
        downside_returns = returns[returns < 0]
        downside_deviation = downside_returns.std() * np.sqrt(252)
        excess_return = (returns.mean() * 252) - risk_free_rate
        return excess_return / max(downside_deviation, 1e-8)

    @staticmethod
    def calculate_calmar(returns: pd.Series, max_drawdown: float) -> float:
        annual_return = returns.mean() * 252
        return annual_return / max(abs(max_drawdown), 1e-8)

# ============================================================================
# CAPITAL ALLOCATION LINE
# ============================================================================

class CapitalAllocationLine:
    def __init__(self, optimizer: PortfolioOptimizer):
        self.optimizer = optimizer
        self.risk_free_rate = config.risk_free_rate
        self.tangency = optimizer.optimize_sharpe_ratio()

        if self.tangency:
            self.tangency_return = self.tangency['return']
            self.tangency_vol = self.tangency['volatility']

    def capital_market_line(self, volatility: float) -> float:
        if not self.tangency or self.tangency_vol == 0:
            return None
        return self.risk_free_rate + ((self.tangency_return - self.risk_free_rate) / self.tangency_vol) * volatility

# ============================================================================
# VISUALIZATIONS
# ============================================================================

def plot_efficient_frontier(optimizer: PortfolioOptimizer, cml: CapitalAllocationLine = None):
    if optimizer.efficient_frontier is None:
        optimizer.generate_efficient_frontier()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=optimizer.efficient_frontier['Volatility'] * 100,
        y=optimizer.efficient_frontier['Return'] * 100,
        mode='lines',
        name='Efficient Frontier',
        line=dict(color='#4d96ff', width=2),
        fill='tozeroy',
        fillcolor='rgba(77, 150, 255, 0.1)'
    ))

    for symbol in optimizer.symbols:
        idx = optimizer.symbols.index(symbol)
        fig.add_trace(go.Scatter(
            x=[optimizer.std_dev[idx] * 100],
            y=[optimizer.expected_returns.iloc[idx] * 100],
            mode='markers+text',
            name=symbol,
            text=[symbol],
            textposition='top center',
            marker=dict(size=10, color='#ffd93d')
        ))

    min_vol = optimizer.optimize_min_volatility()
    if min_vol:
        fig.add_trace(go.Scatter(
            x=[min_vol['volatility'] * 100],
            y=[min_vol['return'] * 100],
            mode='markers+text',
            name='Min Volatility',
            text=['Min Vol'],
            textposition='bottom center',
            marker=dict(size=15, color='#00b894', symbol='star')
        ))

    max_sharpe = optimizer.optimize_sharpe_ratio()
    if max_sharpe:
        fig.add_trace(go.Scatter(
            x=[max_sharpe['volatility'] * 100],
            y=[max_sharpe['return'] * 100],
            mode='markers+text',
            name='Tangency Portfolio',
            text=['Tangency'],
            textposition='bottom center',
            marker=dict(size=15, color='#ef5350', symbol='star')
        ))

    if cml and cml.tangency:
        x_range = np.linspace(0, optimizer.efficient_frontier['Volatility'].max() * 1.2, 100)
        y_cml = []
        for x in x_range:
            val = cml.capital_market_line(x)
            y_cml.append(val * 100 if val is not None else np.nan)

        fig.add_trace(go.Scatter(
            x=x_range * 100,
            y=y_cml,
            mode='lines',
            name='CML',
            line=dict(color='#fdcb6e', width=2, dash='dash')
        ))

    fig.update_layout(
        title='Efficient Frontier & Capital Market Line',
        template='plotly_dark',
        xaxis_title='Risk (Volatility %)',
        yaxis_title='Expected Return (%)',
        height=700,
        hovermode='closest',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )

    fig.show()

def plot_portfolio_composition(portfolio: Dict, title: str = "Portfolio Composition"):
    if not portfolio:
        print("No portfolio to display")
        return

    weights = portfolio['weights']
    symbols = portfolio.get('symbols', [f"Asset {i+1}" for i in range(len(weights))])

    non_zero = [(s, w) for s, w in zip(symbols, weights) if w > 0.001]
    if non_zero:
        symbols, weights = zip(*non_zero)
        weights = np.array(weights) / np.sum(weights)

    fig = go.Figure(data=[
        go.Pie(
            labels=symbols,
            values=np.array(weights) * 100,
            hole=0.4,
            marker=dict(colors=px.colors.qualitative.Plotly),
            textinfo='label+percent',
            textposition='inside'
        )
    ])

    fig.update_layout(
        title=f"{title}<br>Return: {portfolio['return']:.2%} | Volatility: {portfolio['volatility']:.2%} | Sharpe: {portfolio['sharpe_ratio']:.2f}",
        template='plotly_dark',
        height=500
    )

    fig.show()

def plot_risk_dashboard(returns: pd.DataFrame, portfolio_values: pd.Series):
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('Rolling Volatility', 'Rolling Sharpe Ratio',
                       'VaR Analysis', 'Expected Shortfall',
                       'Drawdown', 'Risk Distribution'),
        vertical_spacing=0.08
    )

    rolling_vol = returns.rolling(20).std() * np.sqrt(252)
    fig.add_trace(
        go.Scatter(
            x=rolling_vol.index,
            y=rolling_vol.mean(axis=1) * 100,
            name='Volatility',
            line=dict(color='#4d96ff', width=2),
            fill='tozeroy',
            fillcolor='rgba(77, 150, 255, 0.2)'
        ),
        row=1, col=1
    )

    rolling_sharpe = (returns.rolling(20).mean() * 252 - config.risk_free_rate) / rolling_vol
    fig.add_trace(
        go.Scatter(
            x=rolling_sharpe.index,
            y=rolling_sharpe.mean(axis=1),
            name='Sharpe',
            line=dict(color='#00b894', width=2)
        ),
        row=1, col=2
    )
    fig.add_hline(y=0, line_dash="dash", line_color="#ef5350", row=1, col=2)

    portfolio_returns = portfolio_values.pct_change().dropna()
    var_metrics = RiskMetrics.calculate_var(portfolio_returns, config.confidence_level)

    fig.add_trace(
        go.Histogram(
            x=portfolio_returns * 100,
            name='Returns Distribution',
            nbinsx=50,
            marker_color='#4d96ff',
            opacity=0.7
        ),
        row=2, col=1
    )
    fig.add_vline(x=var_metrics['historical_var'] * 100, line_dash="dash",
                  line_color="#ef5350", row=2, col=1)

    fig.add_trace(
        go.Bar(
            x=['Historical VaR', 'Parametric VaR', 'Expected Shortfall'],
            y=[var_metrics['historical_var'] * 100,
               var_metrics['parametric_var'] * 100,
               var_metrics['expected_shortfall'] * 100],
            marker_color=['#4d96ff', '#ffd93d', '#ef5350'],
            text=[f"{v:.2f}%" for v in [
                var_metrics['historical_var'] * 100,
                var_metrics['parametric_var'] * 100,
                var_metrics['expected_shortfall'] * 100
            ]],
            textposition='outside'
        ),
        row=2, col=2
    )

    drawdown_metrics = RiskMetrics.calculate_drawdown(portfolio_values)
    fig.add_trace(
        go.Scatter(
            x=drawdown_metrics['drawdown_series'].index,
            y=drawdown_metrics['drawdown_series'] * 100,
            name='Drawdown',
            fill='tozeroy',
            fillcolor='rgba(239, 83, 80, 0.3)',
            line=dict(color='#ef5350', width=2)
        ),
        row=3, col=1
    )
    fig.add_hline(y=0, line_dash="solid", line_color="#7f8c8d", row=3, col=1)

    risk_contributions = returns.std() * np.sqrt(252)
    fig.add_trace(
        go.Bar(
            x=risk_contributions.index[:10],
            y=risk_contributions.values[:10] * 100,
            marker_color='#6c5ce7',
            text=[f"{v:.2f}%" for v in risk_contributions.values[:10] * 100],
            textposition='outside'
        ),
        row=3, col=2
    )

    fig.update_layout(
        title='Risk Analytics Dashboard',
        template='plotly_dark',
        height=1000,
        showlegend=False,
        hovermode='x unified'
    )

    fig.show()

# ============================================================================
# SUMMARY GENERATOR
# ============================================================================

def generate_summary(optimizer: PortfolioOptimizer, min_vol: Dict, max_sharpe: Dict,
                     var_metrics: Dict, drawdown_metrics: Dict, sortino: float,
                     calmar: float, betas: pd.DataFrame) -> str:
    """Generate comprehensive summary and conclusion"""

    summary = []
    summary.append("=" * 80)
    summary.append("PORTFOLIO OPTIMIZATION SUMMARY & CONCLUSION")
    summary.append("=" * 80)

    # 1. Portfolio Comparison
    summary.append("\n1. PORTFOLIO COMPARISON")
    summary.append("-" * 50)
    summary.append(f"{'Metric':<25} {'Min Volatility':<20} {'Max Sharpe':<20}")
    summary.append("-" * 50)
    summary.append(f"{'Expected Return':<25} {min_vol['return']:>19.2%} {max_sharpe['return']:>19.2%}")
    summary.append(f"{'Volatility':<25} {min_vol['volatility']:>19.2%} {max_sharpe['volatility']:>19.2%}")
    summary.append(f"{'Sharpe Ratio':<25} {min_vol['sharpe_ratio']:>19.2f} {max_sharpe['sharpe_ratio']:>19.2f}")
    summary.append("-" * 50)

    # 2. Optimal Portfolio Recommendation
    summary.append("\n2. RECOMMENDED PORTFOLIO")
    summary.append("-" * 50)
    summary.append("Based on risk-return analysis, the MAXIMUM SHARPE RATIO portfolio is recommended:")
    summary.append("")
    summary.append(f"  Expected Return: {max_sharpe['return']:.2%}")
    summary.append(f"  Volatility: {max_sharpe['volatility']:.2%}")
    summary.append(f"  Sharpe Ratio: {max_sharpe['sharpe_ratio']:.2f}")
    summary.append("")
    summary.append("  Asset Allocation:")
    for i, symbol in enumerate(max_sharpe['symbols']):
        if max_sharpe['weights'][i] > 0.001:
            summary.append(f"    {symbol}: {max_sharpe['weights'][i]:.2%}")
    summary.append("-" * 50)

    # 3. Risk Assessment
    summary.append("\n3. RISK ASSESSMENT")
    summary.append("-" * 50)
    summary.append(f"  Value at Risk (95% confidence): {var_metrics['historical_var']:.2%}")
    summary.append(f"  Expected Shortfall (CVaR): {var_metrics['expected_shortfall']:.2%}")
    summary.append(f"  Maximum Drawdown: {drawdown_metrics['max_drawdown']:.2%}")
    summary.append(f"  Max Drawdown Duration: {drawdown_metrics['max_drawdown_duration']} days")
    summary.append(f"  Current Drawdown: {drawdown_metrics['current_drawdown']:.2%}")
    summary.append("")
    summary.append(f"  Sortino Ratio: {sortino:.2f} (Focuses on downside risk)")
    summary.append(f"  Calmar Ratio: {calmar:.2f} (Return vs. maximum drawdown)")
    summary.append("-" * 50)

    # 4. Beta Analysis
    summary.append("\n4. MARKET SENSITIVITY (BETA)")
    summary.append("-" * 50)
    summary.append("Beta measures stock sensitivity to market movements:")
    summary.append("  Beta > 1: More volatile than market")
    summary.append("  Beta = 1: Moves with market")
    summary.append("  Beta < 1: Less volatile than market")
    summary.append("")
    for symbol, row in betas.iterrows():
        beta = row['Beta']
        if beta > 1.0:
            interpretation = "HIGHER volatility than market"
        elif beta > 0.8:
            interpretation = "Similar to market"
        else:
            interpretation = "LOWER volatility than market"
        summary.append(f"  {symbol}: {beta:.2f} ({interpretation})")
    summary.append("-" * 50)

    # 5. Stocks to Avoid
    summary.append("\n5. STOCKS TO AVOID")
    summary.append("-" * 50)
    negative_returns = []
    for symbol, ret in optimizer.expected_returns.items():
        if ret < 0:
            negative_returns.append((symbol, ret))

    if negative_returns:
        summary.append("Stocks with negative expected returns:")
        for symbol, ret in sorted(negative_returns, key=lambda x: x[1]):
            summary.append(f"  {symbol}: {ret:.2%} expected annual return")
        summary.append("")
        summary.append("  RECOMMENDATION: Exclude these stocks from the portfolio")
    else:
        summary.append("  No stocks with negative expected returns identified")
    summary.append("-" * 50)

    # 6. Final Conclusion
    summary.append("\n6. CONCLUSION")
    summary.append("-" * 50)

    if max_sharpe['sharpe_ratio'] > 0.5:
        summary.append("  The optimal portfolio demonstrates GOOD risk-adjusted returns.")
    elif max_sharpe['sharpe_ratio'] > 0:
        summary.append("  The optimal portfolio demonstrates MODERATE risk-adjusted returns.")
    else:
        summary.append("  The optimal portfolio demonstrates POOR risk-adjusted returns.")

    if drawdown_metrics['max_drawdown'] > -0.20:
        summary.append("  Maximum drawdown is within acceptable limits.")
    else:
        summary.append("  Maximum drawdown is significant; consider risk tolerance.")

    if sortino > 0.5:
        summary.append("  Sortino ratio indicates good downside risk management.")
    elif sortino > 0:
        summary.append("  Sortino ratio indicates moderate downside risk management.")
    else:
        summary.append("  Sortino ratio indicates poor downside risk management.")

    summary.append("")
    summary.append("  Overall Assessment:")
    if max_sharpe['sharpe_ratio'] > 0.5 and drawdown_metrics['max_drawdown'] > -0.25:
        summary.append("  STRONG: Portfolio offers attractive risk-return characteristics.")
    elif max_sharpe['sharpe_ratio'] > 0 and drawdown_metrics['max_drawdown'] > -0.30:
        summary.append("  MODERATE: Portfolio offers acceptable risk-return characteristics.")
    else:
        summary.append("  CAUTION: Portfolio may not be suitable for conservative investors.")

    summary.append("")
    summary.append("  RECOMMENDATION:")
    summary.append("  Allocate capital to the Maximum Sharpe Ratio portfolio.")
    summary.append("  Consider adding risk-free assets if volatility is a concern.")
    summary.append("  Monitor drawdown and rebalance periodically.")

    summary.append("-" * 50)
    summary.append("")
    summary.append("DISCLAIMER: This analysis is for educational purposes only.")
    summary.append("Past performance does not guarantee future results.")
    summary.append("Consult a qualified financial advisor before making investment decisions.")
    summary.append("=" * 80)

    return "\n".join(summary)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_portfolio_optimization():
    print("=" * 70)
    print("PORTFOLIO OPTIMIZATION & RISK ANALYTICS")
    print("=" * 70)

    import yfinance as yf

    symbols = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS',
               'KOTAKBANK.NS', 'SBIN.NS', 'ITC.NS', 'WIPRO.NS',
               'SUNPHARMA.NS', 'TITAN.NS', 'MARUTI.NS']

    print(f"\nFetching data for {len(symbols)} stocks...")
    print("-" * 50)

    data = {}
    for symbol in symbols:
        print(f"  Loading {symbol}...")
        df = yf.download(symbol, period="3y", progress=False)
        if not df.empty:
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.columns = [str(c).strip().lower() for c in df.columns]
            data[symbol] = df['close']

    prices = pd.DataFrame(data)
    prices = prices.dropna()

    returns = np.log(prices / prices.shift(1)).dropna()

    print(f"\nData loaded: {len(prices)} days, {len(symbols)} stocks")
    print("-" * 50)

    print("\nCalculating efficient frontier...")
    optimizer = PortfolioOptimizer(returns)
    optimizer.generate_efficient_frontier()

    print("\nIndividual Stock Metrics:")
    print("-" * 50)
    stock_metrics = pd.DataFrame({
        'Symbol': optimizer.symbols,
        'Expected Return': optimizer.expected_returns,
        'Volatility': optimizer.std_dev,
        'Sharpe': (optimizer.expected_returns - config.risk_free_rate) / optimizer.std_dev
    })
    print(stock_metrics.to_string(index=False))
    print("-" * 50)

    print("\nOptimal Portfolios:")
    print("-" * 50)

    min_vol = optimizer.optimize_min_volatility()
    if min_vol:
        print(f"\nMinimum Volatility Portfolio:")
        print(f"  Expected Return: {min_vol['return']:.2%}")
        print(f"  Volatility: {min_vol['volatility']:.2%}")
        print(f"  Sharpe Ratio: {min_vol['sharpe_ratio']:.2f}")
        print(f"  Weights:")
        for i, symbol in enumerate(optimizer.symbols):
            if min_vol['weights'][i] > 0.001:
                print(f"    {symbol}: {min_vol['weights'][i]:.2%}")
        plot_portfolio_composition(min_vol, "Minimum Volatility Portfolio")

    max_sharpe = optimizer.optimize_sharpe_ratio()
    if max_sharpe:
        print(f"\nMaximum Sharpe Ratio Portfolio (Tangency):")
        print(f"  Expected Return: {max_sharpe['return']:.2%}")
        print(f"  Volatility: {max_sharpe['volatility']:.2%}")
        print(f"  Sharpe Ratio: {max_sharpe['sharpe_ratio']:.2f}")
        print(f"  Weights:")
        for i, symbol in enumerate(optimizer.symbols):
            if max_sharpe['weights'][i] > 0.001:
                print(f"    {symbol}: {max_sharpe['weights'][i]:.2%}")
        plot_portfolio_composition(max_sharpe, "Maximum Sharpe Ratio Portfolio")

    print("\nGenerating Capital Allocation Line...")
    cml = CapitalAllocationLine(optimizer)

    print("\nGenerating efficient frontier chart...")
    plot_efficient_frontier(optimizer, cml)

    if max_sharpe:
        portfolio_weights = max_sharpe['weights']
        portfolio_values = (prices * portfolio_weights).sum(axis=1)

        print("\nRisk Analytics:")
        print("-" * 50)

        portfolio_returns = portfolio_values.pct_change().dropna()
        var_metrics = RiskMetrics.calculate_var(portfolio_returns, config.confidence_level)
        print(f"\nValue at Risk ({config.confidence_level:.0%} confidence):")
        print(f"  Historical VaR: {var_metrics['historical_var']:.2%}")
        print(f"  Parametric VaR: {var_metrics['parametric_var']:.2%}")
        print(f"  Expected Shortfall: {var_metrics['expected_shortfall']:.2%}")

        drawdown_metrics = RiskMetrics.calculate_drawdown(portfolio_values)
        print(f"\nDrawdown Analysis:")
        print(f"  Maximum Drawdown: {drawdown_metrics['max_drawdown']:.2%}")
        print(f"  Max Drawdown Duration: {drawdown_metrics['max_drawdown_duration']} days")
        print(f"  Current Drawdown: {drawdown_metrics['current_drawdown']:.2%}")

        sortino = RiskMetrics.calculate_sortino(portfolio_returns)
        calmar = RiskMetrics.calculate_calmar(portfolio_returns, drawdown_metrics['max_drawdown'])
        print(f"\nAdvanced Risk Metrics:")
        print(f"  Sortino Ratio: {sortino:.2f}")
        print(f"  Calmar Ratio: {calmar:.2f}")

        print(f"\nBeta Calculation:")
        print("  Note: Using NIFTY 50 as market proxy...")

        nifty = yf.download('^NSEI', period="3y", progress=False)
        if not nifty.empty:
            if isinstance(nifty.columns, pd.MultiIndex):
                nifty.columns = nifty.columns.get_level_values(0)
            nifty.columns = [str(c).strip().lower() for c in nifty.columns]
            nifty_returns = np.log(nifty['close'] / nifty['close'].shift(1)).dropna()

            common_idx = returns.index.intersection(nifty_returns.index)
            if len(common_idx) > 0:
                betas = optimizer.calculate_beta(nifty_returns)
                print("\n  Betas:")
                for symbol, beta in betas.iterrows():
                    print(f"    {symbol}: {beta['Beta']:.2f}")

        print("\nGenerating risk dashboard...")
        plot_risk_dashboard(returns, portfolio_values)

        # Generate and display summary
        print("\n" + generate_summary(optimizer, min_vol, max_sharpe,
                                      var_metrics, drawdown_metrics,
                                      sortino, calmar, betas))

    print("\n" + "=" * 70)
    print("PORTFOLIO OPTIMIZATION COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    run_portfolio_optimization()

PORTFOLIO OPTIMIZATION & RISK ANALYTICS

Fetching data for 11 stocks...
--------------------------------------------------
  Loading RELIANCE.NS...
  Loading TCS.NS...
  Loading INFY.NS...
  Loading HDFCBANK.NS...
  Loading KOTAKBANK.NS...
  Loading SBIN.NS...
  Loading ITC.NS...
  Loading WIPRO.NS...
  Loading SUNPHARMA.NS...
  Loading TITAN.NS...
  Loading MARUTI.NS...

Data loaded: 744 days, 11 stocks
--------------------------------------------------

Calculating efficient frontier...

Individual Stock Metrics:
--------------------------------------------------
      Symbol  Expected Return  Volatility    Sharpe
 RELIANCE.NS        -0.044021    0.209028 -0.545483
      TCS.NS        -0.164251    0.226330 -1.034996
     INFY.NS        -0.250072    0.257533 -1.242842
 HDFCBANK.NS         0.024914    0.200631 -0.224718
KOTAKBANK.NS        -0.008135    0.223807 -0.349117
     SBIN.NS         0.162802    0.244164  0.380083
      ITC.NS        -0.231140    0.187609 -1.605142
    WIPRO.NS


Maximum Sharpe Ratio Portfolio (Tangency):
  Expected Return: 19.02%
  Volatility: 13.95%
  Sharpe Ratio: 0.86
  Weights:
    HDFCBANK.NS: 9.54%
    SBIN.NS: 25.00%
    SUNPHARMA.NS: 25.00%
    TITAN.NS: 25.00%
    MARUTI.NS: 15.46%



Generating Capital Allocation Line...

Generating efficient frontier chart...



Risk Analytics:
--------------------------------------------------

Value at Risk (95% confidence):
  Historical VaR: -1.66%
  Parametric VaR: -1.65%
  Expected Shortfall: -2.26%

Drawdown Analysis:
  Maximum Drawdown: -18.83%
  Max Drawdown Duration: 220 days
  Current Drawdown: -8.01%

Advanced Risk Metrics:
  Sortino Ratio: 0.82
  Calmar Ratio: 0.84

Beta Calculation:
  Note: Using NIFTY 50 as market proxy...

  Betas:
    RELIANCE.NS: 1.09
    TCS.NS: 0.77
    INFY.NS: 0.86
    HDFCBANK.NS: 1.06
    KOTAKBANK.NS: 0.94
    SBIN.NS: 1.19
    ITC.NS: 0.61
    WIPRO.NS: 0.97
    SUNPHARMA.NS: 0.53
    TITAN.NS: 0.83
    MARUTI.NS: 0.90

Generating risk dashboard...



PORTFOLIO OPTIMIZATION SUMMARY & CONCLUSION

1. PORTFOLIO COMPARISON
--------------------------------------------------
Metric                    Min Volatility       Max Sharpe          
--------------------------------------------------
Expected Return                         1.82%              19.02%
Volatility                             11.71%              13.95%
Sharpe Ratio                            -0.44                0.86
--------------------------------------------------

2. RECOMMENDED PORTFOLIO
--------------------------------------------------
Based on risk-return analysis, the MAXIMUM SHARPE RATIO portfolio is recommended:

  Expected Return: 19.02%
  Volatility: 13.95%
  Sharpe Ratio: 0.86

  Asset Allocation:
    HDFCBANK.NS: 9.54%
    SBIN.NS: 25.00%
    SUNPHARMA.NS: 25.00%
    TITAN.NS: 25.00%
    MARUTI.NS: 15.46%
--------------------------------------------------

3. RISK ASSESSMENT
--------------------------------------------------
  Value at Risk (95% confiden

# CELL 8: OPTIONS PRICING & GREEKS ANALYSIS

In [10]:
# CELL 8: OPTIONS PRICING & GREEKS ANALYSIS

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm
from scipy.optimize import brentq
from typing import Dict, List, Optional, Tuple, Any
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class OptionsConfig:
    risk_free_rate: float = 0.07
    trading_days: int = 252
    default_volatility: float = 0.25
    default_strike_percent: float = 1.0

config = OptionsConfig()

# ============================================================================
# BLACK-SCHOLES MODEL
# ============================================================================

class BlackScholes:
    """
    Black-Scholes option pricing model implementation
    European options, no dividends, constant volatility, lognormal prices
    """

    @staticmethod
    def d1(S: float, K: float, T: float, r: float, sigma: float) -> float:
        """Calculate d1 parameter"""
        if T <= 0 or sigma <= 0:
            return np.inf
        return (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

    @staticmethod
    def d2(S: float, K: float, T: float, r: float, sigma: float) -> float:
        """Calculate d2 parameter"""
        if T <= 0 or sigma <= 0:
            return np.inf
        return BlackScholes.d1(S, K, T, r, sigma) - sigma * np.sqrt(T)

    @staticmethod
    def call_price(S: float, K: float, T: float, r: float, sigma: float) -> float:
        """Calculate European call option price"""
        if T <= 0:
            return max(S - K, 0)
        if sigma <= 0:
            return max(S - K * np.exp(-r * T), 0)
        d1 = BlackScholes.d1(S, K, T, r, sigma)
        d2 = BlackScholes.d2(S, K, T, r, sigma)
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    @staticmethod
    def put_price(S: float, K: float, T: float, r: float, sigma: float) -> float:
        """Calculate European put option price"""
        if T <= 0:
            return max(K - S, 0)
        if sigma <= 0:
            return max(K * np.exp(-r * T) - S, 0)
        d1 = BlackScholes.d1(S, K, T, r, sigma)
        d2 = BlackScholes.d2(S, K, T, r, sigma)
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    @staticmethod
    def implied_volatility(S: float, K: float, T: float, r: float,
                          market_price: float, option_type: str = 'call') -> float:
        """
        Calculate implied volatility using Brent's method
        """
        if T <= 0 or market_price <= 0:
            return np.nan

        def objective(sigma):
            if option_type.lower() == 'call':
                price = BlackScholes.call_price(S, K, T, r, sigma)
            else:
                price = BlackScholes.put_price(S, K, T, r, sigma)
            return price - market_price

        try:
            implied_vol = brentq(objective, 0.01, 5.0)
            return implied_vol
        except ValueError:
            return np.nan

# ============================================================================
# GREEKS CALCULATOR
# ============================================================================

class Greeks:
    """
    Calculate option Greeks (sensitivity measures)
    """

    @staticmethod
    def delta(S: float, K: float, T: float, r: float, sigma: float,
              option_type: str = 'call') -> float:
        """Delta: Rate of change of option price with respect to underlying price"""
        if T <= 0 or sigma <= 0:
            return 1.0 if option_type == 'call' and S > K else 0.0
        d1 = BlackScholes.d1(S, K, T, r, sigma)
        if option_type.lower() == 'call':
            return norm.cdf(d1)
        else:
            return norm.cdf(d1) - 1

    @staticmethod
    def gamma(S: float, K: float, T: float, r: float, sigma: float) -> float:
        """Gamma: Rate of change of delta with respect to underlying price"""
        if T <= 0 or sigma <= 0:
            return 0.0
        d1 = BlackScholes.d1(S, K, T, r, sigma)
        return norm.pdf(d1) / (S * sigma * np.sqrt(T))

    @staticmethod
    def vega(S: float, K: float, T: float, r: float, sigma: float) -> float:
        """
        Vega: Rate of change of option price with respect to volatility
        Returns per 1% change in volatility (industry standard)
        """
        if T <= 0 or sigma <= 0:
            return 0.0
        d1 = BlackScholes.d1(S, K, T, r, sigma)
        return S * norm.pdf(d1) * np.sqrt(T) / 100

    @staticmethod
    def theta(S: float, K: float, T: float, r: float, sigma: float,
              option_type: str = 'call') -> float:
        """
        Theta: Rate of change of option price with respect to time
        Returns per day (industry standard)
        """
        if T <= 0 or sigma <= 0:
            return 0.0
        d1 = BlackScholes.d1(S, K, T, r, sigma)
        d2 = BlackScholes.d2(S, K, T, r, sigma)

        term1 = -S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))

        if option_type.lower() == 'call':
            term2 = -r * K * np.exp(-r * T) * norm.cdf(d2)
        else:
            term2 = r * K * np.exp(-r * T) * norm.cdf(-d2)

        return (term1 + term2) / 365

    @staticmethod
    def rho(S: float, K: float, T: float, r: float, sigma: float,
            option_type: str = 'call') -> float:
        """
        Rho: Rate of change of option price with respect to risk-free rate
        Returns per 1% change in interest rate (industry standard)
        """
        if T <= 0 or sigma <= 0:
            return 0.0
        d2 = BlackScholes.d2(S, K, T, r, sigma)

        if option_type.lower() == 'call':
            return K * T * np.exp(-r * T) * norm.cdf(d2) / 100
        else:
            return -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    @staticmethod
    def calculate_all(S: float, K: float, T: float, r: float, sigma: float,
                      option_type: str = 'call') -> Dict:
        """Calculate all Greeks"""
        return {
            'delta': Greeks.delta(S, K, T, r, sigma, option_type),
            'gamma': Greeks.gamma(S, K, T, r, sigma),
            'vega': Greeks.vega(S, K, T, r, sigma),
            'theta': Greeks.theta(S, K, T, r, sigma, option_type),
            'rho': Greeks.rho(S, K, T, r, sigma, option_type)
        }

# ============================================================================
# OPTION STRATEGY BUILDER
# ============================================================================

class OptionStrategy:
    """
    Build and analyze option trading strategies
    """

    @staticmethod
    def covered_call(S: float, K: float, T: float, r: float, sigma: float) -> Dict:
        """Covered Call: Buy stock + Sell Call"""
        call_price = BlackScholes.call_price(S, K, T, r, sigma)

        return {
            'name': 'Covered Call',
            'description': 'Buy stock, sell call option',
            'components': [
                {'action': 'Buy', 'asset': 'Stock', 'quantity': 1, 'price': S, 'strike': None},
                {'action': 'Sell', 'asset': 'Call', 'quantity': 1, 'price': call_price, 'strike': K}
            ],
            'net_cost': S - call_price,
            'max_profit': K - S + call_price,
            'max_loss': S - call_price,
            'break_even': S - call_price,
            'strike': K
        }

    @staticmethod
    def protective_put(S: float, K: float, T: float, r: float, sigma: float) -> Dict:
        """Protective Put: Buy stock + Buy Put"""
        put_price = BlackScholes.put_price(S, K, T, r, sigma)

        return {
            'name': 'Protective Put',
            'description': 'Buy stock, buy put option (insurance)',
            'components': [
                {'action': 'Buy', 'asset': 'Stock', 'quantity': 1, 'price': S, 'strike': None},
                {'action': 'Buy', 'asset': 'Put', 'quantity': 1, 'price': put_price, 'strike': K}
            ],
            'net_cost': S + put_price,
            'max_profit': float('inf'),
            'max_loss': S - K + put_price,
            'break_even': S + put_price,
            'strike': K
        }

    @staticmethod
    def straddle(S: float, K: float, T: float, r: float, sigma: float) -> Dict:
        """Straddle: Buy Call + Buy Put at same strike"""
        call_price = BlackScholes.call_price(S, K, T, r, sigma)
        put_price = BlackScholes.put_price(S, K, T, r, sigma)

        return {
            'name': 'Straddle',
            'description': 'Buy call and put at same strike (bet on volatility)',
            'components': [
                {'action': 'Buy', 'asset': 'Call', 'quantity': 1, 'price': call_price, 'strike': K},
                {'action': 'Buy', 'asset': 'Put', 'quantity': 1, 'price': put_price, 'strike': K}
            ],
            'net_cost': call_price + put_price,
            'max_profit': float('inf'),
            'max_loss': call_price + put_price,
            'break_even_low': K - (call_price + put_price),
            'break_even_high': K + (call_price + put_price),
            'strike': K
        }

    @staticmethod
    def strangle(S: float, K1: float, K2: float, T: float, r: float, sigma: float) -> Dict:
        """Strangle: Buy Call at higher strike + Buy Put at lower strike"""
        call_price = BlackScholes.call_price(S, K2, T, r, sigma)
        put_price = BlackScholes.put_price(S, K1, T, r, sigma)

        return {
            'name': 'Strangle',
            'description': 'Buy OTM call and put (cheaper volatility play)',
            'components': [
                {'action': 'Buy', 'asset': 'Put', 'quantity': 1, 'price': put_price, 'strike': K1},
                {'action': 'Buy', 'asset': 'Call', 'quantity': 1, 'price': call_price, 'strike': K2}
            ],
            'net_cost': call_price + put_price,
            'max_profit': float('inf'),
            'max_loss': call_price + put_price,
            'break_even_low': K1 - (call_price + put_price),
            'break_even_high': K2 + (call_price + put_price),
            'strike_low': K1,
            'strike_high': K2
        }

# ============================================================================
# OPTION PRICING VISUALIZATIONS
# ============================================================================

def plot_option_price_surface(S: float, K: float, r: float, sigma_range: np.ndarray,
                               T_range: np.ndarray, option_type: str = 'call'):
    """
    Create 3D surface plot of option price as function of volatility and time
    """
    T_mesh, sigma_mesh = np.meshgrid(T_range, sigma_range)

    prices = np.zeros_like(T_mesh)
    for i in range(len(sigma_range)):
        for j in range(len(T_range)):
            if option_type == 'call':
                prices[i, j] = BlackScholes.call_price(S, K, T_range[j], r, sigma_range[i])
            else:
                prices[i, j] = BlackScholes.put_price(S, K, T_range[j], r, sigma_range[i])

    fig = go.Figure(data=[
        go.Surface(
            z=prices,
            x=T_range,
            y=sigma_range,
            colorscale='Viridis',
            colorbar=dict(title='Option Price')
        )
    ])

    fig.update_layout(
        title=f'{option_type.capitalize()} Option Price Surface',
        scene=dict(
            xaxis_title='Time to Expiry (Years)',
            yaxis_title='Volatility',
            zaxis_title='Option Price'
        ),
        template='plotly_dark',
        height=600
    )

    fig.show()

def plot_greeks_dashboard(S: float, K: float, T: float, r: float, sigma: float,
                          option_type: str = 'call'):
    """
    Create comprehensive Greeks dashboard
    """
    price_range = np.linspace(S * 0.5, S * 1.5, 300)

    deltas = [Greeks.delta(p, K, T, r, sigma, option_type) for p in price_range]
    gammas = [Greeks.gamma(p, K, T, r, sigma) for p in price_range]
    vegas = [Greeks.vega(p, K, T, r, sigma) for p in price_range]
    thetas = [Greeks.theta(p, K, T, r, sigma, option_type) for p in price_range]
    rhos = [Greeks.rho(p, K, T, r, sigma, option_type) for p in price_range]

    if option_type == 'call':
        prices = [BlackScholes.call_price(p, K, T, r, sigma) for p in price_range]
    else:
        prices = [BlackScholes.put_price(p, K, T, r, sigma) for p in price_range]

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('Option Price', 'Delta',
                       'Gamma', 'Vega',
                       'Theta', 'Rho'),
        vertical_spacing=0.1
    )

    fig.add_trace(
        go.Scatter(
            x=price_range,
            y=prices,
            name='Option Price',
            line=dict(color='#4d96ff', width=2)
        ),
        row=1, col=1
    )
    fig.add_vline(x=S, line_dash="dash", line_color="#ef5350", row=1, col=1)

    fig.add_trace(
        go.Scatter(
            x=price_range,
            y=deltas,
            name='Delta',
            line=dict(color='#00b894', width=2)
        ),
        row=1, col=2
    )
    fig.add_hline(y=0.5, line_dash="dot", line_color="#7f8c8d", row=1, col=2)
    fig.add_vline(x=S, line_dash="dash", line_color="#ef5350", row=1, col=2)

    fig.add_trace(
        go.Scatter(
            x=price_range,
            y=gammas,
            name='Gamma',
            line=dict(color='#ffd93d', width=2),
            fill='tozeroy',
            fillcolor='rgba(255, 217, 61, 0.2)'
        ),
        row=2, col=1
    )
    fig.add_vline(x=S, line_dash="dash", line_color="#ef5350", row=2, col=1)

    fig.add_trace(
        go.Scatter(
            x=price_range,
            y=vegas,
            name='Vega',
            line=dict(color='#6c5ce7', width=2)
        ),
        row=2, col=2
    )
    fig.add_vline(x=S, line_dash="dash", line_color="#ef5350", row=2, col=2)

    fig.add_trace(
        go.Scatter(
            x=price_range,
            y=thetas,
            name='Theta',
            line=dict(color='#fd79a8', width=2)
        ),
        row=3, col=1
    )
    fig.add_hline(y=0, line_dash="solid", line_color="#7f8c8d", row=3, col=1)
    fig.add_vline(x=S, line_dash="dash", line_color="#ef5350", row=3, col=1)

    fig.add_trace(
        go.Scatter(
            x=price_range,
            y=rhos,
            name='Rho',
            line=dict(color='#fdcb6e', width=2)
        ),
        row=3, col=2
    )
    fig.add_vline(x=S, line_dash="dash", line_color="#ef5350", row=3, col=2)

    fig.update_layout(
        title=f'{option_type.capitalize()} Option Greeks (S=₹{S:.2f}, K=₹{K:.2f}, T={T:.2f}y, σ={sigma:.1%})',
        template='plotly_dark',
        height=1000,
        showlegend=False,
        hovermode='x unified'
    )

    fig.show()

def plot_payoff_diagram(strategy: Dict, S_range: np.ndarray):
    """
    Plot payoff diagram for option strategy
    """
    if strategy['name'] == 'Covered Call':
        K = strategy['strike']
        call_price = strategy['components'][1]['price']
        payoffs = S_range - strategy['net_cost'] - np.maximum(0, S_range - K)

    elif strategy['name'] == 'Protective Put':
        K = strategy['strike']
        put_price = strategy['components'][1]['price']
        payoffs = np.maximum(0, K - S_range) + S_range - strategy['net_cost']

    elif strategy['name'] == 'Straddle':
        K = strategy['strike']
        call_price = strategy['components'][0]['price']
        put_price = strategy['components'][1]['price']
        payoffs = np.maximum(0, S_range - K) + np.maximum(0, K - S_range) - strategy['net_cost']

    elif strategy['name'] == 'Strangle':
        K1 = strategy['strike_low']
        K2 = strategy['strike_high']
        payoffs = np.maximum(0, K1 - S_range) + np.maximum(0, S_range - K2) - strategy['net_cost']

    else:
        payoffs = S_range - strategy['net_cost']

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=S_range,
        y=payoffs,
        mode='lines',
        name='Payoff',
        line=dict(color='#4d96ff', width=2),
        fill='tozeroy',
        fillcolor='rgba(77, 150, 255, 0.2)'
    ))

    fig.add_hline(y=0, line_dash="solid", line_color="#7f8c8d")

    fig.update_layout(
        title=f'{strategy["name"]} - Payoff Diagram',
        xaxis_title='Underlying Price at Expiry',
        yaxis_title='Profit/Loss',
        template='plotly_dark',
        height=500,
        hovermode='x unified'
    )

    fig.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_options_analysis():
    """
    Complete options analysis workflow
    """
    print("=" * 70)
    print("OPTIONS PRICING & GREEKS ANALYSIS")
    print("=" * 70)

    # Fetch actual stock data for realistic volatility
    print("\nFetching market data for volatility estimation...")
    try:
        stock = yf.Ticker("RELIANCE.NS")
        hist = stock.history(period="1y")
        returns = np.log(hist['Close'] / hist['Close'].shift(1)).dropna()
        historical_vol = returns.std() * np.sqrt(252)
        current_price = hist['Close'].iloc[-1]
        print(f"  Current Price: ₹{current_price:.2f}")
        print(f"  Historical Volatility: {historical_vol:.1%}")
    except:
        historical_vol = 0.25
        current_price = 1000
        print("  Using default parameters")

    # Set parameters
    S = current_price
    K = round(S * 1.05, -2)  # 5% OTM strike rounded to nearest 100
    T = 0.5
    r = config.risk_free_rate
    sigma = historical_vol

    np.random.seed(42)

    print("\n1. BLACK-SCHOLES PRICING")
    print("-" * 50)
    print(f"Parameters:")
    print(f"  Spot Price (S): ₹{S:,.2f}")
    print(f"  Strike Price (K): ₹{K:,.2f}")
    print(f"  Time to Expiry (T): {T:.2f} years")
    print(f"  Risk-Free Rate (r): {r:.1%}")
    print(f"  Volatility (σ): {sigma:.1%}")
    print("-" * 50)

    # Calculate prices
    call_price = BlackScholes.call_price(S, K, T, r, sigma)
    put_price = BlackScholes.put_price(S, K, T, r, sigma)

    print(f"\nOption Prices:")
    print(f"  Call Option Price: ₹{call_price:.2f}")
    print(f"  Put Option Price: ₹{put_price:.2f}")

    # Put-Call Parity verification
    parity = S - K * np.exp(-r * T)
    difference = abs((call_price - put_price) - parity)
    print(f"\nPut-Call Parity:")
    print(f"  Call - Put: {call_price - put_price:.2f}")
    print(f"  Theoretical: {parity:.2f}")
    if difference < 1e-6:
        print("  Status: VERIFIED")
    else:
        print(f"  Status: VIOLATED (diff: {difference:.2f})")
    print("-" * 50)

    # Calculate Greeks
    print("\n2. GREEKS ANALYSIS")
    print("-" * 50)

    greeks = Greeks.calculate_all(S, K, T, r, sigma, 'call')
    print(f"\nCall Option Greeks:")
    print(f"  Delta (Δ): {greeks['delta']:.4f}")
    print(f"  Gamma (Γ): {greeks['gamma']:.4f}")
    print(f"  Vega (V): {greeks['vega']:.4f} (per 1% IV change)")
    print(f"  Theta (Θ): {greeks['theta']:.4f} (per day)")
    print(f"  Rho (ρ): {greeks['rho']:.4f} (per 1% rate change)")

    greeks_put = Greeks.calculate_all(S, K, T, r, sigma, 'put')
    print(f"\nPut Option Greeks:")
    print(f"  Delta (Δ): {greeks_put['delta']:.4f}")
    print(f"  Gamma (Γ): {greeks_put['gamma']:.4f}")
    print(f"  Vega (V): {greeks_put['vega']:.4f} (per 1% IV change)")
    print(f"  Theta (Θ): {greeks_put['theta']:.4f} (per day)")
    print(f"  Rho (ρ): {greeks_put['rho']:.4f} (per 1% rate change)")
    print("-" * 50)

    # Greeks Interpretation
    print("\nGreeks Interpretation:")
    print(f"  Delta ({greeks['delta']:.2f}): For ₹1 change in stock, option moves ₹{greeks['delta']:.2f}")
    print(f"  Gamma ({greeks['gamma']:.4f}): Delta changes by {greeks['gamma']:.4f} per ₹1 move")
    print(f"  Vega ({greeks['vega']:.2f}): For 1% change in volatility, option changes ₹{greeks['vega']:.2f}")
    print(f"  Theta ({greeks['theta']:.2f}): Option loses ₹{abs(greeks['theta']):.2f} per day")
    print("-" * 50)

    # Option Strategies
    print("\n3. OPTION STRATEGIES")
    print("-" * 50)

    strategies = [
        OptionStrategy.covered_call(S, K, T, r, sigma),
        OptionStrategy.protective_put(S, K, T, r, sigma),
        OptionStrategy.straddle(S, K, T, r, sigma),
        OptionStrategy.strangle(S, K*0.9, K*1.1, T, r, sigma)
    ]

    for strategy in strategies:
        print(f"\n{strategy['name']}:")
        print(f"  {strategy['description']}")
        print(f"  Net Cost: ₹{strategy['net_cost']:.2f}")
        if 'max_profit' in strategy:
            if strategy['max_profit'] == float('inf'):
                print("  Max Profit: Unlimited")
            else:
                print(f"  Max Profit: ₹{strategy['max_profit']:.2f}")
        if 'max_loss' in strategy:
            print(f"  Max Loss: ₹{strategy['max_loss']:.2f}")
        if 'break_even' in strategy:
            print(f"  Break Even: ₹{strategy['break_even']:.2f}")
        if 'break_even_low' in strategy:
            print(f"  Break Even Low: ₹{strategy['break_even_low']:.2f}")
            print(f"  Break Even High: ₹{strategy['break_even_high']:.2f}")
        print(f"  Components:")
        for comp in strategy['components']:
            strike_info = f" @ Strike ₹{comp['strike']}" if comp['strike'] else ""
            print(f"    {comp['action']} {comp['quantity']} {comp['asset']} @ ₹{comp['price']:.2f}{strike_info}")

    print("-" * 50)

    # Visualizations
    print("\n4. GENERATING VISUALIZATIONS")
    print("-" * 50)

    print("  Creating price surface...")
    sigma_range = np.linspace(0.1, 0.5, 20)
    T_range = np.linspace(0.1, 1.0, 20)
    plot_option_price_surface(S, K, r, sigma_range, T_range, 'call')

    print("  Creating Greeks dashboard...")
    plot_greeks_dashboard(S, K, T, r, sigma, 'call')

    print("  Creating payoff diagrams...")
    S_range = np.linspace(S * 0.6, S * 1.4, 100)
    for strategy in strategies:
        plot_payoff_diagram(strategy, S_range)

    # Implied Volatility
    print("\n5. IMPLIED VOLATILITY")
    print("-" * 50)

    # Generate synthetic market price with slight noise
    market_call = call_price * (1 + np.random.uniform(-0.05, 0.05))
    implied_vol = BlackScholes.implied_volatility(S, K, T, r, market_call, 'call')

    print(f"  Market Call Price: ₹{market_call:.2f}")
    print(f"  Implied Volatility: {implied_vol:.1%}")
    print(f"  Historical Volatility: {sigma:.1%}")
    if not np.isnan(implied_vol):
        print(f"  Difference: {(implied_vol - sigma)*100:.1f} percentage points")
    print("-" * 50)

    # Model assumptions
    print("\n6. MODEL ASSUMPTIONS")
    print("-" * 50)
    print("  Black-Scholes model assumptions:")
    print("  1. European option (can only be exercised at expiry)")
    print("  2. No dividends paid during option life")
    print("  3. Constant volatility throughout option life")
    print("  4. Lognormally distributed stock prices")
    print("  5. No transaction costs or taxes")
    print("  6. Continuous trading possible")
    print("  7. Risk-free rate is constant")
    print("-" * 50)

    print("\n" + "=" * 70)
    print("OPTIONS ANALYSIS COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    run_options_analysis()

OPTIONS PRICING & GREEKS ANALYSIS

Fetching market data for volatility estimation...
  Current Price: ₹1327.20
  Historical Volatility: 20.4%

1. BLACK-SCHOLES PRICING
--------------------------------------------------
Parameters:
  Spot Price (S): ₹1,327.20
  Strike Price (K): ₹1,400.00
  Time to Expiry (T): 0.50 years
  Risk-Free Rate (r): 7.0%
  Volatility (σ): 20.4%
--------------------------------------------------

Option Prices:
  Call Option Price: ₹65.43
  Put Option Price: ₹90.08

Put-Call Parity:
  Call - Put: -24.65
  Theoretical: -24.65
  Status: VERIFIED
--------------------------------------------------

2. GREEKS ANALYSIS
--------------------------------------------------

Call Option Greeks:
  Delta (Δ): 0.4780
  Gamma (Γ): 0.0021
  Vega (V): 3.7383 (per 1% IV change)
  Theta (Θ): -0.3184 (per day)
  Rho (ρ): 2.8449 (per 1% rate change)

Put Option Greeks:
  Delta (Δ): -0.5220
  Gamma (Γ): 0.0021
  Vega (V): 3.7383 (per 1% IV change)
  Theta (Θ): -0.0591 (per day)
  Rh

  Creating Greeks dashboard...


  Creating payoff diagrams...



5. IMPLIED VOLATILITY
--------------------------------------------------
  Market Call Price: ₹64.61
  Implied Volatility: 20.2%
  Historical Volatility: 20.4%
  Difference: -0.2 percentage points
--------------------------------------------------

6. MODEL ASSUMPTIONS
--------------------------------------------------
  Black-Scholes model assumptions:
  1. European option (can only be exercised at expiry)
  2. No dividends paid during option life
  3. Constant volatility throughout option life
  4. Lognormally distributed stock prices
  5. No transaction costs or taxes
  6. Continuous trading possible
  7. Risk-free rate is constant
--------------------------------------------------

OPTIONS ANALYSIS COMPLETE


# CELL 9: MACHINE LEARNING FOR RISK & RETURN PREDICTION


In [13]:
# CELL 9: MACHINE LEARNING FOR RISK & RETURN PREDICTION

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import RandomizedSearchCV

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ============================================================================
# CONFIGURATION
# ============================================================================

class MLConfig:
    n_splits: int = 5
    n_estimators: int = 100
    max_depth: int = 10
    random_state: int = 42
    lookback_days: int = 60
    forecast_days: int = 5
    test_size: float = 0.2
    n_iter: int = 20

config = MLConfig()

# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

class FeatureEngineer:
    """
    Create features for machine learning models
    """
    
    @staticmethod
    def create_price_features(df: pd.DataFrame) -> pd.DataFrame:
        """Create price-based features"""
        df = df.copy()
        
        # Returns with lags
        df['return_1d'] = df['close'].pct_change()
        df['return_5d'] = df['close'].pct_change(5)
        df['return_10d'] = df['close'].pct_change(10)
        df['return_20d'] = df['close'].pct_change(20)
        
        # Lagged returns
        for lag in [1, 2, 3, 5, 10]:
            df[f'return_1d_lag_{lag}'] = df['return_1d'].shift(lag)
        
        # Log returns
        df['log_return'] = np.log(df['close'] / df['close'].shift(1))
        
        # Price ratios
        df['high_low_ratio'] = df['high'] / df['low']
        df['close_open_ratio'] = df['close'] / df['open']
        
        # Price position in range
        df['price_range'] = df['high'] - df['low']
        df['price_position'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-8)
        
        return df
    
    @staticmethod
    def create_moving_averages(df: pd.DataFrame) -> pd.DataFrame:
        """Create moving average features"""
        df = df.copy()
        
        windows = [5, 10, 20, 50, 100, 200]
        for window in windows:
            df[f'sma_{window}'] = df['close'].rolling(window).mean()
            df[f'ema_{window}'] = df['close'].ewm(span=window, adjust=False).mean()
            
            # Price relative to moving average
            df[f'close_sma_{window}_ratio'] = df['close'] / df[f'sma_{window}'] - 1
            df[f'close_ema_{window}_ratio'] = df['close'] / df[f'ema_{window}'] - 1
        
        return df
    
    @staticmethod
    def create_volatility_features(df: pd.DataFrame) -> pd.DataFrame:
        """Create volatility-based features"""
        df = df.copy()
        
        windows = [5, 10, 20, 50]
        for window in windows:
            # Historical volatility
            df[f'volatility_{window}'] = df['return_1d'].rolling(window).std() * np.sqrt(252)
            
            # Volume volatility
            df[f'volume_volatility_{window}'] = df['volume'].pct_change().rolling(window).std()
            
            # Rolling skewness and kurtosis
            df[f'skewness_{window}'] = df['return_1d'].rolling(window).skew()
            df[f'kurtosis_{window}'] = df['return_1d'].rolling(window).kurt()
            
            # Rolling z-score
            df[f'z_score_{window}'] = (df['close'] - df['close'].rolling(window).mean()) / df['close'].rolling(window).std()
        
        return df
    
    @staticmethod
    def create_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
        """Create momentum features"""
        df = df.copy()
        
        # RSI
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))
        df['rsi_lag_1'] = df['rsi'].shift(1)
        df['rsi_lag_2'] = df['rsi'].shift(2)
        
        # MACD
        exp1 = df['close'].ewm(span=12, adjust=False).mean()
        exp2 = df['close'].ewm(span=26, adjust=False).mean()
        df['macd'] = exp1 - exp2
        df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
        df['macd_histogram'] = df['macd'] - df['macd_signal']
        
        # ROC
        for period in [5, 10, 20]:
            df[f'roc_{period}'] = (df['close'] - df['close'].shift(period)) / df['close'].shift(period) * 100
        
        return df
    
    @staticmethod
    def create_volume_features(df: pd.DataFrame) -> pd.DataFrame:
        """Create volume-based features"""
        df = df.copy()
        
        # Volume moving averages
        df['volume_sma_5'] = df['volume'].rolling(5).mean()
        df['volume_sma_20'] = df['volume'].rolling(20).mean()
        df['volume_ratio'] = df['volume'] / df['volume_sma_20']
        df['volume_ratio_lag_1'] = df['volume_ratio'].shift(1)
        
        # Money flow
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        money_flow = typical_price * df['volume']
        df['money_flow'] = money_flow.rolling(14).mean()
        
        return df
    
    @staticmethod
    def create_target(df: pd.DataFrame, forecast_days: int = 5) -> pd.DataFrame:
        """Create target variables (return and direction)"""
        df = df.copy()
        df[f'target_return_{forecast_days}d'] = df['close'].shift(-forecast_days) / df['close'] - 1
        df[f'target_direction_{forecast_days}d'] = (df[f'target_return_{forecast_days}d'] > 0).astype(int)
        return df
    
    @staticmethod
    def create_all_features(df: pd.DataFrame, forecast_days: int = 5) -> pd.DataFrame:
        """Create all features"""
        df = df.copy()
        
        df = FeatureEngineer.create_price_features(df)
        df = FeatureEngineer.create_moving_averages(df)
        df = FeatureEngineer.create_volatility_features(df)
        df = FeatureEngineer.create_momentum_features(df)
        df = FeatureEngineer.create_volume_features(df)
        df = FeatureEngineer.create_target(df, forecast_days)
        
        # Drop rows with NaN
        df = df.dropna()
        
        return df

# ============================================================================
# MACHINE LEARNING MODELS
# ============================================================================

class MLModels:
    """
    Collection of machine learning models for stock prediction
    """
    
    @staticmethod
    def random_forest(**kwargs) -> Pipeline:
        """Random Forest Regressor"""
        return Pipeline([
            ('model', RandomForestRegressor(
                n_estimators=kwargs.get('n_estimators', config.n_estimators),
                max_depth=kwargs.get('max_depth', config.max_depth),
                random_state=config.random_state,
                n_jobs=-1
            ))
        ])
    
    @staticmethod
    def gradient_boosting(**kwargs) -> Pipeline:
        """Gradient Boosting Regressor"""
        return Pipeline([
            ('model', GradientBoostingRegressor(
                n_estimators=kwargs.get('n_estimators', config.n_estimators),
                max_depth=kwargs.get('max_depth', config.max_depth),
                random_state=config.random_state
            ))
        ])
    
    @staticmethod
    def linear_regression() -> Pipeline:
        """Linear Regression with feature selection"""
        return Pipeline([
            ('scaler', StandardScaler()),
            ('selector', SelectKBest(f_regression, k=20)),
            ('model', LinearRegression())
        ])
    
    @staticmethod
    def svr(**kwargs) -> Pipeline:
        """Support Vector Regression"""
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', SVR(
                kernel=kwargs.get('kernel', 'rbf'),
                C=kwargs.get('C', 1.0),
                epsilon=kwargs.get('epsilon', 0.1)
            ))
        ])

# ============================================================================
# MODEL TRAINER
# ============================================================================

class ModelTrainer:
    """
    Train and evaluate machine learning models
    """
    
    def __init__(self, features: pd.DataFrame, target: pd.Series, target_direction: pd.Series = None):
        self.features = features
        self.target = target
        self.target_direction = target_direction
        self.models = {}
        self.results = {}
        self.predictions = {}
        self.direction_predictions = {}
        
        self.tscv = TimeSeriesSplit(n_splits=config.n_splits)
    
    def train_model_cv(self, name: str, model_pipeline) -> Dict:
        """Train model with time series cross-validation"""
        
        cv_scores = {
            'rmse': [],
            'mae': [],
            'r2': [],
            'accuracy': [],
            'precision': [],
            'recall': [],
            'f1': []
        }
        
        fold_predictions = []
        fold_true = []
        
        for fold, (train_idx, test_idx) in enumerate(self.tscv.split(self.features)):
            X_train = self.features.iloc[train_idx]
            X_test = self.features.iloc[test_idx]
            y_train = self.target.iloc[train_idx]
            y_test = self.target.iloc[test_idx]
            
            # Train model
            model_pipeline.fit(X_train, y_train)
            
            # Predictions
            y_pred = model_pipeline.predict(X_test)
            
            # Metrics
            cv_scores['rmse'].append(np.sqrt(mean_squared_error(y_test, y_pred)))
            cv_scores['mae'].append(mean_absolute_error(y_test, y_pred))
            cv_scores['r2'].append(r2_score(y_test, y_pred))
            
            # Direction predictions
            if self.target_direction is not None:
                y_true_dir = self.target_direction.iloc[test_idx]
                y_pred_dir = (y_pred > 0).astype(int)
                
                cv_scores['accuracy'].append(accuracy_score(y_true_dir, y_pred_dir))
                cv_scores['precision'].append(precision_score(y_true_dir, y_pred_dir, zero_division=0))
                cv_scores['recall'].append(recall_score(y_true_dir, y_pred_dir, zero_division=0))
                cv_scores['f1'].append(f1_score(y_true_dir, y_pred_dir, zero_division=0))
            
            fold_predictions.extend(y_pred)
            fold_true.extend(y_test)
        
        result = {
            'model': model_pipeline,
            'cv_rmse': np.mean(cv_scores['rmse']),
            'cv_rmse_std': np.std(cv_scores['rmse']),
            'cv_mae': np.mean(cv_scores['mae']),
            'cv_r2': np.mean(cv_scores['r2']),
            'cv_r2_std': np.std(cv_scores['r2']),
            'fold_predictions': np.asarray(fold_predictions, dtype=float),
            'fold_true': np.asarray(fold_true, dtype=float),
            'all_metrics': cv_scores
        }
        
        if self.target_direction is not None:
            result['cv_accuracy'] = np.mean(cv_scores['accuracy'])
            result['cv_precision'] = np.mean(cv_scores['precision'])
            result['cv_recall'] = np.mean(cv_scores['recall'])
            result['cv_f1'] = np.mean(cv_scores['f1'])
        
        self.models[name] = model_pipeline
        self.results[name] = result
        self.predictions[name] = fold_predictions
        
        return result
    
    def train_all_models(self) -> Dict:
        """Train all predefined models"""
        
        # Parameter grid for Random Forest
        rf_param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
        
        models = {
            'Random Forest': MLModels.random_forest()
        }
        
        # Use RandomizedSearchCV for Random Forest
        rf_model = MLModels.random_forest()
        rf_search = RandomizedSearchCV(
            rf_model.named_steps['model'],
            rf_param_grid,
            n_iter=config.n_iter,
            cv=TimeSeriesSplit(n_splits=3),
            random_state=config.random_state,
            n_jobs=-1
        )
        
        print("  Training Random Forest with hyperparameter tuning...")
        # Get full data for tuning
        rf_search.fit(self.features, self.target)
        best_rf = rf_search.best_estimator_
        
        # Create pipeline with best model
        best_rf_pipeline = Pipeline([
            ('model', best_rf)
        ])
        
        # Re-train with CV
        self.train_model_cv('Random Forest (Optimized)', best_rf_pipeline)
        
        # Train other models without tuning
        other_models = {
            'Gradient Boosting': MLModels.gradient_boosting(),
            'Linear Regression': MLModels.linear_regression(),
            'SVR': MLModels.svr()
        }
        
        for name, model in other_models.items():
            print(f"  Training {name}...")
            self.train_model_cv(name, model)
        
        return self.results
    
    def get_best_model(self) -> Tuple[str, Dict]:
        """Get the best model based on CV R²"""
        best_name = None
        best_r2 = -np.inf
        
        for name, result in self.results.items():
            if result['cv_r2'] > best_r2:
                best_r2 = result['cv_r2']
                best_name = name
        
        return best_name, self.results[best_name]
    
    def get_feature_importance(self, model_name: str) -> pd.DataFrame:
        """Get feature importance from tree-based models"""
        if model_name not in self.models:
            return pd.DataFrame()
        
        model = self.models[model_name]
        
        # Extract the actual model from pipeline
        if hasattr(model, 'named_steps'):
            actual_model = model.named_steps['model']
        else:
            actual_model = model
        
        if hasattr(actual_model, 'feature_importances_'):
            importance = actual_model.feature_importances_
            feature_names = self.features.columns
            
            return pd.DataFrame({
                'Feature': feature_names,
                'Importance': importance
            }).sort_values('Importance', ascending=False)
        
        return pd.DataFrame()
    
    def get_naive_baseline(self) -> Dict:
        """Calculate naive baseline (predict today's return for tomorrow)"""
        # Use last available return as prediction
        predictions = self.target.shift(1).dropna()
        actual = self.target.iloc[1:]  # Align
        
        rmse = np.sqrt(mean_squared_error(actual, predictions))
        mae = mean_absolute_error(actual, predictions)
        r2 = r2_score(actual, predictions)
        
        # Direction accuracy
        direction_actual = (actual > 0).astype(int)
        direction_pred = (predictions > 0).astype(int)
        accuracy = accuracy_score(direction_actual, direction_pred)
        
        return {
            'name': 'Naive Baseline',
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'accuracy': accuracy
        }

# ============================================================================
# VISUALIZATIONS
# ============================================================================

def plot_cv_results(result: Dict, symbol: str, model_name: str):
    """
    Plot cross-validation results
    """
    y_true = result['fold_true']
    y_pred = result['fold_predictions']
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Actual vs Predicted', 'Prediction Error',
                       'Prediction Distribution', 'Residuals'),
        vertical_spacing=0.12
    )
    
    # Actual vs Predicted
    fig.add_trace(
        go.Scatter(
            x=y_true,
            y=y_pred,
            mode='markers',
            name='Predictions',
            marker=dict(color='#4d96ff', size=6, opacity=0.6)
        ),
        row=1, col=1
    )
    
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode='lines',
            name='Perfect Prediction',
            line=dict(color='#ef5350', dash='dash')
        ),
        row=1, col=1
    )
    
    # Prediction Error
    errors = y_pred - y_true
    fig.add_trace(
        go.Histogram(
            x=errors,
            name='Error Distribution',
            marker_color='#ffd93d',
            opacity=0.7
        ),
        row=1, col=2
    )
    
    # Prediction Distribution
    fig.add_trace(
        go.Histogram(
            x=y_pred,
            name='Predicted',
            marker_color='#00b894',
            opacity=0.5,
            histnorm='probability'
        ),
        row=2, col=1
    )
    fig.add_trace(
        go.Histogram(
            x=y_true,
            name='Actual',
            marker_color='#ef5350',
            opacity=0.5,
            histnorm='probability'
        ),
        row=2, col=1
    )
    
    # Residuals over time
    fig.add_trace(
        go.Scatter(
            x=list(range(len(errors))),
            y=errors,
            mode='markers+lines',
            name='Residuals',
            line=dict(color='#6c5ce7', width=1),
            marker=dict(size=4, color='#6c5ce7')
        ),
        row=2, col=2
    )
    fig.add_hline(y=0, line_dash="solid", line_color="#7f8c8d", row=2, col=2)
    
    # Metrics annotation
    metrics_text = f"""
    <b>Model: {model_name}</b><br><br>
    CV RMSE: {result['cv_rmse']:.4f} (±{result['cv_rmse_std']:.4f})<br>
    CV R²: {result['cv_r2']:.4f} (±{result['cv_r2_std']:.4f})<br>
    """
    
    if 'cv_accuracy' in result:
        metrics_text += f"CV Accuracy: {result['cv_accuracy']:.4f}<br>"
        metrics_text += f"CV F1 Score: {result['cv_f1']:.4f}"
    
    fig.add_annotation(
        text=metrics_text,
        xref="paper", yref="paper",
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
        showarrow=False,
        font=dict(size=12, color="white"),
        bgcolor="rgba(0,0,0,0.7)",
        bordercolor="gray",
        borderwidth=1
    )
    
    fig.update_layout(
        title=f'{symbol} - Cross-Validation Results ({model_name})',
        template='plotly_dark',
        height=800,
        showlegend=True
    )
    
    fig.show()

def plot_feature_importance(importance_df: pd.DataFrame, top_n: int = 15):
    """
    Plot feature importance
    """
    if importance_df.empty:
        print("No feature importance available")
        return
    
    top_features = importance_df.head(top_n)
    
    fig = go.Figure()
    
    fig.add_trace(
        go.Bar(
            x=top_features['Importance'],
            y=top_features['Feature'],
            orientation='h',
            marker_color='#4d96ff',
            text=top_features['Importance'].round(4),
            textposition='outside'
        )
    )
    
    fig.update_layout(
        title=f'Top {top_n} Feature Importance',
        xaxis_title='Importance',
        yaxis_title='Features',
        template='plotly_dark',
        height=600
    )
    
    fig.show()

def plot_model_comparison(results: Dict):
    """
    Compare performance of all models
    """
    model_names = list(results.keys())
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('RMSE (Lower is Better)', 'R² Score (Higher is Better)',
                       'Accuracy (If Available)', 'F1 Score (If Available)'),
        vertical_spacing=0.12
    )
    
    # RMSE
    rmse_values = [results[name]['cv_rmse'] for name in model_names]
    rmse_stds = [results[name]['cv_rmse_std'] for name in model_names]
    
    fig.add_trace(
        go.Bar(
            x=model_names,
            y=rmse_values,
            error_y=dict(type='data', array=rmse_stds, visible=True),
            name='RMSE',
            marker_color='#4d96ff',
            text=[f'{v:.4f}' for v in rmse_values],
            textposition='outside'
        ),
        row=1, col=1
    )
    
    # R²
    r2_values = [results[name]['cv_r2'] for name in model_names]
    r2_stds = [results[name]['cv_r2_std'] for name in model_names]
    
    fig.add_trace(
        go.Bar(
            x=model_names,
            y=r2_values,
            error_y=dict(type='data', array=r2_stds, visible=True),
            name='R²',
            marker_color='#00b894',
            text=[f'{v:.4f}' for v in r2_values],
            textposition='outside'
        ),
        row=1, col=2
    )
    
    # Accuracy
    has_accuracy = 'cv_accuracy' in results[model_names[0]]
    if has_accuracy:
        acc_values = [results[name].get('cv_accuracy', 0) for name in model_names]
        fig.add_trace(
            go.Bar(
                x=model_names,
                y=acc_values,
                name='Accuracy',
                marker_color='#ffd93d',
                text=[f'{v:.4f}' for v in acc_values],
                textposition='outside'
            ),
            row=2, col=1
        )
        
        f1_values = [results[name].get('cv_f1', 0) for name in model_names]
        fig.add_trace(
            go.Bar(
                x=model_names,
                y=f1_values,
                name='F1 Score',
                marker_color='#6c5ce7',
                text=[f'{v:.4f}' for v in f1_values],
                textposition='outside'
            ),
            row=2, col=2
        )
    
    fig.update_layout(
        title='Model Performance Comparison',
        template='plotly_dark',
        height=700,
        showlegend=False
    )
    
    fig.show()

def plot_trading_simulation(returns: pd.Series, predictions: np.ndarray, symbol: str):
    """
    Simulate trading based on predictions
    """
    # Align returns and predictions
    min_len = min(len(returns), len(predictions))
    returns_aligned = returns.iloc[-min_len:]
    predictions_aligned = predictions[-min_len:]
    
    # Strategy: Buy if prediction > 0, hold otherwise
    strategy_returns = returns_aligned * (predictions_aligned > 0).astype(int)
    strategy_cumulative = (1 + strategy_returns).cumprod()
    buy_hold_cumulative = (1 + returns_aligned).cumprod()
    
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Cumulative Returns', 'Performance Metrics'),
        vertical_spacing=0.15,
        row_heights=[0.7, 0.3]
    )
    
    # Cumulative returns
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(strategy_cumulative)),
            y=strategy_cumulative,
            name='Strategy',
            line=dict(color='#00b894', width=2),
            fill='tozeroy',
            fillcolor='rgba(0, 184, 148, 0.2)'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(buy_hold_cumulative)),
            y=buy_hold_cumulative,
            name='Buy & Hold',
            line=dict(color='#4d96ff', width=2, dash='dash')
        ),
        row=1, col=1
    )
    
    # Performance metrics
    strategy_sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252) if strategy_returns.std() > 0 else 0
    buy_hold_sharpe = returns_aligned.mean() / returns_aligned.std() * np.sqrt(252) if returns_aligned.std() > 0 else 0
    
    strategy_peak = strategy_cumulative.cummax()
    strategy_drawdown = ((strategy_peak - strategy_cumulative) / strategy_peak).max()
    
    buy_hold_peak = buy_hold_cumulative.cummax()
    buy_hold_drawdown = ((buy_hold_peak - buy_hold_cumulative) / buy_hold_peak).max()
    
    metrics_text = f"""
    <b>Performance Metrics</b><br><br>
    <b>Strategy</b><br>
    Total Return: {(strategy_cumulative.iloc[-1] - 1) * 100:.2f}%<br>
    Sharpe Ratio: {strategy_sharpe:.2f}<br>
    Max Drawdown: {strategy_drawdown * 100:.2f}%<br><br>
    <b>Buy & Hold</b><br>
    Total Return: {(buy_hold_cumulative.iloc[-1] - 1) * 100:.2f}%<br>
    Sharpe Ratio: {buy_hold_sharpe:.2f}<br>
    Max Drawdown: {buy_hold_drawdown * 100:.2f}%
    """
    
    fig.add_annotation(
        text=metrics_text,
        xref="paper", yref="paper",
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
        showarrow=False,
        font=dict(size=12, color="white"),
        bgcolor="rgba(0,0,0,0.7)",
        bordercolor="gray",
        borderwidth=1
    )
    
    fig.update_layout(
        title=f'{symbol} - Trading Simulation',
        template='plotly_dark',
        height=700,
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.show()
    
    return {
        'strategy_return': (strategy_cumulative.iloc[-1] - 1) * 100,
        'buy_hold_return': (buy_hold_cumulative.iloc[-1] - 1) * 100,
        'strategy_sharpe': strategy_sharpe,
        'buy_hold_sharpe': buy_hold_sharpe,
        'strategy_drawdown': strategy_drawdown * 100,
        'buy_hold_drawdown': buy_hold_drawdown * 100
    }

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_ml_prediction():
    """
    Complete machine learning prediction workflow
    """
    print("=" * 70)
    print("MACHINE LEARNING FOR RISK & RETURN PREDICTION")
    print("=" * 70)
    
    # Fetch data
    symbol = 'RELIANCE.NS'
    print(f"\nFetching data for {symbol}...")
    
    df = yf.download(symbol, period="5y", progress=False)
    if df.empty:
        print("No data available")
        return
    
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = [str(c).strip().lower() for c in df.columns]
    
    print(f"Data points: {len(df)}")
    print("-" * 50)
    
    # Feature engineering
    print("\nCreating features...")
    forecast_days = config.forecast_days
    df_features = FeatureEngineer.create_all_features(df, forecast_days)
    
    print(f"  Features created: {len(df_features.columns)}")
    print(f"  Data points after feature engineering: {len(df_features)}")
    print("-" * 50)
    
    # Prepare data
    feature_columns = [col for col in df_features.columns 
                       if not col.startswith('target_') and not col.endswith('_lag_1')]
    target_column = f'target_return_{forecast_days}d'
    direction_column = f'target_direction_{forecast_days}d'
    
    X = df_features[feature_columns]
    y = df_features[target_column]
    y_direction = df_features[direction_column]
    
    # Remove any remaining NaN
    mask = ~(X.isna().any(axis=1) | y.isna() | y_direction.isna())
    X = X[mask]
    y = y[mask]
    y_direction = y_direction[mask]
    
    print(f"\nFinal dataset:")
    print(f"  Features: {X.shape[1]}")
    print(f"  Samples: {X.shape[0]}")
    print("-" * 50)
    
    # Train models
    print("\nTraining models...")
    trainer = ModelTrainer(X, y, y_direction)
    results = trainer.train_all_models()
    
    # Naive baseline
    print("\nCalculating naive baseline...")
    baseline = trainer.get_naive_baseline()
    print(f"  Naive Baseline RMSE: {baseline['rmse']:.4f}")
    print(f"  Naive Baseline R²: {baseline['r2']:.4f}")
    print(f"  Naive Baseline Accuracy: {baseline['accuracy']:.4f}")
    print("-" * 50)
    
    # Display results
    print("\nModel Performance (Cross-Validation):")
    print("-" * 60)
    for name, result in results.items():
        print(f"\n{name}:")
        print(f"  RMSE: {result['cv_rmse']:.4f} (±{result['cv_rmse_std']:.4f})")
        print(f"  R²: {result['cv_r2']:.4f} (±{result['cv_r2_std']:.4f})")
        if 'cv_accuracy' in result:
            print(f"  Accuracy: {result['cv_accuracy']:.4f}")
            print(f"  F1 Score: {result['cv_f1']:.4f}")
    print("-" * 60)
    
    # Best model
    best_name, best_result = trainer.get_best_model()
    print(f"\nBest Model: {best_name}")
    print(f"  CV RMSE: {best_result['cv_rmse']:.4f}")
    print(f"  CV R²: {best_result['cv_r2']:.4f}")
    if 'cv_accuracy' in best_result:
        print(f"  CV Accuracy: {best_result['cv_accuracy']:.4f}")
    print("-" * 50)
    
    # Check if model beats baseline
    if best_result['cv_r2'] > baseline['r2']:
        print(f"\nModel outperforms naive baseline (R² improvement: {best_result['cv_r2'] - baseline['r2']:.4f})")
    else:
        print(f"\nModel does NOT outperform naive baseline")
    print("-" * 50)
    
    # Feature importance
    print("\nFeature Importance Analysis...")
    importance_df = trainer.get_feature_importance(best_name)
    if not importance_df.empty:
        print("\nTop 10 Most Important Features:")
        print(importance_df.head(10).to_string(index=False))
        print("-" * 50)
        plot_feature_importance(importance_df)
    
    # Visualizations
    print("\nGenerating visualizations...")
    plot_cv_results(best_result, symbol, best_name)
    plot_model_comparison(results)
    
    # Trading simulation
    print("\nTrading Simulation...")
    # Use returns from feature engineered data
    returns = df_features['return_1d'].dropna()
    
    # Align returns with predictions (keep as Series)
    min_len = min(len(returns), len(best_result['fold_predictions']))
    returns_aligned = returns.iloc[-min_len:]
    preds_aligned = best_result['fold_predictions'][-min_len:]
    
    trading_results = plot_trading_simulation(returns_aligned, preds_aligned, symbol)
    
    # Summary
    print("\n" + "=" * 70)
    print("SUMMARY & RECOMMENDATIONS")
    print("=" * 70)
    
    print("\nModel Performance Summary:")
    print("-" * 50)
    print(f"  Best Model: {best_name}")
    print(f"  CV R²: {best_result['cv_r2']:.4f}")
    print(f"  RMSE: {best_result['cv_rmse']:.4f}")
    
    if best_result['cv_r2'] > 0.05:
        print("  Interpretation: Model shows moderate predictive power")
    elif best_result['cv_r2'] > 0.02:
        print("  Interpretation: Model shows weak but potentially useful predictive power")
    else:
        print("  Interpretation: Model shows no meaningful predictive power")
    
    print("\nTrading Simulation Results:")
    print("-" * 50)
    print(f"  Strategy Return: {trading_results['strategy_return']:.2f}%")
    print(f"  Buy & Hold Return: {trading_results['buy_hold_return']:.2f}%")
    print(f"  Strategy Sharpe: {trading_results['strategy_sharpe']:.2f}")
    print(f"  Buy & Hold Sharpe: {trading_results['buy_hold_sharpe']:.2f}")
    
    if trading_results['strategy_return'] > trading_results['buy_hold_return']:
        print("  Strategy outperformed buy-and-hold")
    else:
        print("  Strategy underperformed buy-and-hold")
    
    print("\nRecommendations:")
    print("-" * 50)
    print("  1. Use ensemble methods (combine multiple models)")
    print("  2. Include more features (macroeconomic data, sentiment)")
    print("  3. Experiment with different forecast horizons")
    print("  4. Implement walk-forward validation")
    print("  5. Consider transaction costs in simulation")
    print("  6. Use SHAP for better model interpretability")
    print("-" * 50)
    
    print("\n" + "=" * 70)
    print("MACHINE LEARNING PREDICTION COMPLETE")
    print("=" * 70)

if __name__ == "__main__":
    run_ml_prediction()

MACHINE LEARNING FOR RISK & RETURN PREDICTION

Fetching data for RELIANCE.NS...
Data points: 1238
--------------------------------------------------

Creating features...
  Features created: 79
  Data points after feature engineering: 884
--------------------------------------------------

Final dataset:
  Features: 74
  Samples: 884
--------------------------------------------------

Training models...
  Training Random Forest with hyperparameter tuning...
  Training Gradient Boosting...
  Training Linear Regression...
  Training SVR...

Calculating naive baseline...
  Naive Baseline RMSE: 0.0191
  Naive Baseline R²: 0.5542
  Naive Baseline Accuracy: 0.8007
--------------------------------------------------

Model Performance (Cross-Validation):
------------------------------------------------------------

Random Forest (Optimized):
  RMSE: 0.0317 (±0.0051)
  R²: -0.4559 (±0.6444)
  Accuracy: 0.5116
  F1 Score: 0.4754

Gradient Boosting:
  RMSE: 0.0367 (±0.0066)
  R²: -0.9657 (±0.9557


Trading Simulation...



SUMMARY & RECOMMENDATIONS

Model Performance Summary:
--------------------------------------------------
  Best Model: SVR
  CV R²: -0.0631
  RMSE: 0.0278
  Interpretation: Model shows no meaningful predictive power

Trading Simulation Results:
--------------------------------------------------
  Strategy Return: 0.00%
  Buy & Hold Return: 12.64%
  Strategy Sharpe: 0.00
  Buy & Hold Sharpe: 0.30
  Strategy underperformed buy-and-hold

Recommendations:
--------------------------------------------------
  1. Use ensemble methods (combine multiple models)
  2. Include more features (macroeconomic data, sentiment)
  3. Experiment with different forecast horizons
  4. Implement walk-forward validation
  5. Consider transaction costs in simulation
  6. Use SHAP for better model interpretability
--------------------------------------------------

MACHINE LEARNING PREDICTION COMPLETE
